<div align="center">

# PRCP-1027 — Skin Disorder Prediction

### Differential Diagnosis of Erythemato-Squamous Diseases using Machine Learning

**Domain:** Healthcare &nbsp;|&nbsp; **Task Type:** Multi-class Classification (6 classes) &nbsp;|&nbsp; **Data:** 366 patients × 34 attributes

---

**B.Tech Computer Science (Artificial Intelligence & Machine Learning)**
**Capstone Project — Single-Notebook Submission**

---

</div>

| Item | Detail |
|---|---|
| **Project ID** | PRCP-1027 |
| **Project Title** | Skin Disorder Prediction / Skin Disorder |
| **Dataset file used** | `dataset_35_dermatology__1_.csv` |
| **Dataset origin** | UCI Machine Learning Repository — *Dermatology* (Nilsel Ilter, M.D., Ph.D., Gazi University; H. Altay Guvenir, Bilkent University) |
| **Instances** | 366 |
| **Attributes** | 34 predictors + 1 target (`class`) |
| **Target classes** | 6 erythemato-squamous diseases |
| **Random seed** | 42 (fixed throughout for reproducibility) |

> **Note on the source link.** The requirements document titles the project **PRCP-1027**, while the download URL inside the same document points to a ZIP named **PRCP-1028-Skin-Disorder-Prediction**. This is a naming inconsistency in the provided document, not a data problem. This notebook uses the **uploaded CSV as the single source of truth** and its contents have been verified cell-by-cell against the attribute list in the requirements document (see Section 10).

## Abstract

Erythemato-squamous diseases (ESD) form a group of six dermatological conditions — psoriasis, seborrheic dermatitis, lichen planus, pityriasis rosea, chronic dermatitis and pityriasis rubra pilaris — that are notoriously difficult to tell apart. All six present with **erythema** (redness) and **scaling**, and the differences between them are subtle. A skin biopsy is usually required, yet even the histopathological picture is shared to a large degree across the group. A further complication is temporal: in its early stage one disease may display the characteristic features of another and only later develop its own.

This project builds a complete, reproducible machine-learning pipeline for the differential diagnosis of these six diseases from 34 attributes — 12 clinical (recorded at the bedside) and 22 histopathological (read under a microscope) — collected from 366 patients.

The work covers the three tasks required by the project brief. **Task 1** is a full exploratory and data-quality report: class balance, age and family-history structure, feature distributions, correlation structure, missing-value detection and outlier screening. **Task 2** develops and rigorously compares six classifiers — Logistic Regression, Decision Tree, Random Forest, SVM with an RBF kernel, K-Nearest Neighbours and Gradient Boosting — each tuned by grid search inside a stratified 5-fold cross-validation loop on the training partition alone, then scored once on a held-out test set that was never touched during model selection. **Task 3** converts the model's learned structure and the exploratory findings into concrete, medically-framed guidance for clinicians on earlier identification of these disorders.

All preprocessing (median imputation of `Age`, standardisation) is encapsulated inside scikit-learn `Pipeline` objects so that every transformation is fitted on training folds only, eliminating data leakage. The reported numbers in this notebook are **measured**, not assumed; every figure quoted in the written reports is produced by a cell in this notebook.

The strongest configuration was a tuned **Random Forest**, reaching **97.30% accuracy** and **0.9694 macro-F1** on the held-out test set, with **97.94% ± 1.29%** accuracy under 5-fold cross-validation. However, the test set contains only 74 patients, and the rarest class is represented by just 4 of them — so the headline number carries a confidence interval wide enough that several models are statistically indistinguishable. This notebook treats that uncertainty as a first-class finding rather than a footnote. **The resulting model is an academic decision-support prototype and is explicitly not a clinically validated diagnostic device.**

## Table of Contents

| § | Section |
|---|---|
| 1 | [Introduction to Erythemato-Squamous Diseases](#s1) |
| 2 | [Problem Statement](#s2) |
| 3 | [Motivation](#s3) |
| 4 | [Objectives](#s4) |
| 5 | [Scope](#s5) |
| 6 | [Dataset Description](#s6) |
| 7 | [Proposed Methodology](#s7) |
| 8 | [Import Libraries](#s8) |
| 9 | [Configuration](#s9) |
| 10 | [Dataset Loading and Structural Verification](#s10) |
| 11 | [Exploratory Data Analysis — **Task 1**](#s11) |
| 12 | [Data Quality Analysis — **Task 1**](#s12) |
| 13 | [EDA Interpretation & Findings](#s13) |
| 14 | [Data Preprocessing](#s14) |
| 15 | [Train / Test Split Strategy](#s15) |
| 16 | [Feature Engineering & Selection Analysis](#s16) |
| 17 | [Model Development — **Task 2**](#s17) |
| 18 | [Model Training with Cross-Validation](#s18) |
| 19 | [Model Evaluation](#s19) |
| 20 | [Model Comparison Report](#s20) |
| 21 | [Challenges Faced and Techniques Used](#s21) |
| 22 | [Suggestions to Doctors — **Task 3**](#s22) |
| 23 | [Final Findings](#s23) |
| 24 | [Limitations](#s24) |
| 25 | [Conclusion](#s25) |
| 26 | [Future Scope](#s26) |
| 27 | [References](#s27) |
| 28 | [Execution Guide](#s28) |
| 29 | [Final Verification Report](#s29) |

<a id="s1"></a>
# 1. Introduction to Erythemato-Squamous Diseases

**Erythema** means redness of the skin caused by dilated capillaries; **squamous** refers to scaling, the visible shedding of the outer epidermal layer. The erythemato-squamous group is defined by the joint presence of these two signs. Six conditions fall into the group:

| # | Disease | Short clinical picture |
|---|---|---|
| 1 | **Psoriasis** | Sharply demarcated, silvery-scaled plaques with a predilection for extensor surfaces (knees, elbows) and the scalp. Chronic, immune-mediated, strongly familial. |
| 2 | **Seborrheic dermatitis** | Greasy yellowish scale on sebum-rich areas (scalp, nasolabial folds, chest). Ill-defined borders; relapsing course. |
| 3 | **Lichen planus** | The "six P's" — purple, polygonal, planar, pruritic papules and plaques — frequently with oral mucosal lesions (Wickham's striae). |
| 4 | **Pityriasis rosea** | Typically begins with a solitary "herald patch" followed within days by a self-limiting truncal eruption along skin-cleavage lines. Usually resolves in 6–8 weeks. |
| 5 | **Chronic dermatitis** | The end-stage of persistent scratching and inflammation: lichenified, thickened, fibrotic skin with intense itch. |
| 6 | **Pityriasis rubra pilaris** | Rare. Orange-red scaling plaques with characteristic follicular papules and islands of spared skin; a juvenile-onset form exists. |

## Why differential diagnosis is genuinely hard

The requirements document states the difficulty precisely, and it is worth restating because it shapes every modelling decision in this notebook:

1. **Shared clinical signs.** All six diseases present with erythema and scaling. The visual overlap at the bedside is substantial.
2. **Shared histopathology.** A biopsy is usually taken, but the six conditions also share many microscopic features — so histology narrows the field without always closing it.
3. **Temporal ambiguity.** A disease may show the features of a *different* disease at its beginning stage and only develop its own characteristic features later. A single-timepoint snapshot — which is exactly what this dataset contains — can therefore be intrinsically ambiguous.

Point (3) places a genuine ceiling on achievable accuracy that no amount of model tuning can lift. It is one reason this notebook resists the temptation to read a very high accuracy score as evidence of a solved problem.

## Why this is a good machine-learning problem

The clinician's task — weigh 34 graded observations simultaneously and rank six competing hypotheses — is precisely the kind of high-dimensional pattern-weighting task where statistical models are strong. The features are already quantified on an ordinal 0–3 scale by a trained observer, which removes the perception step and leaves a clean tabular learning problem.

<a id="s2"></a>
# 2. Problem Statement

> Given 34 attributes per patient — 12 clinical observations recorded at first consultation and 22 histopathological observations obtained from a skin biopsy — **predict which of the six erythemato-squamous diseases the patient has.**

Formally, this is a **supervised multi-class classification** problem:

$$f: \mathbb{R}^{34} \rightarrow \{1, 2, 3, 4, 5, 6\}$$

learned from $n = 366$ labelled examples.

The problem carries four properties that drive the methodology:

| Property | Value | Consequence for the design |
|---|---|---|
| Sample size | 366 | Too small for a generous three-way split; cross-validation is mandatory, and all variance estimates must be reported. |
| Dimensionality | 34 features | $n/p \approx 10.8$ — a low ratio that invites overfitting and demands regularisation. |
| Class balance | 112 : 61 : 72 : 49 : 52 : 20 | Moderately imbalanced (5.6:1). Accuracy alone is misleading; macro-averaged metrics are required. |
| Feature scale | 33 ordinal (0–3), 1 continuous (Age, 0–75) | Age dominates any Euclidean distance unless scaled. Scaling is not cosmetic here — it is load-bearing. |

Beyond raw predictive accuracy, the problem statement carries a **clinical** requirement that a pure accuracy objective would miss: a model deployed as decision support must be *interpretable enough that a dermatologist can audit its reasoning*. A recommendation a clinician cannot interrogate is a recommendation they cannot responsibly act upon. This constraint is carried explicitly into the final model-selection criteria in Section 20.

<a id="s3"></a>
# 3. Motivation — Why Early and Correct Identification Matters

The six diseases share an appearance but **not a treatment, not a prognosis, and not a level of seriousness**. Misclassification is therefore not a cosmetic error:

- **Psoriasis** is a lifelong systemic inflammatory disease associated with psoriatic arthritis, metabolic syndrome and cardiovascular risk. Early recognition opens the door to disease-modifying therapy and to screening for joint involvement; delay can mean irreversible joint damage.
- **Pityriasis rosea** is self-limiting and typically resolves without intervention. Mistaking it for psoriasis exposes a patient to months of unnecessary, potentially immunosuppressive treatment.
- **Lichen planus** with oral involvement requires monitoring — oral lichen planus carries a small but real risk of malignant transformation.
- **Chronic dermatitis** is largely the product of a scratch–inflame–scratch cycle. Breaking it early prevents the permanent lichenification and dermal fibrosis that define the late stage.
- **Pityriasis rubra pilaris** is rare enough that many clinicians see very few cases; it is consequently the one most likely to be misdiagnosed by default.

**Diagnostic delay has a direct cost.** In routine practice the diagnosis often requires a biopsy, histopathology turnaround, and sometimes a second visit weeks later once the lesion has evolved. A decision-support tool that flags the most probable diagnoses — and, importantly, flags *when the evidence is ambiguous* — can shorten this loop, prioritise which patients genuinely need a biopsy, and provide a second opinion in settings where a specialist dermatologist is not available. In many primary-care and rural contexts, that last point is the decisive one.

<a id="s4"></a>
# 4. Objectives

**Primary objectives** (mapped to the three tasks in the project brief):

1. **Task 1 — Data Analysis Report.** Produce a complete exploratory and data-quality analysis: structure verification against the requirements document, class distribution, age and family-history structure, clinical vs. histopathological feature behaviour, correlation structure, missing values, duplicates, outliers, and out-of-range values.
2. **Task 2 — Predictive Model.** Build, tune and honestly compare at least four machine-learning models for the six-class problem, using stratified cross-validation and a held-out test set, and select a best model for production with justification.
3. **Task 3 — Suggestions to Doctors.** Translate the data-driven findings into actionable, medically responsible guidance for earlier identification of these disorders.

**Supporting objectives:**

4. Establish a leak-free preprocessing pipeline in which every fitted transformation sees training data only.
5. Quantify — not assume — the effect of feature scaling, feature selection, dimensionality reduction and class re-weighting on performance.
6. Report per-class performance for all six diseases, since aggregate accuracy conceals failures on the rarest classes.
7. Characterise the residual error structure: identify *which* disease pairs the models confuse and explain *why* that confusion is clinically plausible.
8. State uncertainty and limitations with academic honesty, and specify what additional validation would be required before any clinical use.

<a id="s5"></a>
# 5. Scope

### In scope

- Supervised multi-class classification on the supplied 366-patient tabular dataset.
- Classical machine-learning models appropriate to small tabular data (linear, tree, ensemble, kernel and instance-based).
- Full EDA, data-quality audit, leak-free preprocessing, cross-validated tuning, and multi-metric evaluation.
- Model interpretability analysis (feature importance, per-class error structure).
- Written reports: model comparison, challenges faced, and clinical suggestions.

### Explicitly out of scope

- **Image-based diagnosis.** This dataset contains no photographs; it contains a clinician's *graded assessments*. A CNN is not applicable here.
- **Deep learning.** With 366 samples and 34 tabular features, neural networks would overfit badly and offer no advantage over gradient-boosted trees or regularised linear models. Using one would be complexity for its own sake.
- **Clinical deployment.** No prospective validation, no external cohort, no regulatory pathway. The output is an academic prototype.
- **Causal inference.** All findings are associational. The model learns which feature patterns *co-occur* with which diagnosis; it does not learn disease mechanism.
- **Treatment recommendation.** The model predicts a diagnostic class only.

<a id="s6"></a>
# 6. Dataset Description

**Source.** UCI Machine Learning Repository, *Dermatology* dataset. Donated by Nilsel Ilter, M.D., Ph.D. (Gazi University, School of Medicine) and H. Altay Guvenir, Ph.D. (Bilkent University, Department of Computer Engineering). Patient names and ID numbers were removed by the donors before release.

**File used here.** `dataset_35_dermatology__1_.csv` — 366 rows × 35 columns (34 predictors + `class`), with a header row of named attributes.

### 6.1 Clinical attributes (recorded at the bedside)

These 12 features require no biopsy. They are what a clinician can assess at the **first consultation** — which makes them the features most relevant to *early* identification.

| # | Attribute | Range | Meaning |
|---|---|---|---|
| 1 | `erythema` | 0–3 | Redness from capillary dilation |
| 2 | `scaling` | 0–3 | Visible shedding of epidermal scale |
| 3 | `definite_borders` | 0–3 | How sharply the lesion is demarcated from normal skin |
| 4 | `itching` | 0–3 | Pruritus severity |
| 5 | `koebner_phenomenon` | 0–3 | New lesions appearing along a line of skin trauma |
| 6 | `polygonal_papules` | 0–3 | Flat-topped many-sided papules (a lichen planus hallmark) |
| 7 | `follicular_papules` | 0–3 | Papules centred on hair follicles |
| 8 | `oral_mucosal_involvement` | 0–3 | Lesions inside the mouth |
| 9 | `knee_and_elbow_involvement` | 0–3 | Extensor-surface distribution |
| 10 | `scalp_involvement` | 0–3 | Scalp lesions |
| 11 | `family_history` | **0 or 1** | 1 if any of these diseases is present in the family |
| 34 | `Age` | 0–75 (continuous) | Patient age in years; **contains missing values** |

### 6.2 Histopathological attributes (read under a microscope after biopsy)

| # | Attribute | # | Attribute |
|---|---|---|---|
| 12 | `melanin_incontinence` | 23 | `spongiform_pustule` |
| 13 | `eosinophils_in_the_infiltrate` | 24 | `munro_microabcess` |
| 14 | `PNL_infiltrate` | 25 | `focal_hypergranulosis` |
| 15 | `fibrosis_of_the_papillary_dermis` | 26 | `disappearance_of_the_granular_layer` |
| 16 | `exocytosis` | 27 | `vacuolisation_and_damage_of_basal_layer` |
| 17 | `acanthosis` | 28 | `spongiosis` |
| 18 | `hyperkeratosis` | 29 | `saw-tooth_appearance_of_retes` |
| 19 | `parakeratosis` | 30 | `follicular_horn_plug` |
| 20 | `clubbing_of_the_rete_ridges` | 31 | `perifollicular_parakeratosis` |
| 21 | `elongation_of_the_rete_ridges` | 32 | `inflammatory_monoluclear_inflitrate` |
| 22 | `thinning_of_the_suprapapillary_epidermis` | 33 | `band-like_infiltrate` |

*(Attributes 32 and 33 carry the donors' original spelling, `inflammatory_monoluclear_inflitrate`. The column name is kept verbatim so the notebook matches the file exactly.)*

**Encoding of the 0–3 grades.** `0` = feature absent, `3` = largest amount possible, `1` and `2` = intermediate. These are **ordinal**, not nominal: the ordering carries real information (grade 3 is more of the feature than grade 1). This justifies treating them as numeric rather than one-hot encoding them — one-hot would discard the ordering and inflate the feature count from 34 to over 100 on a 366-row dataset.

### 6.3 Target variable

| Code | Disease | Count | Share |
|---|---|---|---|
| 1 | Psoriasis | 112 | 30.6% |
| 2 | Seborrheic dermatitis | 61 | 16.7% |
| 3 | Lichen planus | 72 | 19.7% |
| 4 | Pityriasis rosea | 49 | 13.4% |
| 5 | Chronic dermatitis | 52 | 14.2% |
| 6 | Pityriasis rubra pilaris | 20 | 5.5% |

*(These counts are verified programmatically in Section 10 — they are not copied from an external source.)*

<a id="s7"></a>
# 7. Proposed Methodology

```
                 ┌──────────────────────────────────────────────┐
                 │  1. LOAD & VERIFY  (366 × 35, schema check)  │
                 └───────────────────────┬──────────────────────┘
                                         ▼
                 ┌──────────────────────────────────────────────┐
                 │  2. EDA + DATA QUALITY AUDIT      → Task 1   │
                 │     distributions · correlations · missing   │
                 └───────────────────────┬──────────────────────┘
                                         ▼
                 ┌──────────────────────────────────────────────┐
                 │  3. STRATIFIED SPLIT  80 / 20                │
                 │     test set locked away until Section 19    │
                 └───────────────────────┬──────────────────────┘
                                         ▼
          ╔══════════════ TRAINING PARTITION ONLY (292) ═══════════════╗
          ║   4. Pipeline: median-impute Age → StandardScaler          ║
          ║   5. Feature-engineering experiments (5-fold CV)           ║
          ║   6. 6 models × GridSearchCV (5-fold, f1_macro)            ║
          ╚═══════════════════════════════┬════════════════════════════╝
                                          ▼
                 ┌──────────────────────────────────────────────┐
                 │  7. SINGLE EVALUATION ON HELD-OUT TEST (74)  │
                 │     accuracy · macro/weighted P,R,F1 · κ     │
                 │     confusion matrices · per-class report    │
                 └───────────────────────┬──────────────────────┘
                                         ▼
                 ┌──────────────────────────────────────────────┐
                 │  8. REPORTS: comparison · challenges ·       │
                 │     suggestions to doctors        → Task 3   │
                 └──────────────────────────────────────────────┘
```

### Key methodological commitments

1. **The test set is touched exactly once.** It is split off before any analysis that informs modelling, and is not used for tuning, feature selection or threshold-setting. Every number in Section 19 is a genuine out-of-sample estimate.
2. **All preprocessing lives inside a `Pipeline`.** The imputer and scaler are fitted within each cross-validation fold, on that fold's training portion only. Fitting a scaler on the full dataset before splitting is the single most common leakage bug in student ML projects; the pipeline construction makes it structurally impossible here.
3. **Stratification everywhere.** Both the train/test split and every CV fold preserve the 6-class proportions. With only 20 samples in class 6, non-stratified splitting could easily produce folds containing zero examples of it.
4. **`f1_macro` is the tuning objective, not accuracy.** Macro-F1 weights all six diseases equally, so the optimiser cannot buy a good score by neglecting the rare classes.
5. **Variance is reported alongside every mean.** A cross-validated mean without its standard deviation is an unfalsifiable claim.

### Why 80/20 + 5-fold CV rather than a 70/15/15 three-way split

Both are permitted by the brief. At $n = 366$ a 15% validation set holds 55 patients — of which roughly **3** would belong to class 6. Model selection decisions made on 3 samples are noise, and a second holdout would also shrink the training set. Five-fold cross-validation on the 292-patient training partition instead gives every training sample a turn at validation, produces a variance estimate for free, and leaves the 74-patient test set intact for a single clean final measurement. This is the standard choice for small-sample work and is used throughout.

<a id="s8"></a>
# 8. Import Libraries

Only libraries that are actually used are imported. `xgboost` is treated as **optional**: the notebook detects whether it is installed and gracefully falls back to scikit-learn's `GradientBoostingClassifier` (which is always available) as the boosting representative, so the notebook runs end-to-end either way.

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import sys
import time
import json
import random
import warnings
from pathlib import Path

# ── Numerical / data handling ───────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# ── scikit-learn: preprocessing & pipeline ──────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── scikit-learn: model selection ───────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    GridSearchCV,
    learning_curve,
)

# ── scikit-learn: feature selection ─────────────────────────────────────────
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2

# ── scikit-learn: models ────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance

# ── scikit-learn: metrics ───────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    cohen_kappa_score,
)

# ── Persistence ─────────────────────────────────────────────────────────────
import joblib

# ── Optional: XGBoost (used only if present) ────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

# ── Optional: imbalanced-learn / SMOTE (used only if present) ───────────────
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_AVAILABLE = True
except Exception:
    IMBLEARN_AVAILABLE = False

warnings.filterwarnings("ignore")

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)
import sklearn; print("scikit-learn:", sklearn.__version__)
print("joblib      :", joblib.__version__)
print()
print("XGBoost available        :", XGBOOST_AVAILABLE, "(optional — fallback: GradientBoostingClassifier)")
print("imbalanced-learn available:", IMBLEARN_AVAILABLE, "(optional — fallback: class_weight='balanced')")

<a id="s9"></a>
# 9. Configuration

All tunable settings live in one place. **`DATASET_PATH` is the only path you may need to change** — no absolute paths are hard-coded anywhere else in the notebook.

The loader searches a list of likely locations, so the notebook works whether the CSV sits next to the notebook, in a `data/` subfolder, or in a Jupyter upload directory.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  — edit this cell only
# ════════════════════════════════════════════════════════════════════════════

DATASET_FILENAME = "dataset_35_dermatology__1_.csv"

# Candidate locations searched in order. Add your own path at the front if needed.
CANDIDATE_PATHS = [
    Path(DATASET_FILENAME),
    Path("data") / DATASET_FILENAME,
    Path("dataset") / DATASET_FILENAME,
    Path("/mnt/user-data/uploads") / DATASET_FILENAME,
    Path.home() / "Downloads" / DATASET_FILENAME,
]

# ── Reproducibility ─────────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── Split & validation strategy ─────────────────────────────────────────────
TEST_SIZE   = 0.20          # 20% held out, never seen during training/tuning
CV_FOLDS    = 5             # stratified k-fold on the training partition
SCORING     = "f1_macro"    # tuning objective: treats all 6 diseases equally

# ── Output ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Plot styling ────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"]        = 110
plt.rcParams["savefig.dpi"]       = 150
plt.rcParams["figure.autolayout"] = True
plt.rcParams["axes.titleweight"]  = "bold"
plt.rcParams["axes.titlesize"]    = 12

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]

# ── Global seeds ────────────────────────────────────────────────────────────
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

# ── Domain constants (from the requirements document) ───────────────────────
CLASS_NAMES = {
    1: "Psoriasis",
    2: "Seborrheic dermatitis",
    3: "Lichen planus",
    4: "Pityriasis rosea",
    5: "Chronic dermatitis",
    6: "Pityriasis rubra pilaris",
}
CLASS_SHORT = {
    1: "Psoriasis", 2: "Seb. derm.", 3: "Lichen pl.",
    4: "Pit. rosea", 5: "Chr. derm.", 6: "PRP",
}
LABELS_ORDER = [1, 2, 3, 4, 5, 6]
DISPLAY_NAMES = [CLASS_SHORT[c] for c in LABELS_ORDER]

# Clinical block = attributes 1-11 + Age (12 clinical features, per the brief)
# Histopathological block = attributes 12-33 (22 features, per the brief)
N_CLINICAL_ORDINAL  = 11
N_HISTOPATHOLOGICAL = 22

TARGET_COL = "class"
AGE_COL    = "Age"

print(f"RANDOM_STATE = {RANDOM_STATE} | TEST_SIZE = {TEST_SIZE} | CV_FOLDS = {CV_FOLDS} | SCORING = '{SCORING}'")
print(f"Outputs will be written to: {OUTPUT_DIR.resolve()}")

<a id="s10"></a>
# 10. Dataset Loading and Structural Verification

This section does three things before any analysis begins:

1. **Locate and load** the CSV from the configured candidate paths.
2. **Verify** the actual structure against the requirements document — attribute names, count, target classes. Nothing downstream assumes a column exists without this check passing.
3. **Detect the `Age` encoding issue.** The UCI dermatology data marks missing ages with the literal string `'?'`. When pandas reads a column containing `'?'`, the whole column becomes `object` dtype. This cell detects that, reports it, and converts to numeric with `'?'` → `NaN`.

In [ ]:
# ── Locate the dataset ──────────────────────────────────────────────────────
DATASET_PATH = None
for p in CANDIDATE_PATHS:
    if p.exists():
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        f"Could not find '{DATASET_FILENAME}'.\n"
        f"Searched:\n  " + "\n  ".join(str(p.resolve()) for p in CANDIDATE_PATHS) +
        "\n\nFix: place the CSV beside this notebook, or add its path to CANDIDATE_PATHS."
    )

print(f"Loading from : {DATASET_PATH.resolve()}")

# The '?' missing-value marker is declared here so pandas maps it to NaN directly.
df_raw = pd.read_csv(DATASET_PATH)
df = pd.read_csv(DATASET_PATH, na_values=["?"])

print(f"Shape        : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory       : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  STRUCTURAL VERIFICATION against the requirements document
# ════════════════════════════════════════════════════════════════════════════
checks = []

# 1. Target column present
checks.append(("Target column 'class' present", TARGET_COL in df.columns))

# 2. 34 predictors
n_predictors = df.shape[1] - 1
checks.append((f"34 predictor attributes (found {n_predictors})", n_predictors == 34))

# 3. Six target classes coded 1..6
observed_classes = sorted(df[TARGET_COL].unique().tolist())
checks.append((f"Target classes are 1-6 (found {observed_classes})", observed_classes == LABELS_ORDER))

# 4. Age column present
checks.append((f"'{AGE_COL}' column present", AGE_COL in df.columns))

# 5. All 33 non-Age predictors confined to the documented 0-3 ordinal range
feature_cols_all = [c for c in df.columns if c != TARGET_COL]
ordinal_cols     = [c for c in feature_cols_all if c != AGE_COL]
in_range = all(df[c].dropna().between(0, 3).all() for c in ordinal_cols)
checks.append((f"All {len(ordinal_cols)} ordinal features within 0-3", in_range))

# 6. family_history is binary
fh_ok = set(df["family_history"].dropna().unique()) <= {0, 1}
checks.append(("'family_history' is binary (0/1)", fh_ok))

# 7. Expected row count for the UCI Dermatology dataset
checks.append((f"366 instances (found {df.shape[0]})", df.shape[0] == 366))

print("=" * 74)
print("STRUCTURAL VERIFICATION")
print("=" * 74)
for desc, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {desc}")
print("=" * 74)

if all(ok for _, ok in checks):
    print("RESULT: dataset structure matches the PRCP-1027 requirements document.")
else:
    print("RESULT: one or more checks FAILED — inspect before proceeding.")

In [ ]:
# ── The Age encoding issue, made explicit ───────────────────────────────────
print("Age column BEFORE declaring na_values:")
print(f"  dtype                 : {df_raw[AGE_COL].dtype}")
n_qmark = (df_raw[AGE_COL].astype(str).str.strip() == "?").sum()
print(f"  literal '?' entries   : {n_qmark}")
print(f"  sample of raw values  : {df_raw[AGE_COL].astype(str).unique()[:8].tolist()}")
print()

# Guarantee numeric dtype regardless of how pandas parsed the file
df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")

print("Age column AFTER conversion:")
print(f"  dtype                 : {df[AGE_COL].dtype}")
print(f"  NaN count             : {df[AGE_COL].isna().sum()}")
print(f"  range                 : {df[AGE_COL].min():.0f} - {df[AGE_COL].max():.0f} years")
print()
print(f"All {df.shape[1]} columns are now numeric: {all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)}")

In [ ]:
# ── Feature block definitions (clinical vs histopathological) ───────────────
FEATURE_COLS      = [c for c in df.columns if c != TARGET_COL]
CLINICAL_ORDINAL  = FEATURE_COLS[:N_CLINICAL_ORDINAL]                       # attrs 1-11
HISTOPATHOLOGICAL = FEATURE_COLS[N_CLINICAL_ORDINAL:N_CLINICAL_ORDINAL + N_HISTOPATHOLOGICAL]  # attrs 12-33
CLINICAL          = CLINICAL_ORDINAL + [AGE_COL]                            # 12 clinical features
ORDINAL_COLS      = [c for c in FEATURE_COLS if c != AGE_COL]               # 33 ordinal features

assert len(CLINICAL) == 12,          f"expected 12 clinical, got {len(CLINICAL)}"
assert len(HISTOPATHOLOGICAL) == 22, f"expected 22 histopathological, got {len(HISTOPATHOLOGICAL)}"
assert len(FEATURE_COLS) == 34,      f"expected 34 predictors, got {len(FEATURE_COLS)}"
assert set(CLINICAL) | set(HISTOPATHOLOGICAL) == set(FEATURE_COLS), "blocks do not partition the features"

print(f"CLINICAL          ({len(CLINICAL):2d}) : {CLINICAL}")
print()
print(f"HISTOPATHOLOGICAL ({len(HISTOPATHOLOGICAL):2d}) : {HISTOPATHOLOGICAL}")
print()
print("Partition verified: 12 clinical + 22 histopathological = 34 predictors.")

In [ ]:
# ── First look at the data ──────────────────────────────────────────────────
print("First 5 rows (subset of columns for readability):")
display(df[CLINICAL + HISTOPATHOLOGICAL[:4] + [TARGET_COL]].head())

print("\nDataFrame info:")
df.info()

In [ ]:
# ── Descriptive statistics ──────────────────────────────────────────────────
print("Ordinal features (0-3 scale) — summary of summaries:")
desc_ord = df[ORDINAL_COLS].describe().T
display(desc_ord[["count", "mean", "std", "min", "50%", "max"]].round(2))

In [ ]:
print("Age (continuous):")
display(df[[AGE_COL]].describe().T.round(2))

print(f"\nTarget distribution ({TARGET_COL}):")
dist = df[TARGET_COL].value_counts().sort_index()
dist_tbl = pd.DataFrame({
    "Disease":    [CLASS_NAMES[c] for c in dist.index],
    "Count":      dist.values,
    "Percentage": (dist.values / len(df) * 100).round(2),
})
dist_tbl.index = dist.index
dist_tbl.index.name = "Class code"
display(dist_tbl)

imbalance_ratio = dist.max() / dist.min()
print(f"Imbalance ratio (largest : smallest) = {imbalance_ratio:.2f} : 1")

<a id="s11"></a>
# 11. Exploratory Data Analysis — **Task 1**

> **Task 1: Prepare a complete data analysis report on the given data.**

Every figure below is followed by an explicit interpretation. The aim is not to produce charts but to extract statements that later inform preprocessing choices, model selection, and the clinical guidance in Section 22.

## 11.1 Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df[TARGET_COL].value_counts().sort_index()
pct    = counts / counts.sum() * 100

bars = axes[0].bar([CLASS_SHORT[c] for c in counts.index], counts.values,
                   color=PALETTE, edgecolor="black", linewidth=0.7)
for b, n, p in zip(bars, counts.values, pct.values):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5,
                 f"{n}\n({p:.1f}%)", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_title("Class distribution (n = 366)")
axes[0].set_ylabel("Number of patients")
axes[0].set_ylim(0, counts.max() * 1.22)
axes[0].tick_params(axis="x", rotation=25)

axes[1].pie(counts.values, labels=[CLASS_SHORT[c] for c in counts.index],
            autopct="%1.1f%%", colors=PALETTE, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Proportional share of each disease")

plt.suptitle("Target distribution across the six erythemato-squamous diseases",
             fontsize=13, fontweight="bold")
plt.show()

print(f"Majority class : {CLASS_NAMES[counts.idxmax()]} ({counts.max()} patients, {pct.max():.1f}%)")
print(f"Minority class : {CLASS_NAMES[counts.idxmin()]} ({counts.min()} patients, {pct.min():.1f}%)")
print(f"Imbalance ratio: {counts.max() / counts.min():.2f} : 1")
print(f"Majority-class baseline accuracy (predict everything as the most common disease): {pct.max():.2f}%")

**Interpretation.** The distribution is **moderately imbalanced**, not severely so. Psoriasis dominates at 30.6% and pityriasis rubra pilaris — a genuinely rare disease in clinical practice — accounts for only 5.5% (20 patients). The 5.6:1 ratio is mild by the standards of medical datasets, but 20 samples is an absolute-count problem rather than a ratio problem: after an 80/20 split only about **16 training** and **4 test** samples of class 6 remain.

Two consequences follow immediately:

1. **Stratification is mandatory** for the train/test split and every CV fold. A random split could plausibly yield a fold with zero class-6 examples.
2. **Accuracy alone is an inadequate metric.** A model that ignored class 6 entirely would sacrifice at most 5.5% accuracy while being clinically useless for that disease. Macro-averaged precision, recall and F1 — which weight all six classes equally — are therefore the primary metrics throughout.

The majority-class baseline of **30.6%** is the floor any useful model must clear by a wide margin.

## 11.2 Age distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

age = df[AGE_COL].dropna()

axes[0].hist(age, bins=25, color="#4C72B0", edgecolor="black", alpha=0.85)
axes[0].axvline(age.mean(),   color="crimson",   ls="--", lw=2, label=f"mean = {age.mean():.1f}")
axes[0].axvline(age.median(), color="darkgreen", ls="-.", lw=2, label=f"median = {age.median():.1f}")
axes[0].set_title("Age distribution (all patients)")
axes[0].set_xlabel("Age (years)"); axes[0].set_ylabel("Frequency")
axes[0].legend(fontsize=9)

sns.boxplot(data=df, x=TARGET_COL, y=AGE_COL, hue=TARGET_COL,
            palette=PALETTE, ax=axes[1], legend=False)
axes[1].set_xticks(range(6)); axes[1].set_xticklabels(DISPLAY_NAMES, rotation=25, ha="right")
axes[1].set_title("Age by disease class"); axes[1].set_xlabel(""); axes[1].set_ylabel("Age (years)")

sns.violinplot(data=df, x=TARGET_COL, y=AGE_COL, hue=TARGET_COL,
               palette=PALETTE, ax=axes[2], legend=False, cut=0, inner="quartile")
axes[2].set_xticks(range(6)); axes[2].set_xticklabels(DISPLAY_NAMES, rotation=25, ha="right")
axes[2].set_title("Age density by disease class"); axes[2].set_xlabel(""); axes[2].set_ylabel("")

plt.suptitle("Age structure of the cohort", fontsize=13, fontweight="bold")
plt.show()

age_tbl = df.groupby(TARGET_COL)[AGE_COL].agg(["count", "mean", "median", "std", "min", "max"]).round(2)
age_tbl.insert(0, "Disease", [CLASS_NAMES[c] for c in age_tbl.index])
display(age_tbl)

**Interpretation — the single most striking univariate signal in the dataset.**

Five of the six diseases have essentially overlapping age profiles, with means between **35.3** and **40.0** years and standard deviations of 12–16 years. Age carries almost no information for separating those five.

**Pityriasis rubra pilaris (class 6) is the exception, and it is dramatic.** Its mean age is **10.25 years** with a standard deviation of **3.71** and a maximum of **22** — every single one of its 20 patients is a child or young adult, and the class is almost perfectly separated from the other five by age alone. This is consistent with the known juvenile-onset form of PRP.

This finding has a practical consequence that is easy to state and easy to miss: **age is a near-decisive discriminator for exactly the class with the fewest training examples.** It gives the models a strong handle on the hardest-to-learn disease, and it partially explains the surprisingly good class-6 performance seen in Section 19.

It also carries a **caution**. A 20-patient sample cannot establish that adult-onset PRP is rare — it can only show that *this cohort* contains none. A model trained here will likely under-predict PRP in an adult patient. This is recorded as a limitation in Section 24 and flagged to clinicians in Section 22.

## 11.3 Family history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

fh = df.groupby(TARGET_COL)["family_history"].agg(["sum", "mean", "count"])
fh["pct"] = fh["mean"] * 100

bars = axes[0].bar(DISPLAY_NAMES, fh["pct"], color=PALETTE, edgecolor="black", linewidth=0.7)
for b, s, c in zip(bars, fh["sum"], fh["count"]):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 1,
                 f"{int(s)}/{int(c)}", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_title("Positive family history by disease")
axes[0].set_ylabel("% of patients with family history = 1")
axes[0].set_ylim(0, max(fh["pct"].max() * 1.25, 10))
axes[0].tick_params(axis="x", rotation=25)

ct = pd.crosstab(df[TARGET_COL], df["family_history"])
ct.index = DISPLAY_NAMES
ct.columns = ["No family history (0)", "Family history (1)"]
ct.plot(kind="barh", stacked=True, ax=axes[1],
        color=["#B0B0B0", "#C44E52"], edgecolor="black", linewidth=0.6)
axes[1].set_title("Absolute counts by family history")
axes[1].set_xlabel("Number of patients"); axes[1].set_ylabel("")
axes[1].legend(fontsize=8, loc="lower right")

plt.suptitle("Family history as a diagnostic signal", fontsize=13, fontweight="bold")
plt.show()

fh_tbl = fh[["sum", "count", "pct"]].copy()
fh_tbl.columns = ["Positive family history", "Total patients", "% positive"]
fh_tbl.insert(0, "Disease", [CLASS_NAMES[c] for c in fh_tbl.index])
display(fh_tbl.round(2))

**Interpretation.** Family history is **sparse but highly specific** — a classic low-sensitivity / high-specificity marker.

- **Pityriasis rubra pilaris: 50.0%** (10 of 20). The highest rate in the cohort, consistent with the recognised familial form of PRP.
- **Psoriasis: 28.6%** (32 of 112). Psoriasis has a well-documented hereditary component, and the data reflects it.
- **Seborrheic dermatitis: 4.9%**, **lichen planus: 1.4%**.
- **Pityriasis rosea and chronic dermatitis: 0.0%** — not one positive family history between 101 patients.

The clinical reading is direct: a **positive** family history shifts the probability sharply towards psoriasis or PRP and effectively **rules out** pityriasis rosea and chronic dermatitis in this cohort. A **negative** family history, however, says very little — 71% of psoriasis patients here have no family history. The feature is useful when present and near-uninformative when absent, which is exactly why a model that weighs many features jointly extracts more from it than a simple rule would.

## 11.4 Clinical feature distributions

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 10))
for ax, col in zip(axes.ravel(), CLINICAL_ORDINAL + [AGE_COL]):
    if col == AGE_COL:
        ax.hist(df[col].dropna(), bins=20, color="#8172B3", edgecolor="black", alpha=0.85)
        ax.set_ylabel("count")
    else:
        vc = df[col].value_counts().sort_index()
        ax.bar(vc.index.astype(int), vc.values, color="#4C72B0", edgecolor="black", linewidth=0.6)
        ax.set_xticks([0, 1, 2, 3])
        ax.set_ylabel("count")
    ax.set_title(col.replace("_", " ")[:34], fontsize=9.5)
plt.suptitle("Distribution of the 12 CLINICAL features", fontsize=14, fontweight="bold", y=1.01)
plt.show()

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(17, 17))
for ax, col in zip(axes.ravel(), HISTOPATHOLOGICAL):
    vc = df[col].value_counts().sort_index()
    ax.bar(vc.index.astype(int), vc.values, color="#55A868", edgecolor="black", linewidth=0.6)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_title(col.replace("_", " ")[:34], fontsize=9)
    ax.set_ylabel("count", fontsize=8)
for ax in axes.ravel()[len(HISTOPATHOLOGICAL):]:
    ax.axis("off")
plt.suptitle("Distribution of the 22 HISTOPATHOLOGICAL features", fontsize=14, fontweight="bold", y=1.005)
plt.show()

zero_heavy = (df[ORDINAL_COLS] == 0).mean().sort_values(ascending=False)
print("Features that are zero in more than 70% of patients (sparse markers):")
for c, v in zero_heavy[zero_heavy > 0.70].items():
    print(f"  {v*100:5.1f}%  zero   {c}")

**Interpretation.** Two distributional patterns dominate, and they matter for modelling.

**Pattern 1 — universally present features carry little information.** `erythema` is graded ≥1 in nearly every patient; `scaling` is close behind. These are the *definitional* features of the erythemato-squamous group, so their presence is guaranteed and their discriminative value is low. This is a data-driven restatement of the clinical difficulty described in the problem brief: the features a clinician notices first are precisely the ones that do not separate the diseases. Section 16 confirms this quantitatively — `erythema` ranks near the *bottom* of the mutual-information ranking.

**Pattern 2 — sparse, high-grade markers are the informative ones.** Many histopathological features are zero in 70–90% of patients and jump to grade 2–3 in one specific disease. `follicular_horn_plug`, `perifollicular_parakeratosis`, `band-like_infiltrate`, `saw-tooth_appearance_of_retes` and `spongiform_pustule` all behave this way. Rarity is not noise here — these are **pathognomonic markers**: when present they are nearly decisive, and when absent they simply do not contribute.

For modelling, this argues in favour of methods that can exploit sharp conditional structure (trees, ensembles) while still benefiting from many weak signals combined linearly (regularised logistic regression). Both families are tested in Section 17.

## 11.5 Correlation structure

In [ ]:
corr = df[FEATURE_COLS + [TARGET_COL]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(15, 12.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, linecolor="white",
            cbar_kws={"shrink": 0.6, "label": "Pearson r"}, ax=ax)
ax.set_title("Correlation matrix — 34 attributes + target", fontsize=14, fontweight="bold", pad=14)
ax.tick_params(labelsize=7.5)
plt.show()

In [ ]:
# ── Strongly correlated feature pairs (potential multicollinearity) ─────────
cf = df[FEATURE_COLS].corr(numeric_only=True).abs()
pairs = (cf.where(np.triu(np.ones(cf.shape, dtype=bool), k=1))
           .stack().sort_values(ascending=False))

print("Feature pairs with |r| > 0.70 (multicollinearity candidates):")
strong = pairs[pairs > 0.70]
if len(strong) == 0:
    print("  none")
for (a, b), v in strong.items():
    print(f"  r = {v:.3f}   {a}  <->  {b}")

print(f"\nCount of pairs with |r| > 0.70 : {len(strong)}  (out of {len(pairs)} total pairs)")
print(f"Mean absolute inter-feature correlation: {pairs.mean():.3f}")

In [ ]:
# ── Association of each feature with the target ────────────────────────────
tgt_corr = df[FEATURE_COLS].corrwith(df[TARGET_COL]).sort_values()

fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#C44E52" if v < 0 else "#4C72B0" for v in tgt_corr.values]
ax.barh([c.replace("_", " ")[:38] for c in tgt_corr.index], tgt_corr.values,
        color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Pearson r with class code")
ax.set_title("Linear correlation of each attribute with the target code", fontsize=12, fontweight="bold")
ax.tick_params(labelsize=8.5)
plt.show()

print("NOTE: the target is a NOMINAL class code (1-6), not an ordered quantity.")
print("Pearson r against it is therefore only a rough screen, shown for completeness.")
print("Mutual information (Section 16) is the statistically appropriate measure and is used for feature selection.")

**Interpretation.** Three observations.

**Multicollinearity exists and is biologically expected.** The strongest correlations link features that describe the same underlying pathological process — for example `clubbing_of_the_rete_ridges`, `elongation_of_the_rete_ridges` and `thinning_of_the_suprapapillary_epidermis` form a tight cluster, since all three describe the epidermal architecture distortion characteristic of psoriasis. Similarly `saw-tooth_appearance_of_retes`, `band-like_infiltrate`, `vacuolisation_and_damage_of_basal_layer` and `melanin_incontinence` co-vary as the interface-dermatitis signature of lichen planus.

This correlation is **not** a defect to be engineered away. It is the data telling us that the 34 attributes are not 34 independent facts but roughly 5–6 *disease signatures* viewed through multiple lenses — a structure confirmed by the PCA analysis in Section 16.6, where 20 components capture 94% of variance.

**Consequence for model choice.** Multicollinearity destabilises the *coefficients* of an unregularised linear model (two correlated features can trade off arbitrarily) without necessarily hurting its *predictions*. This is a direct argument for L2-regularised logistic regression over plain logistic regression, and it is why `C` (the inverse regularisation strength) is tuned rather than left at its default in Section 18.

**Caution on the target correlation chart.** The class codes 1–6 are arbitrary labels — psoriasis is not "less than" lichen planus. A Pearson correlation against them is a crude screen only, shown here for completeness because the brief requests a correlation analysis. All actual feature selection in this notebook uses **mutual information**, which makes no ordinality assumption about the target.

## 11.6 Disease signature profiles

In [ ]:
# Mean feature grade per disease — the "signature" of each condition
profile = df.groupby(TARGET_COL)[ORDINAL_COLS].mean().T
profile.columns = DISPLAY_NAMES

fig, ax = plt.subplots(figsize=(10, 13))
sns.heatmap(profile, cmap="YlOrRd", vmin=0, vmax=3, linewidths=0.4, linecolor="white",
            annot=True, fmt=".1f", annot_kws={"size": 7.5},
            cbar_kws={"label": "mean grade (0-3)", "shrink": 0.5}, ax=ax)
ax.set_title("Mean feature grade by disease\n(the diagnostic signature of each condition)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel(""); ax.set_xlabel("")
ax.tick_params(labelsize=8)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.show()

In [ ]:
# ── For each disease: features most ELEVATED relative to the other five ────
print("=" * 78)
print("DISCRIMINATIVE FEATURE PROFILE PER DISEASE")
print("(mean grade in this disease minus mean grade across the other five)")
print("=" * 78)
for c in LABELS_ORDER:
    others = profile.drop(columns=CLASS_SHORT[c]).mean(axis=1)
    diff   = (profile[CLASS_SHORT[c]] - others).sort_values(ascending=False)
    n      = int((df[TARGET_COL] == c).sum())
    mean_age = df.loc[df[TARGET_COL] == c, AGE_COL].mean()
    print(f"\n[{c}] {CLASS_NAMES[c]}  (n = {n}, mean age = {mean_age:.1f} yrs)")
    print("   MOST ELEVATED:")
    for k, v in diff.head(5).items():
        print(f"      +{v:.2f}   {k}")
    print("   MOST SUPPRESSED:")
    for k, v in diff.tail(3).items():
        print(f"      {v:+.2f}   {k}")

**Interpretation — this table is the analytic foundation of the clinical guidance in Section 22.**

Each disease has an identifiable signature, and each signature is medically coherent:

| Disease | Elevated markers | Clinical reading |
|---|---|---|
| **Psoriasis** | clubbing of rete ridges (+2.08), thinning of suprapapillary epidermis (+2.05), elongation of rete ridges (+1.83), scalp involvement, knee/elbow involvement | The classic psoriasiform epidermal architecture plus extensor-surface distribution |
| **Seborrheic dermatitis** | spongiosis (+1.24), exocytosis (+0.81), PNL infiltrate (+0.80) | A spongiotic (eczematous) rather than psoriasiform pattern |
| **Lichen planus** | band-like infiltrate (+2.70), basal-layer vacuolisation (+2.30), saw-tooth retes (+2.29), polygonal papules (+2.28), melanin incontinence (+2.06) | Textbook interface dermatitis — five near-pathognomonic markers, the cleanest signature in the dataset |
| **Pityriasis rosea** | spongiosis (+1.00), Koebner phenomenon (+0.77), exocytosis (+0.63) | Mild spongiotic pattern with **no strongly elevated marker** — the weakest signature present |
| **Chronic dermatitis** | fibrosis of papillary dermis (+2.28), elongation of rete ridges (+1.38), itching (+0.72) | Dermal fibrosis is the end-stage scarring signature; itch drives the scratch cycle |
| **Pityriasis rubra pilaris** | follicular papules (+2.14), perifollicular parakeratosis (+2.05), follicular horn plug (+1.74), knee/elbow involvement, family history | An entirely **follicular** signature — a distinct pathological axis from all five others |

**The critical finding is what is *missing* from row 4.** Lichen planus, PRP, psoriasis and chronic dermatitis each have at least one marker elevated by more than +2.0 grades. **Pityriasis rosea's strongest marker is spongiosis at +1.00** — and spongiosis is also seborrheic dermatitis's top marker (+1.24). The two diseases share their most prominent feature.

This predicts, before any model is trained, that **seborrheic dermatitis ↔ pityriasis rosea will be the hardest pair to separate.** Section 19 confirms this exactly: that pair accounts for the large majority of all test-set errors across every one of the six models. The confusion is not a modelling failure — it is a real property of the data, and it is the finding most worth communicating to clinicians.

In [ ]:
# ── Zoom in on the predicted difficult pair: class 2 vs class 4 ────────────
pair = df[df[TARGET_COL].isin([2, 4])]
sep = (pair[pair[TARGET_COL] == 2][ORDINAL_COLS].mean()
       - pair[pair[TARGET_COL] == 4][ORDINAL_COLS].mean()).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
top = sep.head(10)
ax.barh([k.replace("_", " ")[:36] for k in top.index][::-1], top.values[::-1],
        color=["#DD8452" if v > 0 else "#C44E52" for v in top.values][::-1],
        edgecolor="black", linewidth=0.6)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("mean(Seborrheic dermatitis) − mean(Pityriasis rosea)")
ax.set_title("What separates the two most-confusable diseases?", fontsize=12, fontweight="bold")
plt.show()

print("Largest mean-grade gaps between seborrheic dermatitis and pityriasis rosea:")
for k, v in sep.head(6).items():
    higher = "Seborrheic dermatitis" if v > 0 else "Pityriasis rosea"
    print(f"  {abs(v):.2f} grades higher in {higher:<22s}  <-  {k}")

**Interpretation.** Even the *best* separators of this pair are weak. The two largest gaps are **`itching`** (1.15 grades higher in seborrheic dermatitis) and **`koebner_phenomenon`** (1.15 grades higher in pityriasis rosea), followed by `PNL_infiltrate` (0.96). Compare these to the +2.70 band-like infiltrate that marks lichen planus, and the difficulty is quantified: **no single feature reliably splits seborrheic dermatitis from pityriasis rosea.** Separating them requires combining several weak signals — which is precisely the task at which a multivariate model outperforms a diagnostic rule of thumb, and precisely why some residual error is unavoidable.

## 11.7 Feature relationships and separability

In [ ]:
# Pairplot on the highest-signal features (selected by the profile analysis above)
key_feats = ["clubbing_of_the_rete_ridges", "band-like_infiltrate",
             "fibrosis_of_the_papillary_dermis", "follicular_papules", AGE_COL]

pp_df = df[key_feats + [TARGET_COL]].copy()
pp_df["Disease"] = pp_df[TARGET_COL].map(CLASS_SHORT)

g = sns.pairplot(pp_df.drop(columns=[TARGET_COL]), hue="Disease",
                 palette=PALETTE, diag_kind="kde", height=1.9,
                 plot_kws={"alpha": 0.65, "s": 26, "edgecolor": "none"},
                 corner=True)
g.figure.suptitle("Pairwise separability of five high-signal attributes",
                  y=1.01, fontsize=13, fontweight="bold")
plt.show()

**Interpretation.** The pairplot shows that the classes are **largely separable in only a handful of well-chosen dimensions** — the clusters are visibly distinct along `clubbing_of_the_rete_ridges` (psoriasis), `band-like_infiltrate` (lichen planus), `fibrosis_of_the_papillary_dermis` (chronic dermatitis) and `follicular_papules` + low `Age` (PRP).

What the plot also makes visible is the flip side: **seborrheic dermatitis and pityriasis rosea sit near the origin on every one of these axes**, overlapping heavily. They are defined here by the *absence* of the other diseases' markers rather than by the presence of their own — a diagnosis of exclusion, which is a harder learning problem and a well-known clinical reality for both conditions.

This visual structure is what makes ~95%+ accuracy attainable on this dataset while also capping it well short of 100%.

<a id="s12"></a>
# 12. Data Quality Analysis — **Task 1**

A systematic audit across six dimensions. Each finding is followed by the decision it triggers.

In [ ]:
print("=" * 78)
print("DATA QUALITY AUDIT")
print("=" * 78)

# ── 1. MISSING VALUES ──────────────────────────────────────────────────────
print("\n[1] MISSING VALUES")
miss = df.isna().sum()
miss = miss[miss > 0]
if len(miss) == 0:
    print("    No missing values detected.")
else:
    for c, n in miss.items():
        print(f"    {c:<12s} : {n:3d} missing  ({n/len(df)*100:.2f}% of rows)")
    print(f"    Total missing cells : {int(miss.sum())} out of {df.size} ({miss.sum()/df.size*100:.3f}%)")
    print(f"    Rows with >=1 missing value : {df.isna().any(axis=1).sum()} ({df.isna().any(axis=1).mean()*100:.2f}%)")

# Is missingness related to the target? (MCAR vs MAR check)
if len(miss) > 0:
    print("\n    Missing-Age patients by disease class:")
    mrows = df[df[AGE_COL].isna()]
    for c in LABELS_ORDER:
        k = int((mrows[TARGET_COL] == c).sum())
        tot = int((df[TARGET_COL] == c).sum())
        if k:
            print(f"      class {c} ({CLASS_NAMES[c]:<26s}): {k}/{tot}")

In [ ]:
# ── 2. DUPLICATE ROWS ──────────────────────────────────────────────────────
print("\n[2] DUPLICATE ROWS")
dup_full = df.duplicated().sum()
dup_feat = df[FEATURE_COLS].duplicated().sum()
print(f"    Fully duplicated rows (features + label) : {dup_full}")
print(f"    Duplicated feature vectors (any label)   : {dup_feat}")
if dup_feat > dup_full:
    print("    -> WARNING: identical feature vectors carry conflicting labels (irreducible ambiguity).")
elif dup_full == 0:
    print("    -> No duplicates. Every row is a distinct patient record. No de-duplication needed.")

# ── 3. OUT-OF-RANGE / INVALID VALUES ───────────────────────────────────────
print("\n[3] OUT-OF-RANGE VALUES")
viol = {c: int((~df[c].dropna().between(0, 3)).sum()) for c in ORDINAL_COLS}
bad  = {c: v for c, v in viol.items() if v > 0}
print(f"    Ordinal features checked against documented 0-3 range : {len(ORDINAL_COLS)}")
print(f"    Violations found : {sum(bad.values()) if bad else 0}")
if bad:
    for c, v in bad.items():
        print(f"      {c}: {v} value(s) outside 0-3")
else:
    print("    -> All ordinal features conform to the documented encoding.")

fh_bad = int((~df['family_history'].dropna().isin([0, 1])).sum())
print(f"    'family_history' values outside {{0,1}} : {fh_bad}")

age_neg = int((df[AGE_COL].dropna() < 0).sum())
age_hi  = int((df[AGE_COL].dropna() > 120).sum())
print(f"    'Age' values < 0 : {age_neg}   |   'Age' values > 120 : {age_hi}")

In [ ]:
# ── 4. OUTLIERS IN AGE ─────────────────────────────────────────────────────
print("\n[4] OUTLIER ANALYSIS (Age — the only continuous attribute)")
a  = df[AGE_COL].dropna()
q1, q3 = a.quantile(0.25), a.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = a[(a < lo) | (a > hi)]
print(f"    Q1 = {q1:.1f} | Q3 = {q3:.1f} | IQR = {iqr:.1f}")
print(f"    Tukey fence: [{lo:.1f}, {hi:.1f}]")
print(f"    Global outliers : {len(out)}  -> {sorted(out.unique().tolist())}")

z = np.abs((a - a.mean()) / a.std())
print(f"    |z| > 3 count   : {int((z > 3).sum())}")

# Class-conditional outlier check - far more meaningful given the PRP age effect
print("\n    Class-conditional Tukey check (outlier WITHIN its own disease group):")
n_cond = 0
for c in LABELS_ORDER:
    s = df.loc[df[TARGET_COL] == c, AGE_COL].dropna()
    if len(s) < 5:
        continue
    cq1, cq3 = s.quantile(0.25), s.quantile(0.75)
    ciqr = cq3 - cq1
    co = s[(s < cq1 - 1.5 * ciqr) | (s > cq3 + 1.5 * ciqr)]
    n_cond += len(co)
    print(f"      class {c} ({CLASS_SHORT[c]:<11s}): {len(co)} outlier(s) {sorted(co.unique().astype(int).tolist()) if len(co) else ''}")
print(f"    Total class-conditional outliers: {n_cond}")

# Age == 0 inspection
zero_age = df[df[AGE_COL] == 0]
print(f"\n    Records with Age == 0 : {len(zero_age)}")
if len(zero_age):
    for i, r in zero_age.iterrows():
        print(f"      row {i}: class {int(r[TARGET_COL])} ({CLASS_NAMES[int(r[TARGET_COL])]})")

In [ ]:
# ── 5. CLASS IMBALANCE ─────────────────────────────────────────────────────
print("\n[5] CLASS IMBALANCE")
cnt = df[TARGET_COL].value_counts().sort_index()
print(f"    Counts        : {cnt.to_dict()}")
print(f"    Ratio max:min : {cnt.max()/cnt.min():.2f} : 1")
print(f"    Smallest class: {cnt.min()} samples ({CLASS_NAMES[cnt.idxmin()]})")
exp_tr = int(cnt.min() * (1 - TEST_SIZE))
exp_te = cnt.min() - exp_tr
print(f"    After an {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} split the smallest class yields "
      f"~{exp_tr} train and ~{exp_te} test samples.")
print(f"    Within a {CV_FOLDS}-fold CV on the training partition, each validation fold holds "
      f"~{exp_tr//CV_FOLDS} sample(s) of it.")
print("    -> Stratification is MANDATORY; macro-averaged metrics are REQUIRED.")

# Shannon evenness as a compact imbalance summary
p = cnt / cnt.sum()
evenness = float(-(p * np.log(p)).sum() / np.log(len(cnt)))
print(f"    Shannon evenness: {evenness:.4f}  (1.00 = perfectly balanced)")

# ── 6. CONSTANT / NEAR-CONSTANT FEATURES ───────────────────────────────────
print("\n[6] CONSTANT & LOW-VARIANCE FEATURES")
const = [c for c in FEATURE_COLS if df[c].nunique(dropna=True) <= 1]
print(f"    Constant features : {len(const)} {const}")
nz = df[ORDINAL_COLS].std().sort_values()
print("    Five lowest-variance ordinal features:")
for c, v in nz.head(5).items():
    print(f"      std = {v:.3f}   {c}")
print("    -> No feature is constant; none can be dropped on variance grounds alone.")

print("\n" + "=" * 78)
print("AUDIT COMPLETE")
print("=" * 78)

### 12.1 Data quality summary and decisions taken

| # | Check | Finding | Decision & reasoning |
|---|---|---|---|
| 1 | **Missing values** | **8 missing in `Age`** (2.19% of rows); 0 missing in all 34 other columns. Missing rows are spread across 5 of the 6 classes (1 each in classes 1–4, 4 in class 5) | **Median imputation, fitted inside the pipeline.** See reasoning below. |
| 2 | **Duplicate rows** | **0** fully duplicated rows; **0** duplicated feature vectors | No action. Every row is a distinct patient. Critically, this means no duplicate can leak across the train/test boundary. |
| 3 | **Out-of-range values** | **0 violations.** All 33 ordinal features lie within 0–3; `family_history` ∈ {0,1}; no negative or implausible ages | No cleaning required. The file is well-formed and matches the documented encoding exactly. |
| 4 | **Outliers (`Age`)** | No values outside the global Tukey fence. Class-conditional screening flags a small number of within-group extremes, notably one record with **`Age = 0`** (class 1, psoriasis) | **Retained.** See reasoning below. |
| 5 | **Class imbalance** | 5.60 : 1 (112 vs 20); Shannon evenness 0.9411 | **Stratified splitting + `f1_macro` tuning objective + `class_weight` tested empirically** (Section 16.7). |
| 6 | **Constant features** | **0** | No features dropped. |

### Why median imputation for `Age`

**The alternatives were considered and rejected:**

- *Drop the 8 rows.* This discards 2.19% of an already small dataset — including 4 of the 52 chronic-dermatitis patients (7.7% of that class). On 366 samples, throwing away real patient records to avoid imputing one value is a poor trade.
- *Mean imputation.* The age distribution is right-skewed and, more importantly, **bimodal** because of the paediatric PRP cluster (mean age 10.25). The mean is pulled by that cluster; the median is not. Median is the robust choice for skewed data.
- *Class-conditional imputation* (impute each patient's age with the median of *their* disease). This would be the most accurate imputation — but it **uses the target label to build a feature**, which is textbook target leakage. It would inflate cross-validation scores and collapse at inference time, when the label is exactly what is unknown. **Rejected on principle.**
- *Model-based imputation* (KNN / iterative). Defensible, but it adds a fitted component and tuning burden to recover 8 values out of 12,444 cells. Not worth the complexity or the additional overfitting surface.

**Decision: median imputation, fitted inside the `Pipeline`.** The median is computed on each training fold only and applied to the corresponding validation fold — so no information about held-out patients influences the imputed value. With only 8 cells affected, the choice has negligible effect on results; what matters is that it is done *without leakage*.

### Why the `Age = 0` record is retained

One psoriasis patient is recorded with `Age = 0`. Two readings are possible: a genuine infant (infantile psoriasis is uncommon but real), or a data-entry placeholder for "unknown". There is no way to distinguish these from the file alone.

It is retained because: (a) it is a single record out of 366 and cannot meaningfully distort a tree ensemble or a regularised linear model; (b) removing it based on a guess about its provenance would be an unjustified intervention; and (c) in a healthcare context the correct default is to preserve the record and **document the ambiguity**, which is done here. It is noted in the limitations (Section 24).

<a id="s13"></a>
# 13. EDA Interpretation — Consolidated Findings

Bringing Sections 11 and 12 together, seven findings shape everything that follows.

**Finding 1 — The dataset is clean, and that is itself worth stating.** No duplicates, no out-of-range values, no corrupted encodings, and a single missing field affecting 2.19% of rows. The data-cleaning burden is unusually light, which means effort belongs in *methodology* — leakage prevention, validation design, honest uncertainty — rather than in cleaning.

**Finding 2 — Age nearly solves class 6 on its own.** Pityriasis rubra pilaris has a mean age of 10.25 years against 35–40 for every other disease, with no patient over 22. The rarest class has the strongest univariate marker — a fortunate asymmetry that largely explains why models do not collapse on it. But this is a 20-patient observation, and it will not generalise to adult-onset PRP.

**Finding 3 — Family history is a high-specificity, low-sensitivity marker.** Present in 50% of PRP and 28.6% of psoriasis patients, but **0%** of pityriasis rosea and chronic dermatitis. Positive → strongly informative. Negative → almost uninformative.

**Finding 4 — The defining features of the disease group are the least useful for separating it.** `erythema` and `scaling` are present in nearly every patient and rank near the bottom of every importance measure. This is the statistical fingerprint of the clinical problem the brief describes.

**Finding 5 — Each disease has a coherent, medically interpretable signature**, dominated by sparse high-grade histopathological markers: rete-ridge architecture for psoriasis, band-like infiltrate for lichen planus, dermal fibrosis for chronic dermatitis, follicular plugging for PRP.

**Finding 6 — Seborrheic dermatitis and pityriasis rosea are the weak point, and it is predictable from the data.** Both are characterised by mild spongiosis and by the *absence* of other diseases' markers. Their best separators (`itching`, `koebner_phenomenon`) differ by only ~1.15 grades, against +2.70 for lichen planus's top marker. This pair drives the majority of model error in Section 19 — as the EDA predicted before any model was fit.

**Finding 7 — The 34 attributes represent roughly 5–6 underlying disease processes.** Correlated feature clusters map onto pathological mechanisms, and PCA confirms that 20 components retain 94% of the variance. This justifies L2 regularisation for the linear model and predicts that aggressive feature selection will hurt — a prediction tested and confirmed in Section 16.4.

<a id="s14"></a>
# 14. Data Preprocessing

## 14.1 Design principle: every fitted transformation lives inside a `Pipeline`

The central risk in a small-data project is **data leakage** — letting information from validation or test data influence the training process, which produces optimistic scores that evaporate in deployment.

The most common form in student projects is fitting a scaler or imputer on the *entire* dataset before splitting. The scaler's mean and standard deviation then encode information about the test patients, and the reported test score is no longer an honest out-of-sample estimate.

This notebook makes that mistake **structurally impossible** by placing the imputer and scaler inside a `sklearn.Pipeline`. When `GridSearchCV` or `cross_val_score` evaluates a fold, it calls `fit` on the training portion of that fold only; the validation portion sees only `transform`. The guarantee is enforced by the library, not by discipline.

## 14.2 The transformations applied

| Column group | Transformation | Reason |
|---|---|---|
| `Age` (1 column) | `SimpleImputer(strategy="median")` → `StandardScaler()` | Fills the 8 missing values robustly; rescales the 0–75 range so it does not dominate distance metrics |
| 33 ordinal features | `StandardScaler()` | Already on a common 0–3 scale, but standardising equalises their influence in distance- and margin-based models and speeds convergence of the logistic-regression solver |

## 14.3 Why the ordinal features are treated as numeric, not one-hot encoded

The 0–3 grades are **ordinal**: grade 3 means *more* of the feature than grade 1. Numeric treatment preserves that ordering and lets a model learn a monotone response. One-hot encoding would:

- **discard the ordering**, treating grades 1 and 3 as unrelated categories;
- **inflate dimensionality** from 34 features to over 130 — on a 366-row dataset, pushing $n/p$ from a already-tight 10.8 down to below 3, which is a recipe for overfitting.

Numeric treatment is the correct choice here on both statistical and information-theoretic grounds.

## 14.4 Target encoding

The target is already integer-coded 1–6. Scikit-learn handles arbitrary integer class labels natively, so **no `LabelEncoder` is required**. Re-encoding to 0–5 would add a translation layer and a fresh opportunity for an off-by-one bug in the confusion matrices. The codes are kept as-is and mapped to disease names only for display, via the `CLASS_NAMES` dictionary defined in the configuration cell.

In [ ]:
# ── Separate features and target ───────────────────────────────────────────
X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f"X : {X.shape}   (n = {X.shape[0]} patients, p = {X.shape[1]} attributes)")
print(f"y : {y.shape}   classes = {sorted(y.unique().tolist())}")
print(f"\nSamples-to-features ratio n/p = {X.shape[0] / X.shape[1]:.2f}")
print("A ratio near 10 is low. Regularisation and cross-validation are not optional here.")
print(f"\nAll feature columns numeric: {all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns)}")
print(f"Missing cells remaining in X (handled later by the pipeline's imputer): {int(X.isna().sum().sum())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PREPROCESSOR FACTORY
#  Returns an UNFITTED ColumnTransformer. It is fitted only inside a Pipeline,
#  on training folds only. This is the leakage guarantee.
# ════════════════════════════════════════════════════════════════════════════
def build_preprocessor(scale: bool = True) -> ColumnTransformer:
    '''
    scale=True  -> impute Age (median) + standardise everything.
                   Required for LogisticRegression, SVM and KNN.
    scale=False -> impute Age (median) only, leave raw scales.
                   Sufficient for tree-based models, which are scale-invariant.
    '''
    if scale:
        age_pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler()),
        ])
        return ColumnTransformer(
            [("age", age_pipe, [AGE_COL]),
             ("ord", StandardScaler(), ORDINAL_COLS)],
            remainder="drop",
        )
    return ColumnTransformer(
        [("age", SimpleImputer(strategy="median"), [AGE_COL]),
         ("ord", "passthrough", ORDINAL_COLS)],
        remainder="drop",
    )


# Column order produced by the transformer — needed to label importances correctly
TRANSFORMED_FEATURE_NAMES = [AGE_COL] + ORDINAL_COLS
assert len(TRANSFORMED_FEATURE_NAMES) == 34

_demo = build_preprocessor(scale=True)
print("Preprocessor (scale=True):")
print(_demo)
print(f"\nOutput column order: ['{TRANSFORMED_FEATURE_NAMES[0]}', '{TRANSFORMED_FEATURE_NAMES[1]}', ..., "
      f"'{TRANSFORMED_FEATURE_NAMES[-1]}']  ({len(TRANSFORMED_FEATURE_NAMES)} columns)")

<a id="s15"></a>
# 15. Train / Test Split Strategy

## 15.1 The split

```
                          366 patients
                                │
              ┌─────────────────┴─────────────────┐
              ▼                                   ▼
     TRAINING PARTITION  (292, 80%)      TEST SET  (74, 20%)
              │                                   │
    ┌─────────┴──────────┐                        │
    │ Stratified 5-fold  │                   LOCKED AWAY
    │ CV — used for ALL  │                 opened once, in
    │ EDA-driven choices,│                   Section 19
    │ feature selection  │
    │ and hyperparameter │
    │ tuning             │
    └────────────────────┘
```

**The contract:** the test set is not used for imputation statistics, scaling statistics, feature selection, hyperparameter search, model selection, or any decision whatsoever. It is evaluated exactly once, in Section 19, after all choices are frozen.

## 15.2 Why 80/20 + 5-fold CV rather than 70/15/15

The brief permits either. At $n = 366$:

| | 70/15/15 | **80/20 + 5-fold CV** |
|---|---|---|
| Training samples | 256 | **292** (+14%) |
| Validation basis | 55 patients, ~3 of class 6 | **All 292**, each used for validation once |
| Variance estimate | none (single split) | **standard deviation across 5 folds** |
| Test samples | 55 | 74 |

Selecting between models on a 55-patient validation set containing 3 examples of the rarest disease means selecting on noise. Cross-validation uses every training patient for validation exactly once, yields a variance estimate at no extra cost, and leaves a larger test set. For small-sample work this is the standard and defensible choice.

## 15.3 Stratification

Both the split and every CV fold preserve the six-class proportions. With 20 class-6 patients, an unstratified split has a non-trivial chance of producing folds with zero examples of it — which would make the fold's macro-F1 undefined and the resulting model selection meaningless.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("=" * 74)
print("TRAIN / TEST SPLIT")
print("=" * 74)
print(f"Training partition : {X_train.shape[0]:3d} patients ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Held-out test set  : {X_test.shape[0]:3d} patients ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Features           : {X_train.shape[1]}")

split_tbl = pd.DataFrame({
    "Disease":    [CLASS_NAMES[c] for c in LABELS_ORDER],
    "Full":       [int((y == c).sum()) for c in LABELS_ORDER],
    "Train":      [int((y_train == c).sum()) for c in LABELS_ORDER],
    "Test":       [int((y_test == c).sum()) for c in LABELS_ORDER],
    "Train %":    [round((y_train == c).mean() * 100, 1) for c in LABELS_ORDER],
    "Test %":     [round((y_test == c).mean() * 100, 1) for c in LABELS_ORDER],
}, index=LABELS_ORDER)
split_tbl.index.name = "Class"
print()
display(split_tbl)

# Verify stratification actually held
max_dev = max(abs((y_train == c).mean() - (y_test == c).mean()) * 100 for c in LABELS_ORDER)
print(f"Maximum train-vs-test proportion deviation: {max_dev:.2f} percentage points -> stratification verified.")
print(f"\nMissing Age values — train: {int(X_train[AGE_COL].isna().sum())} | test: {int(X_test[AGE_COL].isna().sum())}")
print("(These are imputed inside the pipeline using TRAINING-fold medians only.)")

# Cross-validation object used everywhere from here on
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print(f"\nCross-validation: StratifiedKFold(n_splits={CV_FOLDS}, shuffle=True, random_state={RANDOM_STATE})")
fold_sizes = [len(te) for _, te in cv.split(X_train, y_train)]
print(f"Validation-fold sizes: {fold_sizes}")

**Note on the test set's size.** With 74 patients and 6 classes, the rarest disease is represented by **4 test samples**. A single misclassification on that class moves its recall by 25 percentage points. This is not a flaw that better modelling can fix — it is an inherent property of evaluating a 6-class problem on 366 total records, and it is why Section 19 reports cross-validation results alongside test results, and why Section 20 declines to declare large differences between closely-matched models.

<a id="s16"></a>
# 16. Feature Engineering & Selection Analysis

For tabular data, "data augmentation" has no direct analogue (there is no meaningful way to rotate or crop a patient record). The equivalent analysis is **feature engineering and selection**: which transformations of the input space actually improve generalisation?

This section runs **six controlled experiments**, every one of them scored by 5-fold cross-validation **on the training partition only**. The test set is untouched.

> **Ground rule for this section:** no technique is adopted because it is standard practice. Each is adopted only if the measured cross-validation score justifies it. Where a technique does *not* help, that is reported as the result.

## 16.1 Experiment 1 — Does feature scaling matter?

**Hypothesis.** `Age` spans 0–75 while the 33 ordinal features span 0–3. In any Euclidean distance computation, Age's variance will swamp the other 33 features combined. Distance- and margin-based models (KNN, SVM) should therefore be badly hurt by unscaled input; tree-based models, which split on thresholds within each feature independently, should be completely unaffected.

In [ ]:
scaling_results = []
scaling_models = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "SVM (RBF)":           lambda: SVC(random_state=RANDOM_STATE),
    "KNN":                 lambda: KNeighborsClassifier(),
    "Decision Tree":       lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

print(f"Running {len(scaling_models)*2} cross-validated experiments...\n")
for name, mk in scaling_models.items():
    row = {"Model": name}
    for scale in (False, True):
        pipe = Pipeline([("prep", build_preprocessor(scale=scale)), ("model", mk())])
        sc   = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
        row["Unscaled" if not scale else "Scaled"] = sc.mean()
        row[("Unscaled" if not scale else "Scaled") + " std"] = sc.std()
    row["Delta"] = row["Scaled"] - row["Unscaled"]
    scaling_results.append(row)

scaling_df = pd.DataFrame(scaling_results)
display(scaling_df[["Model", "Unscaled", "Scaled", "Delta"]]
        .style.format({"Unscaled": "{:.4f}", "Scaled": "{:.4f}", "Delta": "{:+.4f}"})
        .background_gradient(subset=["Delta"], cmap="RdYlGn"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
xpos = np.arange(len(scaling_df)); w = 0.36
ax.bar(xpos - w/2, scaling_df["Unscaled"], w, yerr=scaling_df["Unscaled std"], capsize=4,
       label="Without scaling", color="#C44E52", edgecolor="black", linewidth=0.6)
ax.bar(xpos + w/2, scaling_df["Scaled"], w, yerr=scaling_df["Scaled std"], capsize=4,
       label="With StandardScaler", color="#55A868", edgecolor="black", linewidth=0.6)
for i, r in scaling_df.iterrows():
    if abs(r["Delta"]) > 0.01:
        ax.annotate(f"{r['Delta']:+.3f}", (i, max(r['Unscaled'], r['Scaled']) + 0.035),
                    ha="center", fontsize=9, fontweight="bold",
                    color="darkgreen" if r["Delta"] > 0 else "darkred")
ax.set_xticks(xpos); ax.set_xticklabels(scaling_df["Model"], rotation=18, ha="right")
ax.set_ylabel(f"{CV_FOLDS}-fold CV accuracy"); ax.set_ylim(0, 1.12)
ax.axhline(1.0, color="grey", ls=":", lw=1)
ax.set_title("Experiment 1 — impact of feature scaling", fontsize=12, fontweight="bold")
ax.legend(loc="lower right")
plt.show()

**Result — the hypothesis is confirmed, and the magnitude is larger than expected.**

| Model | Unscaled CV accuracy | Scaled CV accuracy | Change |
|---|---|---|---|
| **SVM (RBF)** | **0.6814** | **0.9760** | **+0.2946** |
| **KNN** | **0.8253** | **0.9691** | **+0.1438** |
| Logistic Regression | 0.9657 | 0.9760 | +0.0103 |
| Decision Tree | 0.9486 | 0.9486 | 0.0000 |
| Random Forest | 0.9760 | 0.9760 | 0.0000 |

**Interpretation.**

- **SVM gains 29.5 accuracy points.** Without scaling, the RBF kernel's $\exp(-\gamma\|x-x'\|^2)$ is dominated almost entirely by the Age difference; the 33 ordinal features become effectively invisible. Scaling transforms the SVM from the *worst* model tested to a joint-best one. This single result justifies the entire experiment.
- **KNN gains 14.4 points**, for the same reason — Euclidean neighbourhoods were being computed in Age-space.
- **Logistic Regression gains only 1.0 point.** A linear model can in principle compensate for scale by adjusting coefficients; the small gain comes from better-conditioned optimisation and from L2 regularisation penalising coefficients more fairly when features share a scale.
- **Tree models are exactly unchanged (delta = 0.0000).** This is not approximately zero — it is identically zero, and it is a useful correctness check: trees split on `feature <= threshold`, and a monotone rescaling maps every threshold to an equivalent one, so the fitted tree is unchanged.

**Decision adopted.** `StandardScaler` is applied inside the pipeline for **all** models. It is essential for SVM and KNN, mildly helpful for Logistic Regression, and provably harmless for trees — so a single uniform preprocessing path is used, which keeps the comparison in Section 19 clean.

## 16.2 Experiment 2 — Feature importance by mutual information

**Mutual information** measures how much knowing a feature's value reduces uncertainty about the diagnosis. Unlike Pearson correlation it captures non-linear and non-monotone relationships, and it makes no assumption that the class codes 1–6 are ordered — which matters, because they are not.

*Computed on the training partition only.*

In [ ]:
# Impute Age with the TRAINING median before computing MI (MI cannot accept NaN)
train_age_median = X_train[AGE_COL].median()
X_train_mi = X_train.fillna({AGE_COL: train_age_median})

mi_scores = mutual_info_classif(X_train_mi, y_train, random_state=RANDOM_STATE)
mi = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 11))
block_color = ["#4C72B0" if f in CLINICAL else "#55A868" for f in mi.index][::-1]
ax.barh([f.replace("_", " ")[:40] for f in mi.index][::-1], mi.values[::-1],
        color=block_color, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Mutual information with diagnosis (nats)")
ax.set_title("Experiment 2 — mutual information ranking of all 34 attributes",
             fontsize=12, fontweight="bold")
ax.tick_params(labelsize=8.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="#4C72B0", edgecolor="black", label="Clinical"),
                   Patch(facecolor="#55A868", edgecolor="black", label="Histopathological")],
          loc="lower right")
plt.show()

print("TOP 12 most informative attributes:")
for i, (k, v) in enumerate(mi.head(12).items(), 1):
    block = "clinical" if k in CLINICAL else "histopath."
    print(f"  {i:2d}. {v:.4f}  [{block:<11s}] {k}")
print("\nBOTTOM 6 least informative attributes:")
for k, v in mi.tail(6).items():
    block = "clinical" if k in CLINICAL else "histopath."
    print(f"      {v:.4f}  [{block:<11s}] {k}")

**Result.** The top of the ranking is dominated by histopathological architecture markers:

| Rank | Attribute | MI | Block |
|---|---|---|---|
| 1 | clubbing of the rete ridges | 0.6057 | histopathological |
| 2 | elongation of the rete ridges | 0.5810 | histopathological |
| 3 | thinning of the suprapapillary epidermis | 0.5391 | histopathological |
| 4 | band-like infiltrate | 0.5116 | histopathological |
| 5 | saw-tooth appearance of retes | 0.5029 | histopathological |
| 6 | vacuolisation and damage of basal layer | 0.4893 | histopathological |
| 7 | **polygonal papules** | 0.4860 | **clinical** |
| 8 | focal hypergranulosis | 0.4605 | histopathological |

And the bottom is the finding that matters most:

| Attribute | MI |
|---|---|
| eosinophils in the infiltrate | 0.1077 |
| spongiform pustule | 0.1023 |
| **erythema** | **0.0630** |
| acanthosis | 0.0488 |
| **inflammatory mononuclear infiltrate** | **0.0322** |

**Interpretation — the defining features are the least informative.** `erythema` ranks near the very bottom at 0.0630, roughly **one-tenth** the information content of the top marker. `inflammatory_monoluclear_inflitrate` is nearly uninformative at 0.0322.

This is the whole diagnostic problem expressed as a number. Erythema is the feature that *defines* membership of this disease group — so it is present in essentially every patient and therefore carries almost no information about *which* member they have. The features that a clinician notices first are the features that discriminate least.

**Clinical implication (carried into Section 22):** the presence of redness and scaling establishes that a patient belongs to the erythemato-squamous group. It contributes almost nothing to identifying *which* of the six. That work is done by the architectural markers — rete-ridge morphology, infiltrate pattern — and by a small number of high-yield clinical signs, chief among them `polygonal_papules` (rank 7), which is the highest-ranked clinical feature in the entire dataset.

## 16.3 Experiment 3 — Tree-based feature importance (a second, independent view)

Mutual information scores each feature **in isolation**. A Random Forest scores features **in context**, accounting for redundancy: if two features carry the same information, the forest will use one and assign the other low importance. Comparing the two rankings reveals which features are individually strong versus which are uniquely useful.

In [ ]:
rf_imp_pipe = Pipeline([
    ("prep",  build_preprocessor(scale=False)),
    ("model", RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
]).fit(X_train, y_train)

rf_imp = pd.Series(rf_imp_pipe.named_steps["model"].feature_importances_,
                   index=TRANSFORMED_FEATURE_NAMES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))

top15 = rf_imp.head(15)
axes[0].barh([f.replace("_", " ")[:34] for f in top15.index][::-1], top15.values[::-1],
             color="#8172B3", edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Gini importance")
axes[0].set_title("Random Forest importance (top 15)", fontsize=11, fontweight="bold")
axes[0].tick_params(labelsize=8.5)

# Rank agreement between the two methods
ranks = pd.DataFrame({"MI_rank": mi.rank(ascending=False), "RF_rank": rf_imp.rank(ascending=False)})
axes[1].scatter(ranks["MI_rank"], ranks["RF_rank"], s=55, alpha=0.75,
                color="#4C72B0", edgecolor="black", linewidth=0.5)
axes[1].plot([1, 34], [1, 34], "r--", lw=1.4, label="perfect agreement")
for f in list(mi.head(4).index) + list(rf_imp.head(4).index):
    axes[1].annotate(f.replace("_", " ")[:20], (ranks.loc[f, "MI_rank"], ranks.loc[f, "RF_rank"]),
                     fontsize=7, alpha=0.85, xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("Mutual information rank"); axes[1].set_ylabel("Random Forest rank")
axes[1].set_title("Do the two importance measures agree?", fontsize=11, fontweight="bold")
axes[1].legend()
plt.show()

spearman = ranks["MI_rank"].corr(ranks["RF_rank"], method="spearman")
print(f"Spearman rank correlation between MI and RF importance: {spearman:.3f}")
print("\nTop 10 by Random Forest importance:")
for i, (k, v) in enumerate(rf_imp.head(10).items(), 1):
    print(f"  {i:2d}. {v:.4f}   {k}")

**Result.** The Random Forest's top ranks are:

| Rank | Attribute | Gini importance |
|---|---|---|
| 1 | clubbing of the rete ridges | 0.0949 |
| 2 | thinning of the suprapapillary epidermis | 0.0825 |
| 3 | fibrosis of the papillary dermis | 0.0764 |
| 4 | **koebner phenomenon** | **0.0663** |
| 5 | elongation of the rete ridges | 0.0580 |
| 6 | spongiosis | 0.0562 |

**Interpretation — the disagreements are the interesting part.** The two measures agree broadly (they share `clubbing_of_the_rete_ridges` at rank 1), but two features shift substantially:

- **`koebner_phenomenon` rises from mid-table in the MI ranking to rank 4 in the forest.** In isolation it is a modest signal. In *context* it is valuable, because Section 11.6 showed it is one of only two features that separate pityriasis rosea from seborrheic dermatitis — the hardest pair. The forest rewards it for resolving an ambiguity no other feature resolves. Mutual information, scoring each feature alone, cannot see this.
- **`band-like_infiltrate` falls from MI rank 4 to forest rank 11.** It is a powerful lichen planus marker, but it is highly correlated with `saw-tooth_appearance_of_retes`, `vacuolisation_and_damage_of_basal_layer` and `melanin_incontinence` — all pointing at the same interface dermatitis. Once the forest has split on one of them, the others are largely redundant, and their importance is shared out.

**Methodological lesson.** A single feature-importance measure is not a reliable guide. MI answers "how informative is this feature by itself?"; tree importance answers "how much does this feature add given the others?". Both are reported here because they answer different questions, and the divergence between them is itself diagnostic of the dataset's redundancy structure.

## 16.4 Experiment 4 — Does feature selection help?

**Hypothesis.** With $n/p \approx 10.8$, conventional wisdom suggests trimming features should reduce overfitting and improve generalisation. Findings 5 and 7 from the EDA suggest otherwise: the informative markers are sparse and disease-specific, so removing any of them may remove the *only* evidence for a particular disease.

This experiment sweeps the number of retained features $k$ and measures the cross-validated effect.

In [ ]:
def mi_selector(X_arr, y_arr):
    return mutual_info_classif(X_arr, y_arr, random_state=RANDOM_STATE)

k_values = [5, 10, 15, 20, 25, 34]
fs_rows = []
print("Sweeping k (number of features retained by SelectKBest / mutual information)...\n")
for k in k_values:
    for mname, mk in [("Logistic Regression", lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
                      ("Random Forest",       lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))]:
        pipe = Pipeline([
            ("prep",   build_preprocessor(scale=True)),
            ("select", SelectKBest(score_func=mi_selector, k=k)),
            ("model",  mk()),
        ])
        s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
        fs_rows.append({"k": k, "Model": mname, "CV accuracy": s.mean(), "CV std": s.std()})

fs_df = pd.DataFrame(fs_rows)
display(fs_df.pivot(index="k", columns="Model", values="CV accuracy").round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
for mname, grp in fs_df.groupby("Model"):
    ax.errorbar(grp["k"], grp["CV accuracy"], yerr=grp["CV std"],
                marker="o", capsize=4, lw=2, markersize=7, label=mname)
ax.axvline(34, color="grey", ls="--", lw=1.4, label="all 34 features")
ax.set_xlabel("k — number of features retained")
ax.set_ylabel(f"{CV_FOLDS}-fold CV accuracy")
ax.set_title("Experiment 4 — effect of feature selection", fontsize=12, fontweight="bold")
ax.set_xticks(k_values)
ax.legend()
plt.show()

best = fs_df.loc[fs_df.groupby("Model")["CV accuracy"].idxmax()]
print("Best k per model:")
for _, r in best.iterrows():
    print(f"  {r['Model']:<20s}: k = {int(r['k']):2d}  ->  CV accuracy {r['CV accuracy']:.4f} +/- {r['CV std']:.4f}")

**Result — the hypothesis is rejected. Aggressive feature selection is clearly harmful.**

| k | Logistic Regression | Random Forest |
|---|---|---|
| 5 | 0.7809 | 0.7569 |
| 10 | 0.8183 | 0.8047 |
| 15 | 0.8800 | 0.8525 |
| 20 | 0.9691 | 0.9725 |
| **25** | **0.9863** | 0.9794 |
| 34 (all) | 0.9760 | 0.9760 |

**Interpretation.**

- **Cutting to 5 features costs roughly 20 accuracy points** (0.9760 → 0.7809 for logistic regression). Every feature removed takes genuine diagnostic evidence with it.
- **Performance climbs monotonically up to k ≈ 25**, then flattens. The curve's shape tells the story: there is no large block of useless features to discard. Between k = 15 and k = 20 the accuracy jumps by nearly 9 points — meaning features ranked 16th to 20th are individually decisive for at least one disease.
- **k = 25 gives the nominal best for logistic regression (0.9863 vs 0.9760 for all 34)** — an apparent gain of +1.03 points.

**Why that apparent gain is not acted upon.** The difference is +0.0103 accuracy, while the cross-validation standard deviation at that point is ±0.0129 — **the "improvement" is smaller than its own noise band.** On a 292-sample training set, a 1-point CV difference corresponds to roughly three patients changing fold assignment. Declaring a winner on that basis would be exactly the kind of noise-chasing this notebook is designed to avoid. Furthermore, selecting k on the same CV folds used to report the score introduces a mild optimistic bias in the reported figure.

**Decision adopted: retain all 34 features.** The evidence supports it directly — selection helps only marginally and only within noise, while aggressive selection is severely harmful. This is also the **biologically correct** conclusion: each disease is identified by its own sparse set of markers, so a feature that looks weak globally (because it is zero in 85% of patients) may be the single decisive piece of evidence for the 15% it applies to. `follicular_horn_plug` is uninformative for five diseases and near-decisive for PRP. Global feature ranking systematically undervalues exactly this kind of marker.

## 16.5 Experiment 5 — Clinical vs. histopathological features

This experiment has direct clinical meaning rather than merely technical interest: **how much diagnostic accuracy is available before a biopsy is taken?** The 12 clinical features are obtainable at the first consultation; the 22 histopathological features require a biopsy, laboratory processing and days of turnaround.

In [ ]:
subset_rows = []
for tag, cols in [("Clinical only (12)",        CLINICAL),
                  ("Histopathological only (22)", HISTOPATHOLOGICAL),
                  ("All features (34)",         FEATURE_COLS)]:
    age_part = [AGE_COL] if AGE_COL in cols else []
    ord_part = [c for c in cols if c != AGE_COL]
    steps = []
    if age_part:
        steps.append(("age", Pipeline([("impute", SimpleImputer(strategy="median")),
                                       ("scale", StandardScaler())]), age_part))
    steps.append(("ord", StandardScaler(), ord_part))
    ct = ColumnTransformer(steps, remainder="drop")

    for mname, mk in [("Logistic Regression", lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
                      ("Random Forest",       lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))]:
        pipe = Pipeline([("prep", ct), ("model", mk())])
        s = cross_val_score(pipe, X_train[cols], y_train, cv=cv, scoring="accuracy", n_jobs=-1)
        subset_rows.append({"Feature block": tag, "n features": len(cols), "Model": mname,
                            "CV accuracy": s.mean(), "CV std": s.std()})

subset_df = pd.DataFrame(subset_rows)
display(subset_df.pivot(index="Feature block", columns="Model", values="CV accuracy").round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
piv  = subset_df.pivot(index="Feature block", columns="Model", values="CV accuracy")
pivs = subset_df.pivot(index="Feature block", columns="Model", values="CV std")
order = ["Clinical only (12)", "Histopathological only (22)", "All features (34)"]
piv = piv.loc[order]; pivs = pivs.loc[order]
xp = np.arange(len(piv)); w = 0.36
for i, m in enumerate(piv.columns):
    ax.bar(xp + (i - 0.5) * w, piv[m], w, yerr=pivs[m], capsize=4,
           label=m, color=PALETTE[i], edgecolor="black", linewidth=0.6)
    for j, v in enumerate(piv[m]):
        ax.text(j + (i - 0.5) * w, v + 0.035, f"{v:.3f}", ha="center", fontsize=8.5, fontweight="bold")
ax.set_xticks(xp); ax.set_xticklabels(order, rotation=10)
ax.set_ylabel(f"{CV_FOLDS}-fold CV accuracy"); ax.set_ylim(0, 1.13)
ax.set_title("Experiment 5 — how much accuracy comes from bedside features alone?",
             fontsize=12, fontweight="bold")
ax.legend(loc="lower right")
plt.show()

**Result (Logistic Regression, 5-fold CV on the training partition):**

| Feature block | # features | CV accuracy | When available |
|---|---|---|---|
| **Clinical only** | 12 | **0.8766 ± 0.0354** | First consultation — no biopsy |
| **Histopathological only** | 22 | **0.9520 ± 0.0200** | After biopsy + laboratory turnaround |
| **All features** | 34 | **0.9760 ± 0.0085** | Full workup |

**Interpretation — this is the most clinically actionable result in the notebook.**

**Bedside features alone reach 87.7% accuracy.** Using only what a dermatologist can observe without cutting the patient — erythema, scaling, borders, itching, Koebner phenomenon, papule morphology, distribution, family history and age — a model can identify the correct disease in roughly seven of every eight patients. Against a 30.6% majority-class baseline, that is substantial diagnostic value available *on day one*.

**Histopathology adds 7.5 points on its own and 9.9 points when combined** (87.7% → 97.6%). The biopsy is not redundant; it resolves cases that clinical assessment leaves genuinely ambiguous.

**The practical inference.** A two-stage protocol is well supported by this data:

1. **Stage 1 — bedside.** Score the 12 clinical features. Where the model is confident and clinical judgement agrees, the diagnosis is likely settled and treatment can begin without delay.
2. **Stage 2 — biopsy, targeted.** Reserve biopsy for the cases where stage 1 is *uncertain* — in practice, largely the seborrheic dermatitis / pityriasis rosea pair and atypical presentations.

The benefit is real and two-directional: patients with clear-cut presentations avoid an invasive procedure and a multi-day wait, while laboratory capacity is concentrated on the cases where it changes the answer.

**The necessary caveat.** The 87.7% figure is an *in-sample cross-validation* estimate on 292 patients whose clinical features were graded by experienced dermatologists at a single centre. Grading the 0–3 scales consistently is itself a learned skill, and inter-rater variability in a different setting would lower this number. The finding motivates a prospective study; it does not by itself license changing biopsy policy.

## 16.6 Experiment 6 — Dimensionality reduction with PCA

The EDA (Finding 7) suggested the 34 attributes represent roughly 5–6 underlying disease processes. PCA tests that directly and asks whether a compressed representation performs as well.

In [ ]:
# Variance structure — fit on TRAINING data only
prep_only = Pipeline([("prep", build_preprocessor(scale=True))]).fit(X_train)
Z_train   = prep_only.transform(X_train)
pca_full  = PCA(random_state=RANDOM_STATE).fit(Z_train)
cum       = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8))
axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20],
            color="#4C72B0", edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Principal component"); axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title("Scree plot (first 20 components)", fontsize=11, fontweight="bold")
axes[0].set_xticks(range(1, 21, 2))

axes[1].plot(range(1, len(cum) + 1), cum, marker="o", markersize=4.5, lw=2, color="#C44E52")
for thr, col in [(0.80, "grey"), (0.90, "darkgreen"), (0.95, "navy")]:
    axes[1].axhline(thr, color=col, ls="--", lw=1.2)
    k_need = int(np.argmax(cum >= thr) + 1)
    axes[1].annotate(f"{int(thr*100)}% @ {k_need} PCs", (k_need, thr), fontsize=8.5,
                     color=col, xytext=(5, -13), textcoords="offset points")
axes[1].set_xlabel("Number of components"); axes[1].set_ylabel("Cumulative explained variance")
axes[1].set_title("Cumulative variance explained", fontsize=11, fontweight="bold")
plt.show()

print(f"Variance captured by  5 components : {cum[4]*100:.2f}%")
print(f"Variance captured by 10 components : {cum[9]*100:.2f}%")
print(f"Variance captured by 20 components : {cum[19]*100:.2f}%")

In [ ]:
pca_rows = []
for n in [5, 10, 15, 20]:
    pipe = Pipeline([("prep",  build_preprocessor(scale=True)),
                     ("pca",   PCA(n_components=n, random_state=RANDOM_STATE)),
                     ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    pca_rows.append({"n components": n, "CV accuracy": s.mean(), "CV std": s.std()})

base = Pipeline([("prep", build_preprocessor(scale=True)),
                 ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])
bs = cross_val_score(base, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
pca_rows.append({"n components": 34, "CV accuracy": bs.mean(), "CV std": bs.std()})

pca_df = pd.DataFrame(pca_rows)
display(pca_df.round(4))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.errorbar(pca_df["n components"], pca_df["CV accuracy"], yerr=pca_df["CV std"],
            marker="o", markersize=8, capsize=5, lw=2, color="#8172B3")
ax.axhline(bs.mean(), color="grey", ls="--", lw=1.4, label=f"no PCA (all 34) = {bs.mean():.4f}")
ax.set_xlabel("Number of principal components"); ax.set_ylabel(f"{CV_FOLDS}-fold CV accuracy")
ax.set_title("Experiment 6 — PCA vs. raw features (Logistic Regression)", fontsize=12, fontweight="bold")
ax.legend()
plt.show()

**Result.**

| Components | CV accuracy | Variance retained |
|---|---|---|
| 5 | 0.9553 ± 0.0177 | ~63% |
| 10 | 0.9691 ± 0.0070 | 78.05% |
| 15 | 0.9725 ± 0.0139 | ~88% |
| 20 | 0.9760 ± 0.0085 | 94.41% |
| **34 (no PCA)** | **0.9760 ± 0.0085** | 100% |

**Interpretation.** The EDA hypothesis is quantitatively confirmed: **20 components retain 94.4% of the variance**, so the 34 attributes really do contain substantial redundancy and describe roughly a dozen-and-a-half independent axes of variation.

But PCA **never beats the raw features** — at 20 components it exactly matches them (0.9760 both), and below that it is strictly worse. Compression costs information without buying accuracy.

**Decision: PCA is not used.** Three reasons, in order of weight:

1. **No measured benefit.** At best it ties; there is no performance argument for it.
2. **It destroys interpretability, which is the binding constraint here.** A principal component is a weighted mixture of all 34 attributes. Telling a dermatologist that "PC3 was elevated" is not clinically actionable; telling them "band-like infiltrate at grade 3 drove this prediction" is. In a healthcare decision-support context, this alone would rule PCA out even if it had produced a modest gain.
3. **It adds a fitted component** that must be maintained, versioned and re-fitted, for no return.

**Where PCA would be justified:** far higher dimensionality (hundreds or thousands of features), severe computational constraints, or a downstream algorithm that genuinely requires decorrelated inputs. None applies to a 34-feature clinical dataset.

## 16.7 Experiment 7 — Handling class imbalance

The 5.6:1 imbalance is moderate. Two families of remedy exist.

**Synthetic oversampling (SMOTE)** generates new minority-class samples by interpolating between a minority point and its $k$ nearest minority neighbours. For this dataset there is a specific and serious objection: **the features are ordinal integers on a 0–3 scale.** Interpolating between a patient with `band_like_infiltrate = 1` and one with `= 3` produces a synthetic patient with grade 2.37 — a value that cannot exist in any real biopsy report. More importantly, with only 16 class-6 patients in the training partition, SMOTE would be interpolating within a very sparse neighbourhood, largely manufacturing points along the lines joining a handful of real patients and risking the creation of a spurious, overly smooth class region.

**Cost-sensitive learning (`class_weight='balanced'`)** instead re-weights the loss function so that errors on rare classes are penalised in inverse proportion to their frequency. It invents no data, adds no hyperparameters, and is supported natively by the models used here. It is the more defensible option for ordinal clinical data at this sample size.

The experiment below measures the effect on **macro-F1**, the metric that actually rewards rare-class performance.

In [ ]:
cw_rows = []
cw_models = {
    "Logistic Regression": lambda w: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE, class_weight=w),
    "Random Forest":       lambda w: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight=w),
    "SVM (RBF)":           lambda w: SVC(random_state=RANDOM_STATE, class_weight=w),
}
for name, mk in cw_models.items():
    row = {"Model": name}
    for w, tag in [(None, "No weighting"), ("balanced", "class_weight='balanced'")]:
        pipe = Pipeline([("prep", build_preprocessor(scale=True)), ("model", mk(w))])
        s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
        row[tag] = s.mean(); row[tag + " std"] = s.std()
    row["Delta"] = row["class_weight='balanced'"] - row["No weighting"]
    cw_rows.append(row)

cw_df = pd.DataFrame(cw_rows)
display(cw_df[["Model", "No weighting", "class_weight='balanced'", "Delta"]]
        .style.format({"No weighting": "{:.4f}", "class_weight='balanced'": "{:.4f}", "Delta": "{:+.4f}"}))

print(f"\nSMOTE available in this environment: {IMBLEARN_AVAILABLE}")
if not IMBLEARN_AVAILABLE:
    print("(imbalanced-learn is not installed. It is not required — class_weight is used instead,")
    print(" for the ordinal-data reasons argued above. Install with: pip install imbalanced-learn)")

**Result (macro-F1, 5-fold CV):**

| Model | No weighting | `class_weight='balanced'` | Change |
|---|---|---|---|
| Logistic Regression | 0.9730 | 0.9768 | +0.0038 |
| Random Forest | 0.9704 | 0.9730 | +0.0026 |
| SVM (RBF) | 0.9730 | 0.9768 | +0.0038 |

**Interpretation.** Balanced class weighting produces a **small, consistent, positive** effect — improvements of +0.003 to +0.004 macro-F1 across all three model families. The direction is consistent, which is mildly reassuring, but every one of these deltas is well inside its cross-validation standard deviation (±0.014 or more). **This is not statistically meaningful evidence of benefit.**

**Why the effect is so small.** A 5.6:1 imbalance is simply not severe. All six classes have enough representation for the models to learn them, and Finding 2 supplies an additional reason: the rarest class (PRP) has the *strongest* univariate marker in the dataset (age ≈ 10), so it is intrinsically easy despite having the fewest samples. The usual failure mode of imbalanced learning — the minority class being ignored — does not arise here.

**Decision adopted: `class_weight` is included in the hyperparameter grid rather than fixed by hand.** Section 18 searches `[None, 'balanced']` for Logistic Regression, Random Forest and SVM and lets cross-validated macro-F1 decide per model. This is more honest than either imposing balancing on the assumption that imbalance is always bad, or dismissing it on the assumption that 5.6:1 is always fine.

## 16.8 Summary of feature-engineering decisions

| Technique | Measured effect | Adopted? | Reason |
|---|---|---|---|
| **StandardScaler** | SVM **+0.2946**, KNN **+0.1438**, LR +0.0103, trees 0.0000 | **YES** | Essential for margin/distance models, harmless for trees |
| **Median imputation of `Age`** | Recovers 8 rows (2.19%) | **YES** | Robust to the bimodal age distribution; leak-free inside the pipeline |
| Feature selection (SelectKBest / MI) | k=25 gives +0.0103 (< noise ±0.0129); k=5 costs **−0.1951** | **NO** | Gain is within noise; aggressive selection is severely harmful |
| PCA | Never exceeds raw features; ties at 20 components | **NO** | No benefit, and destroys the interpretability that healthcare requires |
| `class_weight='balanced'` | +0.003 to +0.004 macro-F1, consistent but within noise | **TUNED** | Left in the grid for cross-validation to decide per model |
| SMOTE | Not applied | **NO** | Would produce non-integer grades on an ordinal 0–3 scale; only 16 training samples in the rarest class |
| One-hot encoding of ordinal features | Not applied | **NO** | Discards ordering; would inflate $p$ from 34 to >130 on 366 rows |

**The headline conclusion of this section is a negative result, and it is worth stating plainly:** of six candidate techniques, only **scaling** and **imputation** earned their place on measured evidence. Feature selection, PCA and explicit resampling were all tested and all rejected because the data did not support them. A project that applied all six because they appear in every tutorial would have produced a more complicated pipeline and no better model — and, in the case of PCA, a model no dermatologist could interrogate.

<a id="s17"></a>
# 17. Model Development — **Task 2**

> **Task 2: Create a predictive model using machine learning techniques to predict the various classes of skin disease.**

Six models are developed, spanning four distinct algorithmic families. The brief asks for at least 3–4; six are used because the families make genuinely different assumptions about the data, and the comparison between them is informative in its own right.

## 17.1 Why these six, and why no deep learning

| Family | Model | Core assumption | Why it suits this problem |
|---|---|---|---|
| **Linear** | Logistic Regression | Classes are separable by hyperplanes in feature space | Ordinal grades combine near-additively into a diagnosis; fully interpretable coefficients; L2 regularisation handles the multicollinearity found in Section 11.5 |
| **Single tree** | Decision Tree | Diagnosis follows nested threshold rules | Directly mirrors clinical decision-tree reasoning; the only model whose full logic can be printed and read by a clinician |
| **Ensemble** | Random Forest | Many decorrelated trees average away individual variance | Robust to the small sample; handles feature interactions and correlated features gracefully; provides importance rankings |
| **Ensemble (boosting)** | Gradient Boosting | Sequentially correct the previous model's errors | Strong on tabular data; shallow trees keep capacity in check on 292 samples |
| **Kernel / margin** | SVM (RBF) | A maximum-margin boundary exists in a transformed space | Margin maximisation is well-suited to small $n$; the RBF kernel captures non-linear feature interactions |
| **Instance-based** | KNN | Similar patients have the same diagnosis | The closest analogue to how clinicians reason from remembered cases; a useful non-parametric benchmark |

**Deep learning is deliberately excluded.** A neural network on 292 training samples and 34 tabular features would have more parameters than data points in any non-trivial architecture, would overfit immediately, would offer no mechanism for the interpretability this domain requires, and — on the consistent evidence of the tabular-learning literature — would not outperform gradient-boosted trees. Using one would be complexity without benefit, and the brief explicitly asks to avoid unnecessary complexity.

## 17.2 Hyperparameter grids and their rationale

Every grid is deliberately **modest**. On 292 samples, a large grid searched by 5-fold CV performs so many comparisons that the best score is partly a selection artefact — the winner is the configuration that got lucky on those particular folds. Small, well-motivated grids reduce this optimistic bias.

| Model | Parameter | Values searched | Why |
|---|---|---|---|
| **Logistic Regression** | `C` | 0.01, 0.1, 1, 10 | Inverse L2 strength across four orders of magnitude. Low `C` = heavy shrinkage, which the 10.8:1 n/p ratio and correlated features may require |
| | `class_weight` | None, balanced | Tested empirically per Section 16.7 |
| **Decision Tree** | `max_depth` | 3, 5, 7, None | Depth is the primary overfitting control. Depth 3–5 is also the range a clinician could actually read |
| | `min_samples_leaf` | 1, 3, 5 | Prevents leaves built on one or two patients |
| | `criterion` | gini, entropy | Standard impurity alternatives |
| **Random Forest** | `n_estimators` | 200, 500 | More trees only reduce variance; 200 is generally sufficient at this scale |
| | `max_depth` | None, 10 | Unlimited depth is usually fine for RF (bagging controls variance); 10 tests whether extra restraint helps |
| | `min_samples_leaf` | 1, 2 | Mild additional regularisation |
| | `class_weight` | None, balanced | Per Section 16.7 |
| **SVM (RBF)** | `C` | 0.1, 1, 10, 100 | Margin-softness / error trade-off |
| | `gamma` | scale, 0.01, 0.1 | Kernel width — how local the decision boundary is |
| | `class_weight` | None, balanced | Per Section 16.7 |
| **KNN** | `n_neighbors` | 3, 5, 7, 9, 11 | With ~16–89 training samples per class, k > 11 would start averaging across class boundaries |
| | `weights` | uniform, distance | Whether closer patients count more |
| | `p` | 1, 2 | Manhattan vs. Euclidean. Manhattan is often better for ordinal features, since it sums per-feature grade differences directly |
| **Gradient Boosting** | `n_estimators` | 100, 200 | Boosting rounds |
| | `learning_rate` | 0.05, 0.1 | Shrinkage per round; lower values trade compute for generalisation |
| | `max_depth` | 2, 3 | **Deliberately shallow.** Boosting with deep trees on 292 samples overfits badly; depth 2–3 stumps are the standard safe choice |

## 17.3 Why `f1_macro` is the tuning objective

Hyperparameters are selected to maximise **macro-averaged F1**, not accuracy.

Accuracy weights every *patient* equally, so it is dominated by the majority classes. A model could drop pityriasis rubra pilaris entirely and lose only ~5.5% accuracy. Macro-F1 weights every *disease* equally — class 6 contributes one-sixth of the score regardless of having one-eighteenth of the patients. It also combines precision and recall, both of which matter clinically: precision because a false positive means unnecessary treatment, recall because a false negative means a missed diagnosis.

There is no separate "loss function and optimizer" choice in the deep-learning sense here. Each model optimises its own internal objective (cross-entropy for logistic regression and gradient boosting, Gini or entropy impurity for trees, hinge loss for SVM, none for KNN); `f1_macro` is the **external model-selection criterion** applied uniformly across all six so that the comparison in Section 19 is like-for-like.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  MODEL ZOO — estimator + hyperparameter grid
#  All parameter keys are prefixed 'model__' to address the Pipeline step.
# ════════════════════════════════════════════════════════════════════════════
MODEL_ZOO = {
    "Logistic Regression": {
        "estimator": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        "grid": {
            "model__C":            [0.01, 0.1, 1, 10],
            "model__class_weight": [None, "balanced"],
        },
        "family": "Linear",
        "note":   "Interpretable baseline; L2-regularised; per-class coefficients readable by clinicians.",
    },
    "Decision Tree": {
        "estimator": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "grid": {
            "model__max_depth":        [3, 5, 7, None],
            "model__min_samples_leaf": [1, 3, 5],
            "model__criterion":        ["gini", "entropy"],
        },
        "family": "Tree",
        "note":   "Fully transparent rule set; the only model whose complete logic can be printed.",
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        "grid": {
            "model__n_estimators":     [200, 500],
            "model__max_depth":        [None, 10],
            "model__min_samples_leaf": [1, 2],
            "model__class_weight":     [None, "balanced"],
        },
        "family": "Ensemble (bagging)",
        "note":   "Variance reduction by averaging decorrelated trees; robust on small samples.",
    },
    "SVM (RBF)": {
        "estimator": SVC(random_state=RANDOM_STATE, probability=True),
        "grid": {
            "model__C":            [0.1, 1, 10, 100],
            "model__gamma":        ["scale", 0.01, 0.1],
            "model__class_weight": [None, "balanced"],
        },
        "family": "Kernel / margin",
        "note":   "Maximum-margin boundary in an implicit high-dimensional space; strong at small n.",
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "grid": {
            "model__n_neighbors": [3, 5, 7, 9, 11],
            "model__weights":     ["uniform", "distance"],
            "model__p":           [1, 2],
        },
        "family": "Instance-based",
        "note":   "Non-parametric; predicts from the most similar past patients.",
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "grid": {
            "model__n_estimators":  [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth":     [2, 3],
        },
        "family": "Ensemble (boosting)",
        "note":   "Sequential error correction with deliberately shallow trees to limit capacity.",
    },
}

print(f"{'MODEL':<22s} {'FAMILY':<20s} {'GRID SIZE':>10s}")
print("-" * 56)
total = 0
for name, spec in MODEL_ZOO.items():
    size = int(np.prod([len(v) for v in spec["grid"].values()]))
    total += size
    print(f"{name:<22s} {spec['family']:<20s} {size:>10d}")
print("-" * 56)
print(f"{'TOTAL':<22s} {'':<20s} {total:>10d} configurations")
print(f"Each evaluated with {CV_FOLDS}-fold CV -> {total * CV_FOLDS} model fits during tuning.")

<a id="s18"></a>
# 18. Model Training with Cross-Validation

Each model is tuned by `GridSearchCV` over stratified 5-fold cross-validation **on the 292-patient training partition only**, optimising `f1_macro`. The test set plays no part.

For each model the following are recorded:
- the best hyperparameter configuration,
- the cross-validated macro-F1 (mean and standard deviation) at that configuration,
- the cross-validated accuracy of the refitted best estimator,
- training-set accuracy, to expose overfitting,
- wall-clock fit time and inference time.

### On early stopping and callbacks

The brief asks about callbacks and early stopping. These are constructs of iterative gradient-descent training and apply to only one model here — Gradient Boosting, where the number of boosting rounds is the iteration count. Rather than use a validation-based early-stopping callback (which on 292 samples would carve out a small and noisy validation set), **the number of rounds is treated as a hyperparameter and selected by the same 5-fold cross-validation as everything else** (`n_estimators` ∈ {100, 200}, with `learning_rate` and `max_depth` co-tuned). This is a stronger form of the same regularisation: it uses all the training data for the decision instead of a single arbitrary holdout. The other five models are not trained iteratively and have no equivalent mechanism.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  TRAINING LOOP
#  Runtime is typically 1-3 minutes on a standard laptop.
# ════════════════════════════════════════════════════════════════════════════
fitted_models   = {}
search_objects  = {}
training_log    = []

print("=" * 78)
print(f"TUNING {len(MODEL_ZOO)} MODELS  |  {CV_FOLDS}-fold stratified CV  |  objective = {SCORING}")
print("=" * 78)

for name, spec in MODEL_ZOO.items():
    pipe = Pipeline([
        ("prep",  build_preprocessor(scale=True)),
        ("model", spec["estimator"]),
    ])

    t0 = time.perf_counter()
    search = GridSearchCV(
        estimator=pipe,
        param_grid=spec["grid"],
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    # CV accuracy of the refitted best estimator (same folds, different metric)
    cv_acc = cross_val_score(search.best_estimator_, X_train, y_train,
                             cv=cv, scoring="accuracy", n_jobs=-1)

    t0 = time.perf_counter()
    _ = search.best_estimator_.predict(X_test)
    infer_time_ms = (time.perf_counter() - t0) * 1000

    train_acc = accuracy_score(y_train, search.best_estimator_.predict(X_train))

    fitted_models[name]  = search.best_estimator_
    search_objects[name] = search
    training_log.append({
        "Model":          name,
        "Family":         spec["family"],
        "CV F1-macro":    search.best_score_,
        "CV Acc":         cv_acc.mean(),
        "CV Acc std":     cv_acc.std(),
        "Train Acc":      train_acc,
        "Fit time (s)":   fit_time,
        "Infer (ms)":     infer_time_ms,
        "Best params":    {k.replace("model__", ""): v for k, v in search.best_params_.items()},
    })

    print(f"\n{name}")
    clean_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    print(f"   best params      : {clean_params}")
    print(f"   CV f1_macro      : {search.best_score_:.4f}")
    print(f"   CV accuracy      : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
    print(f"   Train accuracy   : {train_acc:.4f}")
    print(f"   Tuning wall time : {fit_time:.2f} s")

print("\n" + "=" * 78)
print("ALL MODELS TUNED.")
print("=" * 78)

In [ ]:
train_df = pd.DataFrame(training_log)
print("CROSS-VALIDATION SUMMARY (training partition only — test set untouched)\n")
display(train_df[["Model", "Family", "CV F1-macro", "CV Acc", "CV Acc std", "Train Acc", "Fit time (s)"]]
        .sort_values("CV F1-macro", ascending=False)
        .style.format({"CV F1-macro": "{:.4f}", "CV Acc": "{:.4f}", "CV Acc std": "{:.4f}",
                       "Train Acc": "{:.4f}", "Fit time (s)": "{:.2f}"})
        .background_gradient(subset=["CV F1-macro"], cmap="Greens")
        .hide(axis="index"))

print("\nSelected hyperparameters:")
for r in training_log:
    print(f"  {r['Model']:<22s}: {r['Best params']}")

In [ ]:
# ── Overfitting check: training accuracy vs cross-validated accuracy ───────
fig, ax = plt.subplots(figsize=(10.5, 5))
td = train_df.sort_values("CV Acc", ascending=False)
xp = np.arange(len(td)); w = 0.36
ax.bar(xp - w/2, td["Train Acc"], w, label="Training accuracy",
       color="#DD8452", edgecolor="black", linewidth=0.6)
ax.bar(xp + w/2, td["CV Acc"], w, yerr=td["CV Acc std"], capsize=4,
       label=f"{CV_FOLDS}-fold CV accuracy", color="#4C72B0", edgecolor="black", linewidth=0.6)
for i, (_, r) in enumerate(td.iterrows()):
    gap = r["Train Acc"] - r["CV Acc"]
    ax.annotate(f"gap\n{gap:+.3f}", (i, max(r["Train Acc"], r["CV Acc"]) + 0.03),
                ha="center", fontsize=7.5, fontweight="bold",
                color="darkred" if gap > 0.05 else "grey")
ax.set_xticks(xp); ax.set_xticklabels(td["Model"], rotation=18, ha="right")
ax.set_ylim(0, 1.16); ax.axhline(1.0, color="grey", ls=":", lw=1)
ax.set_ylabel("Accuracy")
ax.set_title("Overfitting diagnostic — training vs cross-validated accuracy",
             fontsize=12, fontweight="bold")
ax.legend(loc="lower right")
plt.show()

print("Generalisation gap (train accuracy - CV accuracy):")
for _, r in td.iterrows():
    gap = r["Train Acc"] - r["CV Acc"]
    flag = "  <- memorises training data" if gap > 0.05 else ""
    print(f"  {r['Model']:<22s}: {gap:+.4f}{flag}")

**Interpretation of the overfitting diagnostic.**

Logistic Regression, Random Forest and Gradient Boosting all reach **100% training accuracy** while cross-validating around 97–98%. A generalisation gap of roughly 2–3 points is *expected and acceptable* here — with 34 features and 292 samples, these model classes have ample capacity to fit the training data exactly, and the question is not whether they memorise but whether they generalise. A CV accuracy of 0.9794 says they do.

The gap is worth watching for two reasons. First, it confirms that regularisation is doing real work: without the tuned `C` for logistic regression and the tuned `min_samples_leaf` for the forest, the gap would be wider. Second, it is the reason the test-set evaluation in Section 19 is non-negotiable — cross-validation on the training partition is a good estimate, but the models have now been *selected* using those folds, so the CV score carries a mild optimistic bias. Only the untouched test set gives a clean number.

The **SVM's** training accuracy of 0.9829 against CV accuracy of 0.9794 is the tightest fit of the six — a gap of just +0.0035. Margin maximisation with a tuned `C` is doing exactly what it is supposed to do: finding a boundary that is deliberately not a perfect fit to the training points.

The **Decision Tree**, at 0.9726 training accuracy, is the only model that does not reach 100%. Its tuned `max_depth=5` is a hard capacity limit — and correspondingly it has the weakest CV score. This is the classic bias–variance trade-off made visible in a single row of the table.

In [ ]:
# ── Learning curve for the two strongest families ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5))
for ax, mname in zip(axes, ["Random Forest", "Logistic Regression"]):
    sizes, tr_sc, va_sc = learning_curve(
        fitted_models[mname], X_train, y_train,
        train_sizes=np.linspace(0.2, 1.0, 8),
        cv=cv, scoring="accuracy", n_jobs=-1, random_state=RANDOM_STATE,
    )
    tr_m, tr_s = tr_sc.mean(1), tr_sc.std(1)
    va_m, va_s = va_sc.mean(1), va_sc.std(1)
    ax.plot(sizes, tr_m, "o-", color="#DD8452", lw=2, label="Training score")
    ax.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="#DD8452")
    ax.plot(sizes, va_m, "s-", color="#4C72B0", lw=2, label="Cross-validation score")
    ax.fill_between(sizes, va_m - va_s, va_m + va_s, alpha=0.15, color="#4C72B0")
    ax.set_xlabel("Number of training samples"); ax.set_ylabel("Accuracy")
    ax.set_title(f"Learning curve — {mname}", fontsize=11, fontweight="bold")
    ax.set_ylim(0.6, 1.04); ax.legend(loc="lower right")
plt.suptitle("Would more data help?", fontsize=13, fontweight="bold")
plt.show()

**Interpretation of the learning curves.** Both curves show the validation score rising steeply up to roughly 150–200 training samples and then flattening, with the training and validation curves converging but not meeting.

Two readings follow:

1. **The dataset is large enough to have reached the flat part of the curve.** Performance is no longer limited by sample count in the steep-gains sense — doubling to ~600 patients would likely yield only a modest improvement in the headline accuracy.
2. **But the residual gap does not close**, which indicates the remaining error is not pure variance that more of the *same* data would average away. It is partly irreducible: the seborrheic dermatitis / pityriasis rosea overlap documented in Section 11.6, and the temporal ambiguity described in the brief (a disease showing another's features at its early stage).

The practical implication for Section 26 (Future Scope) is specific: the highest-value additional data is not simply *more patients*, but **more patients of the confusable types**, **adult PRP cases** to correct the age artefact, and ideally **longitudinal follow-up** to resolve the early-stage ambiguity that a single-timepoint record cannot.

<a id="s19"></a>
# 19. Model Evaluation

**The test set is opened here, for the first and only time.** All six models are now frozen; every hyperparameter was chosen using training-partition cross-validation alone. What follows is a genuine out-of-sample estimate on 74 patients who have played no part in any decision.

## 19.1 Metrics used, and why each one

| Metric | Definition | Why it matters here |
|---|---|---|
| **Accuracy** | Fraction of patients correctly diagnosed | Intuitive, but dominated by the majority classes. Reported, never relied upon alone. |
| **Balanced accuracy** | Mean of per-class recall | Accuracy's imbalance-robust counterpart |
| **Precision (macro)** | Mean over classes of TP/(TP+FP) | *If the model says psoriasis, how often is it psoriasis?* Low precision → **unnecessary treatment** |
| **Recall (macro)** | Mean over classes of TP/(TP+FN) | *Of all true psoriasis cases, how many were caught?* Low recall → **missed diagnosis** |
| **F1 (macro)** | Harmonic mean of macro precision and recall | Single balanced number weighting all six diseases equally. **Primary metric.** |
| **Precision/Recall/F1 (weighted)** | Same, weighted by class frequency | Expected performance on a patient population with this disease mix |
| **Cohen's κ** | Agreement corrected for chance | Guards against a high score achievable by exploiting the class priors |
| **Confusion matrix** | Full error structure | Shows *which* diseases are confused — clinically the most important output |

### The medical asymmetry between precision and recall

In dermatology these two errors are not equivalent, and which one matters more depends on the disease:

- **A false negative for psoriasis** (predicted as something milder) delays disease-modifying therapy for a lifelong systemic condition and may allow irreversible joint damage. **Recall is the priority for psoriasis.**
- **A false positive for psoriasis** (pityriasis rosea misread as psoriasis) commits a patient to months of unnecessary, potentially immunosuppressive treatment for a condition that would have resolved on its own in 6–8 weeks. **Precision matters too.**
- **For pityriasis rubra pilaris**, which is rare and easily missed, recall is paramount — a missed diagnosis means the patient continues to be treated for the wrong disease indefinitely.

No single metric captures this structure, which is why the confusion matrices and per-class tables below matter more than any headline number.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  FINAL EVALUATION ON THE HELD-OUT TEST SET
# ════════════════════════════════════════════════════════════════════════════
eval_rows    = []
predictions  = {}
per_class    = {}

for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0)
    p_wt, r_wt, f_wt, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0)

    log = next(r for r in training_log if r["Model"] == name)

    eval_rows.append({
        "Model":            name,
        "Family":           log["Family"],
        "CV Acc":           log["CV Acc"],
        "CV Acc std":       log["CV Acc std"],
        "CV F1-macro":      log["CV F1-macro"],
        "Train Acc":        log["Train Acc"],
        "Test Acc":         accuracy_score(y_test, y_pred),
        "Test Bal Acc":     balanced_accuracy_score(y_test, y_pred),
        "Precision (mac)":  p_mac,
        "Recall (mac)":     r_mac,
        "F1 (mac)":         f_mac,
        "Precision (wt)":   p_wt,
        "Recall (wt)":      r_wt,
        "F1 (wt)":          f_wt,
        "Cohen kappa":      cohen_kappa_score(y_test, y_pred),
        "Fit time (s)":     log["Fit time (s)"],
        "Infer (ms)":       log["Infer (ms)"],
        "Errors":           int((y_test != y_pred).sum()),
    })

    per_class[name] = classification_report(
        y_test, y_pred, labels=LABELS_ORDER,
        target_names=[CLASS_NAMES[c] for c in LABELS_ORDER],
        output_dict=True, zero_division=0)

results = pd.DataFrame(eval_rows).sort_values("F1 (mac)", ascending=False).reset_index(drop=True)
print(f"Evaluated {len(results)} models on {len(y_test)} held-out test patients.\n")

In [ ]:
# ── Headline comparison table ──────────────────────────────────────────────
main_cols = ["Model", "CV Acc", "CV Acc std", "Test Acc", "Test Bal Acc",
             "Precision (mac)", "Recall (mac)", "F1 (mac)", "F1 (wt)", "Cohen kappa", "Errors"]
display(results[main_cols]
        .style.format({c: "{:.4f}" for c in main_cols if c not in ("Model", "Errors")})
        .background_gradient(subset=["F1 (mac)", "Test Acc"], cmap="Greens")
        .set_caption("Held-out test-set performance (n = 74), ranked by macro-F1")
        .hide(axis="index"))

In [ ]:
# ── Visual comparison across the four principal metrics ───────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 9.5))
metrics = [("Test Acc", "Accuracy"), ("F1 (mac)", "Macro F1"),
           ("Precision (mac)", "Macro precision"), ("Recall (mac)", "Macro recall")]
for ax, (col, label) in zip(axes.ravel(), metrics):
    d = results.sort_values(col, ascending=True)
    bars = ax.barh(d["Model"], d[col], color=PALETTE[:len(d)][::-1],
                   edgecolor="black", linewidth=0.6)
    for b, v in zip(bars, d[col]):
        ax.text(v + 0.006, b.get_y() + b.get_height()/2, f"{v:.4f}",
                va="center", fontsize=9, fontweight="bold")
    ax.set_xlim(0.80, 1.045)
    ax.set_xlabel(label)
    ax.set_title(f"{label} on held-out test set", fontsize=11, fontweight="bold")
    ax.tick_params(labelsize=9)
plt.suptitle("Model comparison across four principal metrics (n = 74 test patients)",
             fontsize=13.5, fontweight="bold")
plt.show()

In [ ]:
# ── Generalisation view: CV vs test, with CV error bars ───────────────────
fig, ax = plt.subplots(figsize=(11, 5.4))
d = results.sort_values("Test Acc", ascending=False)
xp = np.arange(len(d)); w = 0.36
ax.bar(xp - w/2, d["CV Acc"], w, yerr=d["CV Acc std"], capsize=4,
       label=f"{CV_FOLDS}-fold CV accuracy (n=292)", color="#4C72B0",
       edgecolor="black", linewidth=0.6)
ax.bar(xp + w/2, d["Test Acc"], w, label="Held-out test accuracy (n=74)",
       color="#55A868", edgecolor="black", linewidth=0.6)
ax.set_xticks(xp); ax.set_xticklabels(d["Model"], rotation=18, ha="right")
ax.set_ylim(0.80, 1.045); ax.set_ylabel("Accuracy")
ax.set_title("Cross-validation vs. held-out test performance", fontsize=12, fontweight="bold")
ax.legend(loc="lower left")
plt.show()

print("CV accuracy minus test accuracy (positive = CV was optimistic):")
for _, r in d.iterrows():
    diff = r["CV Acc"] - r["Test Acc"]
    print(f"  {r['Model']:<22s}: CV {r['CV Acc']:.4f} +/- {r['CV Acc std']:.4f}  |  "
          f"Test {r['Test Acc']:.4f}  |  diff {diff:+.4f}")

### 19.2 Test-set results

| Rank | Model | CV accuracy | Test accuracy | Macro-F1 | Cohen's κ | Errors / 74 |
|---|---|---|---|---|---|---|
| **1** | **Random Forest** | **0.9794 ± 0.0129** | **0.9730** | **0.9694** | **0.9661** | **2** |
| 2 | Logistic Regression | 0.9794 ± 0.0128 | 0.9595 | 0.9574 | 0.9492 | 3 |
| 3 | SVM (RBF) | 0.9794 ± 0.0129 | 0.9595 | 0.9545 | 0.9491 | 3 |
| 4 | KNN | 0.9795 ± 0.0198 | 0.9324 | 0.9310 | 0.9156 | 5 |
| 5 | Gradient Boosting | 0.9691 ± 0.0229 | 0.9324 | 0.9137 | 0.9155 | 5 |
| 6 | Decision Tree | 0.9519 ± 0.0253 | 0.8919 | 0.8675 | 0.8651 | 8 |

**Interpretation — read the error counts, not the decimal places.**

The Random Forest makes **2 errors out of 74**; Logistic Regression and SVM make **3**. The apparent 1.35-percentage-point gap between rank 1 and rank 2 is **one patient**. Presenting that as a meaningful performance difference would be a misreading of the data, and this notebook does not do so. The top three models are, on this evidence, statistically indistinguishable — a conclusion reinforced by their cross-validation scores, which are identical to three decimal places (0.9794 for all three) with overlapping standard deviations.

What *is* meaningful:

- **The top five models are tightly clustered; the Decision Tree is genuinely behind.** Its 8 errors against the leaders' 2–3 is a real gap, visible in both CV (0.9519, clearly the lowest) and test (0.8919). A single depth-5 tree does not have the capacity to represent six overlapping disease signatures. This is a bias problem, not noise.
- **Cross-validation was mildly optimistic for every model**, by 1–4 points. This is exactly as expected: the hyperparameters were *selected* on those folds, so the CV score carries a selection bias. The test set exists to expose precisely this, and its doing so confirms the protocol worked.
- **Cohen's κ of 0.9661 for the Random Forest** confirms the performance is not an artefact of class priors — chance-corrected agreement is nearly as high as raw accuracy.
- **Gradient Boosting underperforms its reputation**, ranking 5th with a macro-F1 of 0.9137 despite competitive accuracy (0.9324). Its macro-precision of 0.9045 is the lowest of the six, indicating its errors are concentrated in the smaller classes. With only 292 training samples, sequential boosting has less material to work with than bagging — the forest's variance-averaging is better matched to this sample size than boosting's bias-reduction.

## 19.3 Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17.5, 11))
order = results["Model"].tolist()
for ax, name in zip(axes.ravel(), order):
    cm = confusion_matrix(y_test, predictions[name], labels=LABELS_ORDER)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES,
                linewidths=0.6, linecolor="white", ax=ax,
                annot_kws={"size": 10, "weight": "bold"})
    acc = accuracy_score(y_test, predictions[name])
    err = int((y_test != predictions[name]).sum())
    ax.set_title(f"{name}\nacc = {acc:.4f}   |   {err} error(s)", fontsize=10.5, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=9); ax.set_ylabel("Actual", fontsize=9)
    ax.tick_params(labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=38, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
plt.suptitle("Confusion matrices — held-out test set (n = 74), ordered by macro-F1",
             fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# ── Aggregate the error structure across all six models ───────────────────
err_counter = {}
for name, yp in predictions.items():
    for a, p in zip(y_test, yp):
        if a != p:
            err_counter[(int(a), int(p))] = err_counter.get((int(a), int(p)), 0) + 1

err_tbl = (pd.DataFrame([{"True": CLASS_NAMES[a], "Predicted": CLASS_NAMES[p],
                          "Times (across all 6 models)": n}
                         for (a, p), n in err_counter.items()])
           .sort_values("Times (across all 6 models)", ascending=False)
           .reset_index(drop=True))

print("ERROR STRUCTURE — every misclassification made by any of the six models\n")
display(err_tbl)

total_err = sum(err_counter.values())
top_pair  = err_tbl.iloc[0]
print(f"Total misclassifications across all 6 models : {total_err}")
print(f"Most frequent single error : {top_pair['True']} -> {top_pair['Predicted']} "
      f"({top_pair['Times (across all 6 models)']} times, "
      f"{top_pair['Times (across all 6 models)']/total_err*100:.1f}% of all errors)")

pair24 = sum(n for (a, p), n in err_counter.items() if {a, p} == {2, 4})
print(f"\nErrors involving the seborrheic dermatitis <-> pityriasis rosea pair : "
      f"{pair24} of {total_err}  ({pair24/total_err*100:.1f}% of all errors)")
print("This is exactly the pair predicted to be hardest by the EDA in Section 11.6.")

### 19.4 The error structure — and why it validates the EDA

**The errors made by all six models fall into a small, predictable set of confusions**, and the dominant one was forecast in Section 11.6 before any model was trained. Across all six models there are **26 misclassifications in total**, of which **16 (61.5%) involve the seborrheic dermatitis ↔ pityriasis rosea pair**. The single most common error — seborrheic dermatitis predicted as pityriasis rosea — accounts for **11 of the 26 (42.3%)** on its own.

Reading the top three confusion matrices:

- **Random Forest (2 errors):** one seborrheic dermatitis → pityriasis rosea, one pityriasis rosea → seborrheic dermatitis. **Both errors are the predicted difficult pair, in both directions.**
- **Logistic Regression (3 errors):** two seborrheic dermatitis → pityriasis rosea, one lichen planus → pityriasis rosea.
- **SVM (3 errors):** two seborrheic dermatitis → pityriasis rosea, one pityriasis rosea → seborrheic dermatitis.

This is a strong coherence result. Six models, spanning linear, tree, ensemble, kernel and instance-based families, with entirely different inductive biases, **converge on the same confusion**. That rules out a modelling artefact. The seborrheic dermatitis / pityriasis rosea boundary is genuinely ambiguous *in the data*, exactly as the Section 11.6 analysis showed: both diseases are characterised by mild spongiosis and by the absence of other diseases' markers, and their best discriminators (`itching`, `koebner_phenomenon`) differ by only ~1.15 grades.

**Equally notable is what is never confused.** Psoriasis is classified perfectly (23/23) by four of the six models. Lichen planus, chronic dermatitis and pityriasis rubra pilaris are near-perfect for the leaders. The diseases with strong, specific markers — rete-ridge architecture, band-like infiltrate, dermal fibrosis, follicular plugging — are essentially solved. **The difficulty is concentrated entirely in the one pair that lacks distinctive markers.**

**This is the single most clinically useful finding in the notebook**, and it is carried directly into the guidance in Section 22: a clinician should treat a model output of "seborrheic dermatitis" or "pityriasis rosea" with more caution than any other prediction, and that pair is where biopsy adds the most value.

## 19.5 Per-class performance

In [ ]:
# ── Per-class F1 across all models ─────────────────────────────────────────
pc_f1 = pd.DataFrame({
    name: [per_class[name][CLASS_NAMES[c]]["f1-score"] for c in LABELS_ORDER]
    for name in results["Model"]
}, index=[f"{c}. {CLASS_NAMES[c]}" for c in LABELS_ORDER])

fig, ax = plt.subplots(figsize=(11, 5.2))
sns.heatmap(pc_f1, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.6, vmax=1.0,
            linewidths=0.6, linecolor="white", ax=ax,
            cbar_kws={"label": "F1-score"}, annot_kws={"size": 9.5, "weight": "bold"})
ax.set_title("Per-class F1-score — every disease × every model (test set, n = 74)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel(""); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=22, ha="right")
plt.show()

print("Mean F1 per disease across all six models (which disease is hardest overall?):")
for d, v in pc_f1.mean(axis=1).sort_values().items():
    n_test = int((y_test == int(d.split('.')[0])).sum())
    print(f"  {v:.4f}   {d}   (n = {n_test} in test set)")

In [ ]:
# ── Detailed per-class report for the best model ───────────────────────────
BEST_MODEL_NAME = results.iloc[0]["Model"]
BEST_MODEL      = fitted_models[BEST_MODEL_NAME]
y_pred_best     = predictions[BEST_MODEL_NAME]

print("=" * 78)
print(f"DETAILED CLASSIFICATION REPORT — {BEST_MODEL_NAME}")
print(f"Held-out test set: {len(y_test)} patients")
print("=" * 78)
print(classification_report(y_test, y_pred_best, labels=LABELS_ORDER,
                            target_names=[f"{c}. {CLASS_NAMES[c]}" for c in LABELS_ORDER],
                            digits=4, zero_division=0))

cm_best = confusion_matrix(y_test, y_pred_best, labels=LABELS_ORDER)
print("\nPer-class breakdown with clinical framing:")
for i, c in enumerate(LABELS_ORDER):
    tp = cm_best[i, i]; support = cm_best[i].sum()
    fp = cm_best[:, i].sum() - tp
    rec  = tp / support if support else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    print(f"\n  [{c}] {CLASS_NAMES[c]}  (n = {support} test patients)")
    print(f"      Recall    {rec:.4f}  -> caught {tp} of {support} true cases "
          f"({support - tp} missed)")
    print(f"      Precision {prec:.4f}  -> of {tp+fp} predicted, {tp} correct "
          f"({fp} false alarm{'s' if fp != 1 else ''})")

### 19.6 Per-class interpretation — where does the model succeed and fail?

**Mean F1 per disease across all six models**, from hardest to easiest, tells a clear story:

| Disease | Mean F1 across 6 models | Test n | Assessment |
|---|---|---|---|
| **Seborrheic dermatitis** | 0.8519 | 12 | **Hardest.** Confused with pityriasis rosea in both directions. No distinctive marker. |
| **Pityriasis rosea** | 0.8599 | 10 | **Second hardest**, and for the same reason — the two form a mutually-confusable pair. |
| Pityriasis rubra pilaris | 0.9398 | 4 | Strong despite being the rarest class — the age signal (mean 10.25 yrs) carries it |
| Chronic dermatitis | 0.9762 | 10 | Dermal fibrosis is a near-unique marker |
| Lichen planus | 0.9770 | 15 | Five near-pathognomonic markers make it the cleanest signature |
| **Psoriasis** | 0.9887 | 23 | **Easiest.** Perfect or near-perfect in every model |

**Three observations.**

**First — rarity is not the driver of difficulty.** Pityriasis rubra pilaris has the fewest samples (20 total, 4 in test) yet outperforms two much larger classes. Seborrheic dermatitis has three times as many patients and does worse. **What determines difficulty is signature distinctiveness, not sample count.** This is a useful corrective to the reflex assumption that the minority class is always the problem, and it is why the class-balancing experiment in Section 16.7 found so little to gain.

**Second — the two hard classes are hard *together*, not separately.** Their errors are almost entirely confusions with each other rather than with the other four diseases. The model is not failing to recognise them as a pair; it is failing to tell them apart.

**Third — a caution on the precision of these numbers.** Pityriasis rubra pilaris has **4** test patients. Its F1 of 1.000 for the Random Forest means 4 correct predictions out of 4. A single error would drop it to roughly 0.86. **These per-class figures should be read as indicative, not precise.** A 95% confidence interval on a recall estimated from 4 samples spans a very wide range, and the same caution applies with less force to every class in a 74-patient test set.

## 19.7 Computational cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
d = results.sort_values("Fit time (s)")
axes[0].barh(d["Model"], d["Fit time (s)"], color="#DD8452", edgecolor="black", linewidth=0.6)
for i, v in enumerate(d["Fit time (s)"]):
    axes[0].text(v + 0.35, i, f"{v:.2f}s", va="center", fontsize=9, fontweight="bold")
axes[0].set_xlabel("Seconds")
axes[0].set_title("Full hyperparameter search wall time", fontsize=11, fontweight="bold")

d2 = results.sort_values("Infer (ms)")
axes[1].barh(d2["Model"], d2["Infer (ms)"], color="#55A868", edgecolor="black", linewidth=0.6)
for i, v in enumerate(d2["Infer (ms)"]):
    axes[1].text(v + 0.2, i, f"{v:.2f}ms", va="center", fontsize=9, fontweight="bold")
axes[1].set_xlabel(f"Milliseconds for all {len(y_test)} test patients")
axes[1].set_title("Inference time", fontsize=11, fontweight="bold")
plt.suptitle("Computational cost", fontsize=13, fontweight="bold")
plt.show()

print("Per-patient inference latency:")
for _, r in results.sort_values("Infer (ms)").iterrows():
    print(f"  {r['Model']:<22s}: {r['Infer (ms)']/len(y_test):.4f} ms/patient")
print("\nEvery model is far below any latency that would matter in a clinical workflow.")
print("Computational cost is therefore NOT a meaningful discriminator for production selection here.")

**Interpretation.** Tuning time ranges from well under a second (Logistic Regression) to roughly 20–26 seconds (Random Forest, Gradient Boosting), reflecting grid size multiplied by ensemble size. Inference is in **microseconds per patient** for all six models.

For a clinical decision-support tool used at the point of care, a handful of microseconds is indistinguishable from zero. **Computational cost is therefore not a meaningful criterion for choosing between these models** — and saying so explicitly matters, because in many production settings it would be the deciding factor. Here, interpretability and reliability carry the decision instead.

## 19.8 What drives the best model's predictions

In [ ]:
# ── Permutation importance on the TEST set for the best model ─────────────
perm = permutation_importance(BEST_MODEL, X_test, y_test,
                              n_repeats=30, random_state=RANDOM_STATE,
                              scoring="f1_macro", n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
perm_e = pd.Series(perm.importances_std,  index=X_test.columns)

fig, ax = plt.subplots(figsize=(9.5, 7))
top = perm_s.head(15)
ax.barh([f.replace("_", " ")[:38] for f in top.index][::-1], top.values[::-1],
        xerr=perm_e[top.index].values[::-1], capsize=3,
        color="#4C72B0", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Drop in macro-F1 when this feature is randomly shuffled")
ax.set_title(f"Permutation importance — {BEST_MODEL_NAME} (test set, 30 repeats)",
             fontsize=12, fontweight="bold")
ax.tick_params(labelsize=8.5)
plt.show()

print(f"Top 12 features actually driving {BEST_MODEL_NAME}'s test predictions:")
for i, (k, v) in enumerate(perm_s.head(12).items(), 1):
    block = "clinical" if k in CLINICAL else "histopath."
    print(f"  {i:2d}. {v:.4f} +/- {perm_e[k]:.4f}  [{block:<11s}] {k}")

**Interpretation.** Permutation importance measures something the earlier rankings did not: **how much the model's actual out-of-sample performance degrades when a feature is destroyed.** Mutual information (Section 16.2) scored features in isolation on the training data; Gini importance (Section 16.3) scored them by their role in training splits. This measures their contribution to genuine predictive performance on unseen patients — the most decision-relevant of the three.

The ranking is dominated by the same architectural and interface markers identified throughout — confirming that the model is not exploiting some incidental artefact but has learned the recognised pathological signatures of these diseases. **This coherence between the model's learned structure and established dermatological knowledge is itself a validation signal**, and it is the property that makes it reasonable to present the model to a clinician as decision support rather than as an oracle.

**One caveat.** Permutation importance is computed here on 74 test patients, so the error bars are wide and small differences in rank should not be over-read. Features with correlated substitutes will also appear less important than they are — shuffling `band_like_infiltrate` barely hurts the forest, because `saw_tooth_appearance_of_retes` still carries the same lichen planus signal. This is the same redundancy effect discussed in Section 16.3.

In [ ]:
# ── Save the best model and all artifacts ─────────────────────────────────
model_path = OUTPUT_DIR / f"best_model_{BEST_MODEL_NAME.replace(' ', '_').replace('(', '').replace(')', '')}.joblib"
joblib.dump(BEST_MODEL, model_path)

results.to_csv(OUTPUT_DIR / "model_comparison_results.csv", index=False)
pc_f1.to_csv(OUTPUT_DIR / "per_class_f1_scores.csv")

metadata = {
    "project_id":            "PRCP-1027",
    "project_title":         "Skin Disorder Prediction",
    "dataset_file":          str(DATASET_PATH.name),
    "n_samples_total":       int(len(df)),
    "n_features":            int(len(FEATURE_COLS)),
    "n_classes":             int(len(LABELS_ORDER)),
    "class_mapping":         {str(k): v for k, v in CLASS_NAMES.items()},
    "class_counts":          {str(k): int(v) for k, v in df[TARGET_COL].value_counts().sort_index().items()},
    "n_train":               int(len(X_train)),
    "n_test":                int(len(X_test)),
    "random_state":          RANDOM_STATE,
    "cv_folds":              CV_FOLDS,
    "tuning_metric":         SCORING,
    "missing_values_age":    int(df[AGE_COL].isna().sum()),
    "best_model":            BEST_MODEL_NAME,
    "best_params":           {k.replace("model__", ""): str(v)
                              for k, v in search_objects[BEST_MODEL_NAME].best_params_.items()},
    "best_cv_accuracy":      float(results.iloc[0]["CV Acc"]),
    "best_cv_accuracy_std":  float(results.iloc[0]["CV Acc std"]),
    "best_test_accuracy":    float(results.iloc[0]["Test Acc"]),
    "best_test_f1_macro":    float(results.iloc[0]["F1 (mac)"]),
    "best_test_kappa":       float(results.iloc[0]["Cohen kappa"]),
    "sklearn_version":       sklearn.__version__,
}
with open(OUTPUT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Artifacts written:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"   {p.name:<55s} ({p.stat().st_size/1024:.1f} KB)")

print(f"\nReload the saved model with:")
print(f"   model = joblib.load(r'{model_path}')")
print(f"   preds = model.predict(new_patient_dataframe)   # same 34 columns, raw values")
print("\nThe saved object is the COMPLETE pipeline — imputation and scaling are included,")
print("so raw unprocessed input can be passed directly. This eliminates train/serve skew.")

### 19.9 Optional — XGBoost comparison

XGBoost is not required for this project: scikit-learn's `GradientBoostingClassifier` already represents the boosting family in the main comparison, and it is guaranteed to be available. The cell below adds XGBoost **only if it is installed**, so the notebook runs end-to-end either way.

If XGBoost is not present, this cell prints a message and moves on. It is a supplementary check, and the main results and conclusions do not depend on it.

In [ ]:
if XGBOOST_AVAILABLE:
    # XGBoost requires labels in 0..n-1, so shift 1-6 -> 0-5 and shift back after predicting
    y_train_x = y_train - 1

    xgb_pipe = Pipeline([
        ("prep", build_preprocessor(scale=True)),
        ("model", XGBClassifier(
            objective="multi:softprob", num_class=6,
            eval_metric="mlogloss", random_state=RANDOM_STATE,
            n_jobs=-1, tree_method="hist",
        )),
    ])
    xgb_grid = {
        "model__n_estimators":  [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth":     [2, 3],
        "model__subsample":     [0.8, 1.0],
    }
    t0 = time.perf_counter()
    xgb_search = GridSearchCV(xgb_pipe, xgb_grid, cv=cv, scoring=SCORING, n_jobs=-1).fit(X_train, y_train_x)
    xgb_time = time.perf_counter() - t0

    y_pred_x = xgb_search.predict(X_test) + 1   # shift back to 1-6
    xp_, xr_, xf_, _ = precision_recall_fscore_support(y_test, y_pred_x, average="macro", zero_division=0)
    xgb_best = {k.replace("model__", ""): v for k, v in xgb_search.best_params_.items()}

    print("XGBoost (supplementary)")
    print(f"  best params   : {xgb_best}")
    print(f"  CV f1_macro   : {xgb_search.best_score_:.4f}")
    print(f"  Test accuracy : {accuracy_score(y_test, y_pred_x):.4f}")
    print(f"  Test macro-F1 : {xf_:.4f}")
    print(f"  Tuning time   : {xgb_time:.2f} s")
    print()
    print(f"  For reference, the best scikit-learn model ({BEST_MODEL_NAME}):")
    print(f"    Test accuracy {results.iloc[0]['Test Acc']:.4f} | Test macro-F1 {results.iloc[0]['F1 (mac)']:.4f}")
    print("\n  NOTE: these XGBoost figures are produced by YOUR run. They are not quoted in the")
    print("  written reports below, which cite only the six core scikit-learn models.")
else:
    print("XGBoost is not installed in this environment — this optional cell is skipped.")
    print()
    print("The boosting family is already represented in the main comparison by")
    print("scikit-learn's GradientBoostingClassifier, which ranked 5th of 6")
    print("(test accuracy 0.9324, macro-F1 0.9137).")
    print()
    print("To include XGBoost as well:  pip install xgboost   then re-run this cell.")

<a id="s20"></a>
# 20. MODEL COMPARISON REPORT

> **Deliverable: "Create a report stating the performance of multiple models on this data and suggest the best model for production."**

---

## 20.1 Models evaluated

| # | Model | Family | Core mechanism |
|---|---|---|---|
| 1 | Logistic Regression | Linear | L2-regularised multinomial log-odds |
| 2 | Decision Tree | Tree | Recursive impurity-minimising splits |
| 3 | Random Forest | Ensemble (bagging) | Majority vote over bootstrap-decorrelated trees |
| 4 | SVM (RBF kernel) | Kernel / margin | Maximum-margin boundary in an implicit feature space |
| 5 | K-Nearest Neighbours | Instance-based | Vote of the k most similar training patients |
| 6 | Gradient Boosting | Ensemble (boosting) | Additive sequential residual correction |

*Deep learning was deliberately excluded — see Section 17.1.*

## 20.2 Approach

- **Split:** stratified 80/20 → 292 training / 74 test patients. The test set was locked away until Section 19 and used exactly once.
- **Validation:** stratified 5-fold cross-validation on the training partition only, for every experiment and every tuning decision.
- **Tuning:** `GridSearchCV`, objective `f1_macro` (chosen so that all six diseases count equally regardless of prevalence).
- **Leakage control:** imputation and scaling encapsulated in `sklearn.Pipeline`, fitted per fold on training data only.
- **Reproducibility:** `random_state=42` fixed in the split, every CV object, and every stochastic model.

## 20.3 Preprocessing applied

| Step | Detail | Justification |
|---|---|---|
| Missing-value handling | Median imputation of `Age` (8 values, 2.19%) | Robust to the bimodal age distribution; class-conditional imputation rejected as target leakage |
| Type conversion | `'?'` → `NaN` → numeric | The source encoding forces the column to `object` dtype |
| Scaling | `StandardScaler` on all 34 features | **Measured:** SVM +0.2946, KNN +0.1438 CV accuracy (Section 16.1) |
| Encoding | Ordinal grades kept numeric; target kept as integers 1–6 | Preserves ordering; avoids inflating $p$ from 34 to >130 |

## 20.4 Feature engineering outcome

Six techniques were tested by cross-validation. **Only two earned adoption:**

| Technique | Measured effect | Adopted |
|---|---|---|
| StandardScaler | SVM +0.2946, KNN +0.1438 | **YES** |
| Median imputation | Recovers 8 patient records | **YES** |
| Feature selection (MI) | k=25: +0.0103 (within ±0.0129 noise); k=5: **−0.1951** | NO |
| PCA | Ties at best (20 PCs), worse below | NO |
| `class_weight='balanced'` | +0.003–0.004 macro-F1, within noise | Left in the grid |
| SMOTE | Not applied — would produce non-integer ordinal grades | NO |

## 20.5 Evaluation metrics

Accuracy, balanced accuracy, macro and weighted precision / recall / F1, Cohen's κ, full confusion matrices, and per-class metrics for all six diseases — with **macro-F1 as the primary metric** because it weights all diseases equally despite the 5.6:1 imbalance.

## 20.6 Measured performance (held-out test set, n = 74)

| Rank | Model | CV Accuracy (n=292) | Test Accuracy | Macro-F1 | Macro-P | Macro-R | Cohen's κ | Errors |
|---|---|---|---|---|---|---|---|---|
| **1** | **Random Forest** | **0.9794 ± 0.0129** | **0.9730** | **0.9694** | **0.9694** | **0.9694** | **0.9661** | **2** |
| 2 | Logistic Regression | 0.9794 ± 0.0128 | 0.9595 | 0.9574 | 0.9615 | 0.9611 | 0.9492 | 3 |
| 3 | SVM (RBF) | 0.9794 ± 0.0129 | 0.9595 | 0.9545 | 0.9545 | 0.9556 | 0.9491 | 3 |
| 4 | KNN | 0.9795 ± 0.0198 | 0.9324 | 0.9310 | 0.9372 | 0.9400 | 0.9156 | 5 |
| 5 | Gradient Boosting | 0.9691 ± 0.0229 | 0.9324 | 0.9137 | 0.9045 | 0.9306 | 0.9155 | 5 |
| 6 | Decision Tree | 0.9519 ± 0.0253 | 0.8919 | 0.8675 | 0.8691 | 0.8716 | 0.8651 | 8 |

**Selected hyperparameters:**

| Model | Configuration |
|---|---|
| Random Forest | `n_estimators=200, max_depth=None, min_samples_leaf=2, class_weight=None` |
| Logistic Regression | `C=10, class_weight=None` |
| SVM (RBF) | `C=1, gamma=0.01, class_weight=None` |
| KNN | `n_neighbors=11, weights='uniform', p=1` (Manhattan) |
| Gradient Boosting | `n_estimators=100, learning_rate=0.05, max_depth=2` |
| Decision Tree | `criterion='gini', max_depth=5, min_samples_leaf=1` |

## 20.7 How large is the difference between the top models, really?

**The honest answer: not large enough to distinguish them.**

The Random Forest's 0.9730 accuracy and Logistic Regression's 0.9595 differ by **exactly one patient out of 74**. Their cross-validation accuracies are identical to four decimal places (0.9794 both) with fully overlapping standard deviations. The SVM sits in the same place.

A Wilson 95% confidence interval on 72/74 correct spans roughly **[0.906, 0.993]**; on 71/74 it spans roughly **[0.885, 0.986]**. These intervals overlap almost entirely.

**Conclusion: the top three models are statistically indistinguishable on this evidence.** Any claim that the Random Forest is "the best model" on the strength of 1.35 accuracy points would be a misreading of a 74-sample test set. Model selection must therefore be decided on criteria *other than* the headline score — which is what Section 20.11 does.

The one difference that **is** real: the **Decision Tree is genuinely behind**, with 8 errors against 2–3 for the leaders, and it is the lowest performer in cross-validation too (0.9519, over two standard deviations below the leaders). That gap is consistent across both evaluations and reflects a real capacity limitation, not noise.

## 20.8 Computational considerations

| Model | Tuning time | Inference (74 patients) | Serialised size |
|---|---|---|---|
| Logistic Regression | ~0.7 s | ~3.2 ms | ~10 KB |
| KNN | ~1.3 s | ~4.6 ms | stores all 292 training rows |
| Decision Tree | ~1.6 s | ~2.9 ms | ~10 KB |
| SVM (RBF) | ~3.0 s | ~3.5 ms | ~50 KB |
| Gradient Boosting | ~22 s | ~3.5 ms | ~500 KB |
| **Random Forest** | **~26 s** | **~14.7 ms** | **~1–2 MB** |

All inference times are **microseconds per patient**. In a clinical workflow where a consultation lasts minutes, this is indistinguishable from zero. **Computational cost is not a meaningful selection criterion here** — a fact worth stating explicitly, because in many production settings it would be decisive.

The one genuine architectural note: **KNN stores the entire training set** and must compute distances to all 292 patients at inference. This scales poorly and, more importantly in healthcare, means the deployed artefact *contains patient records* — a privacy consideration that the other five models do not raise.

## 20.9 Overfitting assessment

| Model | Train Acc | CV Acc | Gap | Assessment |
|---|---|---|---|---|
| Logistic Regression | 1.0000 | 0.9794 | +0.0206 | Fits training data exactly; regularisation (`C=10`) holds generalisation |
| Random Forest | 1.0000 | 0.9794 | +0.0206 | Expected for a forest; bagging controls variance |
| Gradient Boosting | 1.0000 | 0.9691 | +0.0309 | Largest gap; capacity limited by `max_depth=2` |
| **SVM (RBF)** | **0.9829** | **0.9794** | **+0.0035** | **Tightest fit of the six** — margin maximisation working as intended |
| KNN | 0.9795 | 0.9795 | 0.0000 | `k=11` with uniform weights; no memorisation |
| Decision Tree | 0.9726 | 0.9519 | +0.0207 | Lowest training accuracy — `max_depth=5` is a hard capacity ceiling |

Gaps of 2–3 points are acceptable at this sample size. The **learning curves** (Section 18) show validation scores plateauing after ~150–200 samples with a residual gap that does not close — indicating the remaining error is substantially **irreducible** rather than a variance problem more data would solve.

## 20.10 Interpretability for clinicians — the decisive criterion

In healthcare, a model whose reasoning cannot be audited is a model that cannot be responsibly acted upon. The six models differ sharply here:

| Model | Interpretability | What a clinician can actually be shown |
|---|---|---|
| **Decision Tree** | **Highest** | The complete rule set, printable as a flowchart and readable end to end |
| **Logistic Regression** | **High** | A signed coefficient per feature per disease — "grade 3 band-like infiltrate contributes +X to lichen planus" |
| **Random Forest** | **Moderate** | Global feature importances; per-prediction class-probability distribution; individual trees inspectable but 200 of them are not |
| Gradient Boosting | Moderate | Global importances; the additive sequence is not human-readable |
| SVM (RBF) | **Low** | The boundary lives in an implicit infinite-dimensional space. No feature-level explanation without post-hoc tools |
| KNN | Moderate | "These 11 similar past patients had this diagnosis" — intuitive, but exposes training records |

The **SVM's low interpretability is disqualifying** for a first deployment in this domain, despite its joint-best cross-validation score. The **Decision Tree's** transparency is the strongest of the six, but it is bought at a real cost in accuracy (8 errors vs 2).

## 20.11 Final model recommendation

### Recommended: **Random Forest** — `n_estimators=200, max_depth=None, min_samples_leaf=2, class_weight=None`

**Rationale, weighted against the criteria the brief specifies:**

| Criterion | Assessment |
|---|---|
| **Measured performance** | Joint-best CV (0.9794 ± 0.0129); best test accuracy (0.9730), macro-F1 (0.9694) and κ (0.9661). Fewest errors (2/74) |
| **Generalisation** | Bagging over bootstrap samples is the variance-reduction mechanism best matched to $n=292$. CV and test agree closely |
| **Interpretability** | Moderate but **sufficient**: global feature importances tie directly to recognised pathological markers, and per-prediction class probabilities let a clinician see *how confident* the model is and *which* alternatives it considered |
| **Model size / inference** | ~1–2 MB, microseconds per patient — irrelevant constraints at this scale |
| **Reliability across all 6 classes** | The most uniform per-class performance of the six: perfect or near-perfect on 4 of 6 diseases, and the best scores on the two hard ones |
| **Robustness to dataset limitations** | Least sensitive to the small sample, to correlated features (Section 11.5) and to the mild imbalance, since each tree sees a different bootstrap and feature subset |

**Why not the alternatives:**

- **Logistic Regression** is the strongest runner-up and has *better* interpretability. It is the right choice if per-feature explanation is a hard requirement — and is recommended as a **companion model**, not a rejected one (see below).
- **SVM (RBF)** matches on cross-validated score but its opacity is disqualifying for a first clinical deployment.
- **KNN** is competitive but stores all training patients in the deployed artefact — a privacy problem in healthcare.
- **Gradient Boosting** has the lowest macro-precision (0.9045), indicating weakness on the smaller classes, at higher computational cost.
- **Decision Tree** is the most interpretable but makes 4× the errors. Retained as an **explanatory aid**, not a predictor.

### Recommended deployment pattern: a two-model system

Because the top three are statistically indistinguishable, the choice can be made on complementary strengths rather than on a score difference of one patient:

1. **Random Forest as the primary predictor** — best measured reliability across all six diseases, with calibrated-ish class probabilities for confidence reporting.
2. **Logistic Regression as the transparent companion** — run in parallel on every case, presenting per-feature contributions so the clinician can see *why*. Its near-identical accuracy (3 errors vs 2) means this transparency costs essentially nothing.
3. **Flag any case where the two disagree**, or where the Random Forest's top class probability falls below a threshold, for mandatory specialist review. Given the error structure in Section 19.4, most such flags will be seborrheic dermatitis / pityriasis rosea cases — which is exactly where a clinician's judgement and a targeted biopsy add most value.

## 20.12 Generalisation limitations — required reading before any use

**This model is not clinically validated and must not be described as such.** Specifically:

| Limitation | Consequence |
|---|---|
| **Single-source cohort** (366 patients, one clinical setting) | No evidence of transfer to other populations, ethnicities, or grading conventions |
| **74-patient test set** | 95% CI on the headline accuracy spans roughly [0.906, 0.993]. The point estimate is far more precise than the evidence |
| **4 test patients in the rarest class** | Per-class figures for pityriasis rubra pilaris are indicative only |
| **No adult PRP cases** (max age 22 in this cohort) | The model will likely under-predict PRP in adults — a systematic, identifiable failure mode |
| **Assumes expert-graded 0–3 inputs** | Garbage in, garbage out. Grading is a learned skill; inter-rater variability would degrade performance by an unmeasured amount |
| **Requires 22 histopathological features** for full accuracy | Full performance presupposes a biopsy has already been performed |
| **Single-timepoint records** | Cannot handle the early-stage ambiguity the problem brief explicitly describes |
| **Six classes only** | A patient with a seventh condition will be forced into one of these six, with confident-looking output |

### What additional validation would be required

1. **External validation** on an independent cohort from a different institution, ideally a different country and population.
2. **Prospective clinical study** — apply the model at the point of care and compare against the eventual confirmed diagnosis.
3. **Inter-rater reliability study** on the 0–3 grading scales, to quantify how much performance degrades under realistic grading variability.
4. **Dermatologist benchmarking** — compare model output against specialist diagnosis on the same cases, which is the only comparison that establishes clinical value.
5. **Expanded cohort** with adult PRP cases, more seborrheic dermatitis / pityriasis rosea patients, and longitudinal follow-up.
6. **Probability calibration assessment** — for decision support, a "70% confident" output must actually be right about 70% of the time.
7. **Prospective safety and bias audit**, including performance stratified by skin tone, age and sex.

**Current status: an academic prototype demonstrating that the six erythemato-squamous diseases are highly separable from structured clinical and histopathological data. It is a research result, not a medical device.**

<a id="s21"></a>
# 21. CHALLENGES FACED AND TECHNIQUES USED

> **Deliverable: "Create a report which should include challenges you faced on data and what technique used with proper reason."**

Each challenge is documented with its evidence, the technique applied, the reason for that choice, and the limitation that remains.

---

## Challenge 1 — Small sample size with relatively high dimensionality

**Description.** 366 patients described by 34 attributes gives $n/p \approx 10.8$. The usual guideline for stable estimation is 10–20 samples per feature, so this sits at the very bottom of the acceptable range. Every added model parameter is a fresh opportunity to fit noise.

**Evidence.** After the 80/20 split, 292 training patients remain. The rarest class contributes **16** of them. In a 5-fold CV, each validation fold holds roughly **3** patients of that class.

**Technique used.**
- Stratified 5-fold cross-validation rather than a single validation split, so every training patient is used for validation exactly once.
- Deliberately **small** hyperparameter grids (2–24 configurations per model) to limit the number of comparisons and hence the selection bias in the winning score.
- Explicit regularisation tuned per model: `C` for Logistic Regression and SVM, `max_depth` and `min_samples_leaf` for trees, `max_depth ∈ {2,3}` for boosting.
- Deep learning excluded outright.

**Reason.** At this sample size, the dominant risk is not underfitting but **overfitting to the validation procedure itself**. A large grid searched by CV will find a configuration that happens to suit those particular folds, and the reported score becomes a selection artefact. Small grids and strong regularisation are the appropriate response.

**Limitation that remains.** The learning curves (Section 18) show validation performance plateauing after ~150–200 samples, so more data of the *same* kind would yield limited gains. The residual error is partly irreducible.

---

## Challenge 2 — Missing values in `Age` encoded as `'?'`

**Description.** The `Age` column contains the literal string `'?'` for missing entries. Because pandas types a column by its contents, the presence of `'?'` forces the entire column to `object` dtype — so arithmetic and every scikit-learn estimator fail silently or loudly on it.

**Evidence.** **8 entries (2.19% of rows)**, spread across five of the six classes (1 each in classes 1–4, 4 in class 5). All 33 other attributes are complete. Before conversion, `df['Age'].dtype` is `object`.

**Technique used.** `pd.read_csv(..., na_values=['?'])` plus an explicit `pd.to_numeric(errors='coerce')` guard, followed by `SimpleImputer(strategy='median')` **inside the pipeline**.

**Reason.**
- *Median, not mean:* the age distribution is bimodal because of the paediatric PRP cluster (mean 10.25 years). The mean is pulled by that cluster; the median is not.
- *Impute, not drop:* discarding 8 of 366 records — including 4 of 52 chronic-dermatitis patients (7.7% of that class) — is a poor trade on an already small dataset.
- *Not class-conditional:* imputing each patient's age with their disease's median would be more accurate but uses the **target to construct a feature** — textbook leakage that would inflate CV scores and collapse at inference, when the label is exactly what is unknown.
- *Inside the pipeline:* the median is computed per training fold, so no held-out patient influences it.

**Limitation that remains.** Eight patients carry a fabricated age. The effect is negligible at this scale, but it is a fabricated value nonetheless.

---

## Challenge 3 — The diseases share their defining clinical features

**Description.** This is the core difficulty stated in the problem brief, and it is visible in the data. All six conditions present with erythema and scaling — the features a clinician observes first are the features that discriminate least.

**Evidence — quantified in Section 16.2.** Mutual information with the diagnosis:

| Attribute | MI | Rank (of 34) |
|---|---|---|
| clubbing of the rete ridges | 0.6057 | 1 |
| **erythema** | **0.0630** | **32** |
| **inflammatory mononuclear infiltrate** | **0.0322** | **34** |

`erythema` carries roughly **one-tenth** the information of the top marker.

**Technique used.**
- Retained **all 34 features** rather than selecting a "top-k" subset (Experiment 4, Section 16.4).
- Used multivariate models that combine many weak signals rather than relying on single-feature rules.
- Reported per-class metrics and full confusion matrices to expose *where* the overlap bites.

**Reason.** The information is not in any single feature but in the **joint pattern**. Aggressive selection measurably destroys this: cutting to 5 features costs ~20 accuracy points. A feature that looks weak globally may be the decisive evidence for the 15% of patients it applies to — `follicular_horn_plug` is useless for five diseases and near-decisive for PRP.

**Limitation that remains.** Requiring all 34 features means requiring a biopsy for full accuracy. The clinical-only analysis (Section 16.5) shows 87.7% is achievable without one — good, but 9.9 points below the full model.

---

## Challenge 4 — Seborrheic dermatitis and pityriasis rosea are intrinsically confusable

**Description.** Two diseases lack distinctive markers and are defined largely by the *absence* of the other four diseases' signatures — a diagnosis of exclusion, which is a harder learning problem.

**Evidence.**
- *Predicted before modelling* (Section 11.6): pityriasis rosea's strongest marker is spongiosis at **+1.00** above the cohort mean, and spongiosis is *also* seborrheic dermatitis's top marker (+1.24). Compare lichen planus's band-like infiltrate at **+2.70**.
- Their best separators, `itching` and `koebner_phenomenon`, differ by only **1.15 grades**.
- *Confirmed after modelling* (Section 19.4): this pair accounts for **16 of the 26 errors (61.5%)** made across **all six models**, which span four different algorithmic families.

**Technique used.**
- Identified the difficulty from the EDA *before* training, so it could be tested rather than discovered by accident.
- Aggregated the error structure across all six models to establish that the confusion is a property of the data, not of one algorithm's inductive bias.
- Carried the finding into the clinical guidance (Section 22) as a targeted caution rather than burying it in an aggregate score.

**Reason.** Six models with entirely different assumptions converging on the same confusion is strong evidence that the boundary is genuinely ambiguous in the feature space. No amount of tuning removes it; the correct response is to *communicate* it.

**Limitation that remains.** This is irreducible with the current feature set. Resolving it would need additional discriminating information — the presence of a herald patch, lesion distribution along cleavage lines, or dermoscopic findings — none of which is recorded here.

---

## Challenge 5 — Class imbalance (5.6 : 1)

**Description.** Psoriasis has 112 patients; pityriasis rubra pilaris has 20. Accuracy alone would reward a model that neglected the rare classes.

**Evidence.** Counts 112 / 61 / 72 / 49 / 52 / 20; Shannon evenness 0.9411; majority-class baseline accuracy 30.6%.

**Technique used.**
- **Stratified** splitting and stratified CV folds throughout.
- **`f1_macro`** as the tuning objective, so every disease contributes one-sixth of the score regardless of prevalence.
- `class_weight ∈ {None, 'balanced'}` placed **in the hyperparameter grid** and decided by cross-validation per model.
- **SMOTE rejected**, with reasons.
- Per-class metrics reported for all six diseases.

**Reason.** SMOTE interpolates between neighbours, which on an ordinal 0–3 integer scale produces values like `band_like_infiltrate = 2.37` — a grade no biopsy report can contain. With only 16 class-6 training patients, it would also be interpolating within a very sparse neighbourhood, largely manufacturing points along lines joining a handful of real patients. `class_weight` achieves the same goal by re-weighting the loss, inventing no data.

**Empirical finding — and it is a negative one.** Balanced weighting improved macro-F1 by only **+0.003 to +0.004**, well within the ±0.014 CV standard deviation. Section 19.6 explains why: **PRP is the rarest class but not the hardest one**, because it has the strongest univariate marker in the dataset (mean age 10.25). Difficulty here is driven by signature distinctiveness, not sample count.

**Limitation that remains.** Four test patients in class 6 means its per-class metrics are indicative only.

---

## Challenge 6 — Multicollinearity among the histopathological features

**Description.** Many features describe the same underlying pathological process and therefore co-vary strongly.

**Evidence.** Section 11.5 identifies correlated clusters: `clubbing_of_the_rete_ridges` / `elongation_of_the_rete_ridges` / `thinning_of_the_suprapapillary_epidermis` (psoriasiform architecture), and `band-like_infiltrate` / `saw-tooth_appearance_of_retes` / `vacuolisation_and_damage_of_basal_layer` / `melanin_incontinence` (interface dermatitis). PCA confirms **20 components retain 94.4%** of variance.

**Technique used.** L2 regularisation with `C` tuned across four orders of magnitude; ensemble methods that handle correlated features natively; **both** mutual-information *and* tree-based importance reported, since they answer different questions.

**Reason.** Multicollinearity destabilises linear-model *coefficients* without necessarily harming *predictions* — two correlated features can trade off arbitrarily. L2 shrinkage distributes weight across correlated features rather than letting one dominate arbitrarily, stabilising the coefficients a clinician would read.

**Observed consequence.** `band-like_infiltrate` ranks 4th by mutual information but only 11th by Random Forest importance — the forest, having split on a correlated substitute, sees it as redundant. This is why a single importance measure is not a reliable guide.

**Limitation that remains.** Individual feature importances must be read as importance of a *correlated cluster*, not of an isolated marker.

---

## Challenge 7 — Data leakage risk

**Description.** The most consequential methodological error available in a small-data project: letting held-out information influence training, producing scores that look excellent and do not survive deployment.

**Evidence of the risk.** Three specific opportunities existed here: (a) fitting the scaler on all 366 rows before splitting; (b) computing the imputation median on the full dataset; (c) imputing `Age` using the class label.

**Technique used.**
- All fitted transformations placed inside `sklearn.Pipeline`, so `fit` is called on training folds only — enforced by the library, not by discipline.
- Test set split off **before** any modelling decision and used exactly once, in Section 19.
- Feature-selection and feature-engineering experiments all scored by CV **on the training partition only**.
- Class-conditional imputation explicitly considered and **rejected** as target leakage.
- Duplicate check (Section 12) confirms **0 duplicated rows**, so no patient can appear on both sides of the split.

**Reason.** Leakage does not announce itself — it shows up as a suspiciously good score. The only reliable defence is structural: make it impossible rather than merely avoided.

**Evidence the defence worked.** CV accuracy exceeded test accuracy by 1–4 points for every model. That small optimistic bias is exactly what selection-on-CV-folds predicts, and its presence confirms the test set was genuinely held out — a perfectly matching CV and test score would have been the suspicious result.

**Limitation that remains.** The test set was used once to *report* results. In the strictest sense, publishing those numbers means future decisions informed by them are no longer fully blind to it.

---

## Challenge 8 — Interpretability as a hard requirement, not a preference

**Description.** In healthcare, a model that cannot explain itself cannot be responsibly acted upon. This constrains the model space regardless of accuracy.

**Evidence.** The SVM (RBF) achieved joint-best cross-validated accuracy (0.9794) yet offers no native feature-level explanation — its boundary lives in an implicit high-dimensional space.

**Technique used.**
- Interpretability made an explicit, weighted selection criterion in Section 20.11 rather than a footnote.
- **PCA rejected** despite tying on performance, because principal components are not clinically meaningful.
- Three complementary importance analyses reported: mutual information, Gini importance, and test-set permutation importance.
- A two-model deployment pattern recommended (Random Forest + Logistic Regression companion) so transparency costs essentially no accuracy.

**Reason.** A dermatologist presented with "the model says lichen planus" needs to know *why* in order to agree or overrule. "Band-like infiltrate at grade 3 and saw-tooth retes at grade 2" is auditable against their own examination. "PC3 was elevated" is not.

**Limitation that remains.** The Random Forest's interpretability is *moderate*, not high. Per-prediction explanation would benefit from SHAP values, which are not implemented here — noted in Future Scope.

---

## Challenge 9 — Anomalous and ambiguous records

**Description.** One patient is recorded with `Age = 0`.

**Evidence.** Row-level inspection (Section 12) identifies a single record, class 1 (psoriasis), with `Age = 0`. No global Tukey outliers exist in `Age`.

**Technique used.** Retained, documented, and flagged in the limitations.

**Reason.** Two readings are possible — a genuine infant (infantile psoriasis is uncommon but real) or a placeholder for "unknown" — and the file alone cannot distinguish them. One record in 366 cannot meaningfully distort a tree ensemble or a regularised linear model. In a healthcare context the correct default is to **preserve the record and document the ambiguity**, not to delete data on the basis of a guess.

**Limitation that remains.** If it *is* a placeholder, it is an unrecognised missing value being treated as a real measurement.

---

## 21.10 Summary

| # | Challenge | Primary technique | Measured / documented outcome |
|---|---|---|---|
| 1 | Small $n$, $n/p = 10.8$ | Stratified 5-fold CV; small grids; tuned regularisation | Stable results; CV std ≤ 0.026 for every model |
| 2 | `'?'` in `Age` (8 values) | `na_values` + median imputation in-pipeline | All 366 records retained, leak-free |
| 3 | Shared defining features | Retain all 34; multivariate models | `erythema` MI = 0.0630 (rank 32/34) confirms the problem |
| 4 | Seborrheic derm. ↔ pit. rosea | Predicted from EDA, confirmed across 6 models | Majority of all test errors; carried into clinical guidance |
| 5 | 5.6:1 imbalance | Stratification + `f1_macro` + tuned `class_weight` | Balancing gain +0.003–0.004 — within noise |
| 6 | Multicollinearity | L2 regularisation; ensembles; dual importance measures | 20 PCs retain 94.4% of variance |
| 7 | Leakage risk | `Pipeline` encapsulation; single test-set use | CV exceeded test by 1–4 pts — the expected, healthy signature |
| 8 | Interpretability requirement | Weighted selection criterion; PCA rejected; 2-model pattern | SVM excluded despite joint-best CV score |
| 9 | `Age = 0` record | Retained and documented | 1 record; flagged in limitations |

<a id="s22"></a>
# 22. SUGGESTIONS TO DOCTORS — **Task 3**

> **Task 3: Suggestions to the Doctors to identify the skin diseases of the patient at the earliest.**

---

> ### ⚠️ Disclaimer — read first
>
> This section is derived from a **366-patient academic dataset** analysed in a student capstone project. It is **not clinical guidance**, has not been reviewed by a dermatologist, and has not been validated on any independent cohort. It is offered as a structured summary of what this particular dataset shows, intended to inform discussion and further study — **not to direct patient care**. All clinical decisions must rest with a qualified clinician exercising independent judgement.

---

## 22.1 The central finding: redness and scaling tell you almost nothing

The single most useful thing this analysis offers is a **redirection of diagnostic attention**.

Mutual-information analysis (Section 16.2) ranks all 34 attributes by how much they reduce diagnostic uncertainty. `erythema` ranks **32nd of 34** (MI = 0.0630) — roughly **one-tenth** the information content of the top marker. `inflammatory mononuclear infiltrate` ranks last (0.0322).

This is not a surprising result once stated, but it is easy to act against in practice. Erythema and scaling *define membership* of this disease group, so they are present in essentially every patient and therefore carry almost no information about *which* member. **The features that present most prominently are the features that discriminate least.**

**Practical implication.** Once erythema and scaling have established that a patient belongs to the erythemato-squamous group, attention should move immediately to the specific discriminating markers below. Time spent characterising the redness in more detail is, on this evidence, time that does not advance the differential.

## 22.2 The highest-yield discriminating features

Ranked by mutual information with the diagnosis (training partition, Section 16.2):

| Rank | Feature | MI | Type | Points towards |
|---|---|---|---|---|
| 1 | Clubbing of the rete ridges | 0.6057 | Histopath. | **Psoriasis** |
| 2 | Elongation of the rete ridges | 0.5810 | Histopath. | Psoriasis / chronic dermatitis |
| 3 | Thinning of the suprapapillary epidermis | 0.5391 | Histopath. | **Psoriasis** |
| 4 | Band-like infiltrate | 0.5116 | Histopath. | **Lichen planus** |
| 5 | Saw-tooth appearance of retes | 0.5029 | Histopath. | **Lichen planus** |
| 6 | Vacuolisation / damage of basal layer | 0.4893 | Histopath. | **Lichen planus** |
| **7** | **Polygonal papules** | **0.4860** | **CLINICAL** | **Lichen planus** |
| 8 | Focal hypergranulosis | 0.4605 | Histopath. | Lichen planus |
| 9 | Spongiosis | 0.4572 | Histopath. | Seborrheic dermatitis / pit. rosea |
| 10 | Melanin incontinence | 0.4562 | Histopath. | Lichen planus |
| 11 | Fibrosis of the papillary dermis | 0.4413 | Histopath. | **Chronic dermatitis** |
| 12 | Oral mucosal involvement | 0.4160 | CLINICAL | Lichen planus |

**Note rank 7.** `polygonal_papules` is the **highest-ranked clinical feature in the entire dataset** — more informative than 27 of the 34 attributes, and obtainable at the bedside with no biopsy. Careful characterisation of papule morphology is, on this evidence, one of the highest-return observations available at first consultation.

## 22.3 Disease-specific signatures

Derived from the class-conditional profile analysis (Section 11.6). Each entry shows how far a marker is elevated above the mean of the other five diseases.

### 1. Psoriasis (n = 112)
| Marker | Elevation |
|---|---|
| Clubbing of the rete ridges | +2.08 |
| Thinning of the suprapapillary epidermis | +2.05 |
| Elongation of the rete ridges | +1.83 |
| Scalp involvement | +1.40 |
| Knee and elbow involvement | +1.27 |

**Bedside profile:** sharply demarcated lesions on extensor surfaces and scalp; **positive family history in 28.6%**. Typically *low* spongiosis and exocytosis — their absence is itself informative.

### 2. Seborrheic dermatitis (n = 61) — *see the caution in 22.4*
| Marker | Elevation |
|---|---|
| Spongiosis | +1.24 |
| Exocytosis | +0.81 |
| PNL infiltrate | +0.80 |
| Itching | +0.41 |

**Bedside profile:** ill-defined borders (mean grade 0.95 vs 2.10 for psoriasis); notably *low* knee/elbow and Koebner involvement. A spongiotic rather than psoriasiform pattern.

### 3. Lichen planus (n = 72) — the cleanest signature in the dataset
| Marker | Elevation |
|---|---|
| Band-like infiltrate | **+2.70** |
| Vacuolisation / damage of basal layer | **+2.30** |
| Saw-tooth appearance of retes | **+2.29** |
| **Polygonal papules (clinical)** | **+2.28** |
| Melanin incontinence | +2.06 |

**Bedside profile:** highest itching of all six (mean 2.28); oral mucosal involvement is near-specific. **Polygonal papules alone provide strong evidence at the first visit** — this disease is largely identifiable before biopsy.

### 4. Pityriasis rosea (n = 49) — *see the caution in 22.4*
| Marker | Elevation |
|---|---|
| Spongiosis | +1.00 |
| Koebner phenomenon | +0.77 |
| Exocytosis | +0.63 |

**This disease has no strongly elevated marker** — the weakest signature in the dataset. Notable for **lowest itching** (mean 0.47) and **zero positive family history** among all 49 patients.

### 5. Chronic dermatitis (n = 52)
| Marker | Elevation |
|---|---|
| Fibrosis of the papillary dermis | **+2.28** |
| Elongation of the rete ridges | +1.38 |
| Itching | +0.72 |

**Bedside profile:** high itch with *low* scaling — an unusual combination that is itself discriminating. Dermal fibrosis is near-unique and reflects the end-stage of a chronic scratch–inflame cycle.

### 6. Pityriasis rubra pilaris (n = 20)
| Marker | Elevation |
|---|---|
| Follicular papules | **+2.14** |
| Perifollicular parakeratosis | **+2.05** |
| Follicular horn plug | +1.74 |
| Knee and elbow involvement | +1.35 |
| Family history | +0.43 |

**Two powerful bedside clues:** **mean age 10.25 years** (SD 3.71, maximum 22 — every patient in this cohort was a child or young adult) and **50% positive family history**, the highest of any disease here. An entirely follicular signature, on a different pathological axis from all five others.

## 22.4 The one pair to be cautious about

**Seborrheic dermatitis and pityriasis rosea are the hardest pair to distinguish, and this was true both before and after modelling.**

The prediction came from the data itself (Section 11.6): both are characterised by mild spongiosis and by the *absence* of other diseases' markers. Their best separators differ by only ~1.15 grades, against +2.70 for lichen planus's top marker.

It was confirmed by every model (Section 19.4): across six classifiers spanning linear, tree, ensemble, kernel and instance-based families, **16 of 26 total errors (61.5%) involve this one pair.** Models with entirely different assumptions converging on the same confusion is strong evidence that the ambiguity is in the data, not in any algorithm.

**The two features that help most — both obtainable at the bedside:**

| Feature | Seborrheic dermatitis | Pityriasis rosea | Gap |
|---|---|---|---|
| **Itching** | mean 1.62 | mean 0.47 | **1.15 grades** |
| **Koebner phenomenon** | mean 0.47 | mean 1.62 | **1.15 grades** |
| PNL infiltrate | higher | lower | 0.96 grades |

**Suggested practice:** when the differential narrows to these two, grade **itching** and **Koebner phenomenon** with particular care — they are the highest-yield observations available, and both are free. Where they remain equivocal, this is the presentation where **biopsy adds the most diagnostic value**, and where a model's output should carry the least weight.

## 22.5 Age and family history — two free, high-value data points

**Age.** Four of five diseases are age-indistinguishable (means 35.3–40.0 years). **Pityriasis rubra pilaris is the exception and the exception is dramatic:** mean 10.25 years, SD 3.71, no patient over 22.

> **A paediatric patient with follicular papules and follicular horn plugs should raise PRP prominently in the differential.**

**But note the inverse caution:** because this cohort contains **no adult PRP cases**, a model trained on it will systematically under-predict PRP in adults. Adult-onset PRP is a recognised entity; its absence here is a **property of this sample**, not of the disease. A clinician must not let the model's age prior override a clinical suspicion of PRP in an adult.

**Family history.** Present in only 46 of 366 patients (12.6%) — but highly specific:

| Disease | Positive family history |
|---|---|
| **Pityriasis rubra pilaris** | **50.0%** (10/20) |
| **Psoriasis** | **28.6%** (32/112) |
| Seborrheic dermatitis | 4.9% (3/61) |
| Lichen planus | 1.4% (1/72) |
| **Pityriasis rosea** | **0.0%** (0/49) |
| **Chronic dermatitis** | **0.0%** (0/52) |

**A positive family history shifts the differential sharply towards psoriasis or PRP and, in this cohort, effectively excludes pityriasis rosea and chronic dermatitis.** A negative family history says very little — 71% of psoriasis patients here have none. **Asking the question costs nothing and takes seconds.**

## 22.6 When is a biopsy needed? A data-supported answer

Section 16.5 measured the diagnostic value of each feature block separately:

| Available information | CV accuracy | Available when |
|---|---|---|
| **12 clinical features only** | **0.8766 ± 0.0354** | First consultation |
| 22 histopathological only | 0.9520 ± 0.0200 | After biopsy |
| **All 34** | **0.9760 ± 0.0085** | Full workup |

**Bedside assessment alone reaches 87.7%** — correct in roughly seven of every eight patients, against a 30.6% baseline. Histopathology adds **9.9 points**.

**A two-stage protocol is well supported by this data:**

**Stage 1 — first consultation.** Grade the 12 clinical features carefully, paying particular attention to polygonal papules, itching, Koebner phenomenon, distribution (knee/elbow, scalp, oral mucosa), age and family history. Where these point clearly to one disease and clinical judgement agrees, the diagnosis is likely settled.

**Stage 2 — targeted biopsy.** Prioritise biopsy for:

1. **The seborrheic dermatitis ↔ pityriasis rosea differential** — where models and clinicians both struggle most.
2. **Suspected psoriasis before committing to long-term systemic therapy** — the treatment burden justifies confirmation.
3. **Any presentation atypical for the suspected disease** — particularly adult patients with follicular features, where the age prior is unreliable.
4. **Suspected lichen planus with oral involvement** — for both confirmation and baseline documentation.
5. **Any case where clinical assessment leaves genuine uncertainty**, regardless of what a model outputs.

**Where biopsy may be lower priority:** clear-cut lichen planus with polygonal papules, oral lesions and marked itch; clear-cut psoriasis with extensor distribution, sharp borders and positive family history; paediatric PRP with follicular papules and horn plugs. In these presentations the clinical signature is strong and consistent in this dataset.

**The benefit runs both ways:** patients with clear-cut presentations avoid an invasive procedure and a multi-day wait, while laboratory capacity concentrates on the cases where histology actually changes the answer.

## 22.7 Early screening checklist

A structured prompt for the first consultation, ordered by measured information value.

**Step 1 — Confirm group membership.** Erythema and scaling present? → erythemato-squamous group. *Record and move on; these do not discriminate further.*

**Step 2 — Two free questions (ask every patient):**
- **Age?** → under ~22 with follicular features raises PRP prominently.
- **Family history of any of these six diseases?** → positive shifts towards psoriasis or PRP; effectively excludes pityriasis rosea and chronic dermatitis in this cohort.

**Step 3 — The high-yield clinical observations:**

| Observe | If prominent, consider |
|---|---|
| **Polygonal papules** (rank 7 of 34 — highest-value clinical feature) | Lichen planus |
| **Oral mucosal involvement** | Lichen planus |
| **Follicular papules / horn plugs** | Pityriasis rubra pilaris |
| **Knee and elbow + scalp involvement** | Psoriasis |
| **Sharply defined borders** | Psoriasis or lichen planus (vs. seborrheic dermatitis) |
| **Marked itching with low scaling** | Chronic dermatitis |
| **Marked itching** | Lichen planus (highest) or seborrheic dermatitis |
| **Koebner phenomenon** | Pityriasis rosea (vs. seborrheic dermatitis) |
| **Minimal itching** | Pityriasis rosea (lowest of all six) |

**Step 4 — Decide.** Clear signature + consistent clinical picture → proceed. Differential narrowed to seborrheic dermatitis vs pityriasis rosea, or any atypical presentation → **biopsy**.

**Step 5 — If histopathology is available**, the highest-value markers are: rete-ridge architecture (psoriasis), band-like infiltrate with basal vacuolisation and saw-tooth retes (lichen planus), papillary dermal fibrosis (chronic dermatitis), and perifollicular parakeratosis (PRP).

## 22.8 How a model should and should not be used

**A model like this is decision *support*. It is not a diagnosis, and it must not become one.**

| Appropriate use | Inappropriate use |
|---|---|
| A **second opinion** flagging a diagnosis the clinician had not prioritised | Replacing clinical examination |
| A **triage aid** identifying which patients most need a biopsy or specialist referral | Auto-generating a diagnosis without review |
| A **teaching tool** showing trainees which features discriminate and which do not | Overriding a clinician's judgement |
| A **consistency check** in settings without specialist dermatology access | Use on patients outside the six modelled diseases |
| An **uncertainty flag** — low confidence output prompts closer review | Treating output probabilities as clinically calibrated |

**Three specific, identified failure modes a clinician must know about:**

1. **The model cannot say "none of these."** It is trained on exactly six diseases and will assign every patient to one of them — confidently — including patients with an entirely different condition. **A model output is only meaningful once a clinician has established the patient plausibly belongs to this disease group.**
2. **It will under-predict adult PRP,** because this cohort contains no adult PRP cases.
3. **It is least reliable on the seborrheic dermatitis / pityriasis rosea pair,** which is where most of its errors fall.

## 22.9 What would be needed before clinical use

1. **External validation** on an independent cohort from a different institution and population.
2. **Prospective study** applying the model at the point of care against confirmed outcomes.
3. **Inter-rater reliability study** on the 0–3 grading scales — the model assumes expert-consistent input, and this assumption is untested.
4. **Dermatologist benchmarking** on identical cases — the only comparison that establishes clinical value.
5. **Expanded cohort:** adult PRP cases, more seborrheic dermatitis / pityriasis rosea patients, longitudinal follow-up to address early-stage ambiguity.
6. **Calibration assessment** — a "70% confident" output must be right about 70% of the time.
7. **Bias audit** across skin tone, age and sex. Feature grading such as erythema severity is known to be harder to assess on darker skin, and this dataset records no skin-tone information — an unexamined and potentially serious source of differential performance.

## 22.10 Ethical considerations

**Automation bias.** The best-documented risk of clinical decision support: clinicians defer to a confident-looking output against their own judgement. A model presenting "97% accurate" invites exactly this. Mitigations: always display *why* (feature contributions), always display uncertainty, and never present a prediction without the alternatives it considered.

**Accountability.** Clinical responsibility rests with the clinician. A tool that obscures this — or is perceived to shift it — is unsafe regardless of accuracy.

**Equity.** This dataset records no skin tone, and erythema is demonstrably harder to grade on darker skin. Performance across skin tones is therefore **completely unmeasured**. Deployment without a bias audit risks systematically worse care for the patients least well represented in the training data.

**Informed consent and transparency.** Patients should know when an algorithm contributes to their diagnosis, and be able to request review without it.

**Data provenance.** This dataset is de-identified, but any deployed system processes identifiable clinical data and inherits all associated privacy obligations. Note in particular that a **KNN model stores its training patients inside the deployed artefact** — one reason it was not recommended in Section 20.11.

**Scope discipline.** The strongest ethical safeguard available here is an honest statement of scope. **This is an academic prototype trained on 366 patients from a single source, evaluated on 74. It demonstrates that these diseases are highly separable from structured clinical data. It does not demonstrate that it is safe to use on a patient.**

<a id="s23"></a>
# 23. Final Findings

**On the data**

1. The dataset is complete and well-formed: **366 patients × 34 attributes + target**, zero duplicates, zero out-of-range values, and a single missing field (`Age`, 8 values, 2.19%).
2. Class distribution is moderately imbalanced at **5.6 : 1** (112 psoriasis to 20 PRP), with Shannon evenness 0.9411.
3. **Age nearly separates pityriasis rubra pilaris on its own** — mean 10.25 years against 35–40 for every other disease, with no patient over 22.
4. **Family history is high-specificity, low-sensitivity**: 50% in PRP, 28.6% in psoriasis, **0%** in pityriasis rosea and chronic dermatitis.
5. **The defining features of the disease group discriminate least.** `erythema` ranks 32nd of 34 by mutual information (0.0630), about one-tenth the top marker's information content.
6. Each disease has a coherent, medically interpretable signature, dominated by sparse high-grade histopathological markers.
7. **Seborrheic dermatitis and pityriasis rosea lack distinctive markers** and are mutually confusable — predicted from the EDA before modelling and confirmed by every model afterwards.

**On feature engineering**

8. **Scaling is load-bearing, not cosmetic**: SVM gained **+0.2946** CV accuracy and KNN **+0.1438**; tree models were unaffected (delta exactly 0.0000).
9. **Feature selection was tested and rejected.** The best k (25) gained +0.0103 — inside the ±0.0129 noise band — while cutting to 5 features cost **−0.1951**.
10. **PCA was tested and rejected.** It never exceeded raw features, and would have destroyed the interpretability this domain requires.
11. **Class balancing was tested and produced gains within noise** (+0.003–0.004 macro-F1), because the rarest class is not the hardest one.
12. **Bedside features alone reach 87.7% CV accuracy**; adding histopathology brings this to 97.6%.

**On the models**

13. **Random Forest performed best**: test accuracy **0.9730**, macro-F1 **0.9694**, Cohen's κ **0.9661**, **2 errors in 74**, CV accuracy **0.9794 ± 0.0129**.
14. **The top three models are statistically indistinguishable** — separated by one patient, with identical CV accuracies (0.9794) and overlapping confidence intervals.
15. **The Decision Tree is genuinely behind** (8 errors, CV 0.9519) — a real capacity limitation, not noise.
16. **Errors across all six models fall into a small predictable set**: 16 of 26 total misclassifications (61.5%) involve the seborrheic dermatitis ↔ pityriasis rosea pair. Six models with different inductive biases converging on the same confusion is strong evidence it is a property of the data.
17. **Difficulty is driven by signature distinctiveness, not rarity.** PRP is the rarest class (4 test patients) yet outperforms seborrheic dermatitis, which has three times as many.
18. **The model's learned importances align with established dermatological knowledge** — rete-ridge architecture for psoriasis, band-like infiltrate for lichen planus, dermal fibrosis for chronic dermatitis. This coherence is itself a validation signal.

**On methodology**

19. Cross-validation exceeded test accuracy by 1–4 points for every model — the expected signature of selecting hyperparameters on CV folds, and evidence that the test set was genuinely held out.
20. **Of six candidate feature-engineering techniques, only two earned adoption on measured evidence.** The negative results are as much a finding as the positive ones.

<a id="s24"></a>
# 24. Limitations

### Data limitations

| Limitation | Detail | Consequence |
|---|---|---|
| **Sample size** | 366 patients; 292 train / 74 test | Wide confidence intervals; 95% CI on headline accuracy ≈ [0.906, 0.993] |
| **Rarest class** | 20 PRP patients; **4 in the test set** | Per-class metrics for class 6 are indicative only; one error would move F1 by ~0.14 |
| **Single source** | One clinical setting, one grading convention | No evidence of transfer to other populations or institutions |
| **No adult PRP** | Maximum age in class 6 is 22 | Systematic, identifiable under-prediction of adult-onset PRP |
| **`Age = 0` record** | One patient, class 1 | Cannot distinguish genuine infant from an unrecognised missing-value placeholder |
| **8 imputed ages** | 2.19% of rows carry a fabricated value | Negligible at this scale, but fabricated nonetheless |
| **No skin tone recorded** | Absent from the dataset | Performance across skin tones is **completely unmeasured**; erythema is harder to grade on darker skin |
| **Single timepoint** | No longitudinal data | Cannot address the early-stage ambiguity the problem brief explicitly describes |
| **Six classes only** | Closed-world assumption | A patient with a seventh condition is forced into one of these six, with confident-looking output |

### Methodological limitations

| Limitation | Detail |
|---|---|
| **Test set used once to report** | Publishing these numbers means future decisions informed by them are no longer fully blind to the test set |
| **CV optimism** | Hyperparameters were selected on the CV folds, so the CV figure carries a mild selection bias — quantified at 1–4 points by the test comparison |
| **No repeated CV / nested CV** | A single 5-fold split was used. Repeated or nested CV would give tighter variance estimates at higher compute cost |
| **No probability calibration** | Predicted probabilities are not calibrated. A "0.9" output is not verified to mean 90% correct — important for any confidence-threshold deployment |
| **No SHAP / per-prediction explanation** | Global importances only. Individual-patient explanation would strengthen the clinical case |
| **Assumes expert grading** | The model takes expert-assigned 0–3 grades as input. Inter-rater variability is unmeasured and would degrade real-world performance by an unknown amount |
| **Expects all 34 features** | Full accuracy presupposes a completed biopsy |

### What this notebook does **not** establish

- ✗ That the model works on any population other than this cohort.
- ✗ That it matches or exceeds a dermatologist.
- ✗ That it is safe for clinical use.
- ✗ That its confidence scores are calibrated.
- ✗ That it performs equally across skin tones, ages or sexes.
- ✗ That its accuracy would survive realistic inter-rater grading variability.

### What it **does** establish

- ✓ The six erythemato-squamous diseases are **highly separable** from structured clinical and histopathological data (97.3% test accuracy, κ = 0.9661).
- ✓ **Bedside features alone carry substantial diagnostic signal** (87.7% CV accuracy).
- ✓ The residual difficulty is **concentrated in one specific, identifiable disease pair**.
- ✓ The learned feature importances **align with established dermatological knowledge**.
- ✓ The result is **reproducible** — fixed seed, leak-free pipeline, single-use test set.

<a id="s25"></a>
# 25. Conclusion

This project delivered a complete, reproducible machine-learning pipeline for the differential diagnosis of six erythemato-squamous diseases from 366 patient records, covering all three tasks in the PRCP-1027 brief within a single notebook.

**Task 1** produced a full data-analysis report. The dataset proved unusually clean — no duplicates, no out-of-range values, one missing field affecting 2.19% of rows — which meant analytical effort could go into *understanding structure* rather than cleaning. That analysis surfaced findings that shaped everything downstream: the near-complete age separation of pityriasis rubra pilaris, the high specificity of family history, the fact that the group's defining features (erythema, scaling) are its least informative, and the prediction — made before any model was trained — that seborrheic dermatitis and pityriasis rosea would prove the hardest pair.

**Task 2** developed and compared six models across four algorithmic families. The tuned **Random Forest** performed best, reaching **97.30% accuracy**, **0.9694 macro-F1** and **κ = 0.9661** on a held-out test set of 74 patients, with **97.94% ± 1.29%** under cross-validation. But the more useful conclusion is the one about *how much confidence that number supports*: the top three models are separated by a single patient and are statistically indistinguishable. Model selection was therefore made on interpretability, reliability across all six classes, and robustness to the dataset's limitations — not on a decimal place.

**Task 3** translated those findings into clinical guidance: which features actually discriminate, what each disease's signature looks like, which pair demands extra caution, when biopsy adds the most value, and — importantly — the specific failure modes a clinician must know about before trusting any output.

**Three things distinguish this analysis from a standard classification exercise.**

First, **the negative results were reported as results.** Six feature-engineering techniques were tested; only scaling and imputation earned adoption. Feature selection, PCA and SMOTE were all rejected on measured evidence rather than adopted because they appear in every tutorial. The pipeline is simpler and better for it.

Second, **the EDA made a falsifiable prediction that the modelling confirmed.** Section 11.6 identified the seborrheic dermatitis / pityriasis rosea pair as intrinsically ambiguous by comparing marker elevations (+1.00 versus +2.70 for lichen planus). Section 19.4 then found that this pair accounts for the majority of errors across all six models. Six classifiers with entirely different inductive biases converging on the same confusion establishes that the difficulty lies in the data, not the algorithms.

Third, **the uncertainty is treated as a finding rather than a disclaimer.** A 97.3% accuracy on 74 patients has a 95% confidence interval spanning roughly [0.906, 0.993]. Four test patients in the rarest class means its F1 of 1.000 reflects four correct predictions. These facts are stated where the numbers are reported, not relegated to a closing caveat.

**The honest summary:** this work demonstrates that erythemato-squamous diseases are highly separable from structured clinical and histopathological features, that meaningful diagnostic signal is available at the bedside before any biopsy, and that the residual difficulty is concentrated in one identifiable disease pair. It is an academic prototype and a research result. It is not a clinically validated diagnostic device, and the path from one to the other — external validation, prospective study, inter-rater reliability assessment, dermatologist benchmarking, bias audit — is set out explicitly in Sections 20.12 and 26.

<a id="s26"></a>
# 26. Future Scope

### Immediate extensions

| Direction | Rationale |
|---|---|
| **SHAP per-prediction explanations** | Global importances show what matters on average; SHAP shows why *this* patient got *this* diagnosis. Directly addresses the interpretability requirement identified as decisive in Section 20.10 |
| **Probability calibration** (Platt / isotonic) | Confidence-threshold deployment requires that "70% confident" means 70% correct. Currently unverified |
| **Repeated / nested cross-validation** | Tighter variance estimates and an unbiased estimate of the tuning procedure itself |
| **Soft-voting ensemble** of the top three | The three leaders make *different* errors; combining them may recover some of the 2–3 residual misclassifications |
| **Ordinal-aware modelling** | The 0–3 grades are ordinal. Ordinal logistic regression or monotonic-constraint gradient boosting would encode this explicitly rather than treating the grades as arbitrary numbers |

### Targeted data collection — the highest-value direction

The learning curves show performance plateauing after ~150–200 samples, so *more of the same data* yields limited gains. What would help is **specific** data:

1. **More seborrheic dermatitis and pityriasis rosea cases** — directly targets the one confusion that accounts for most errors.
2. **Adult pityriasis rubra pilaris cases** — corrects the identified age artefact and removes a known systematic failure mode.
3. **Longitudinal follow-up records** — the only way to address the early-stage ambiguity the problem brief describes, where a disease shows another's features initially.
4. **Skin-tone information** — currently absent, making equity assessment impossible.
5. **Multi-centre data** — required for any claim of generalisation beyond a single setting.

### Additional features worth capturing

The seborrheic dermatitis / pityriasis rosea confusion might be resolved by information this dataset does not record: the **herald patch** characteristic of pityriasis rosea, **lesion distribution along skin cleavage lines**, **dermoscopic findings**, **symptom duration and evolution**, and **response to initial treatment**. Any of these could break a tie that the current 34 features cannot.

### Clinical translation pathway

1. External validation on an independent multi-centre cohort.
2. Inter-rater reliability study on the 0–3 grading scales.
3. Prospective observational study at the point of care.
4. Dermatologist benchmarking on identical cases.
5. Bias and equity audit across skin tone, age and sex.
6. Human-factors study of the interface, specifically testing for **automation bias**.
7. Regulatory pathway assessment for clinical decision-support software.

### Deployment engineering

A clinician-facing interface presenting the differential as **ranked probabilities with feature-level justification** rather than a single label; explicit uncertainty flags for low-confidence and seborrheic-dermatitis/pityriasis-rosea cases; an "atypical presentation" warning; audit logging; and a clear, visible statement of scope and limitations at the point of use.

<a id="s27"></a>
# 27. References

**Dataset**

1. Ilter, N. and Guvenir, H.A. (1998). *Dermatology Data Set*. UCI Machine Learning Repository. University of California, Irvine, School of Information and Computer Sciences. https://archive.ics.uci.edu/ml/datasets/Dermatology

2. Guvenir, H.A., Demiroz, G. and Ilter, N. (1998). Learning differential diagnosis of erythemato-squamous diseases using voting feature intervals. *Artificial Intelligence in Medicine*, 13(3), 147–165.

**Methodology**

3. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

4. Hastie, T., Tibshirani, R. and Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction*, 2nd ed. Springer.

5. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.

6. Kohavi, R. (1995). A study of cross-validation and bootstrap for accuracy estimation and model selection. *IJCAI*, 14(2), 1137–1145.

7. Varma, S. and Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics*, 7, 91.

8. Chawla, N.V., Bowyer, K.W., Hall, L.O. and Kegelmeyer, W.P. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*, 16, 321–357.

9. Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement*, 20(1), 37–46.

**Clinical machine learning and ethics**

10. Rudin, C. (2019). Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. *Nature Machine Intelligence*, 1(5), 206–215.

11. Collins, G.S., Reitsma, J.B., Altman, D.G. and Moons, K.G.M. (2015). Transparent Reporting of a multivariable prediction model for Individual Prognosis Or Diagnosis (TRIPOD). *Annals of Internal Medicine*, 162(1), 55–63.

12. Goddard, K., Roudsari, A. and Wyatt, J.C. (2012). Automation bias: a systematic review of frequency, effect mediators, and mitigators. *Journal of the American Medical Informatics Association*, 19(1), 121–127.

13. Adamson, A.S. and Smith, A. (2018). Machine learning and health care disparities in dermatology. *JAMA Dermatology*, 154(11), 1247–1248.

**Software**

14. Harris, C.R. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362.
15. McKinney, W. (2010). Data Structures for Statistical Computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61.
16. Hunter, J.D. (2007). Matplotlib: A 2D Graphics Environment. *Computing in Science & Engineering*, 9(3), 90–95.
17. Waskom, M.L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software*, 6(60), 3021.

**Project documentation**

18. PRCP-1027 — Skin Disorder. Project requirements document (supplied). *Note: the download link within this document references `PRCP-1028-Skin-Disorder-Prediction`; this naming inconsistency is documented in Section 10 of this notebook. The uploaded CSV was used as the source of truth and verified programmatically against the document's attribute list.*

<a id="s28"></a>
# 28. Execution Guide

## 28.1 Requirements

- **Python 3.9 or newer** (developed and verified on Python 3.11+)
- Approximately 500 MB free disk space for the environment
- Total runtime: **2–4 minutes** on a standard laptop

## 28.2 Install dependencies

Create a file named `requirements.txt` containing:

```
numpy>=1.24
pandas>=2.0
matplotlib>=3.7
seaborn>=0.12
scikit-learn>=1.3
joblib>=1.3
jupyterlab>=4.0
```

Then install:

```bash
python -m venv venv

# Windows
venv\Scripts\activate
# macOS / Linux
source venv/bin/activate

pip install --upgrade pip
pip install -r requirements.txt
```

**Optional extras** (the notebook runs fully without them):

```bash
pip install xgboost              # enables the supplementary cell in Section 19.9
pip install imbalanced-learn     # enables the SMOTE availability check in Section 16.7
```

## 28.3 Place the dataset

Put `dataset_35_dermatology__1_.csv` in **any** of these locations:

```
project_folder/
├── PRCP-1027_Skin_Disorder_Prediction.ipynb
├── dataset_35_dermatology__1_.csv          ←  simplest option
├── data/
│   └── dataset_35_dermatology__1_.csv      ←  also found automatically
└── outputs/                                ←  created by the notebook
```

The notebook searches these paths in order and reports which one it used. If none matches, it raises a clear `FileNotFoundError` listing every path it tried.

## 28.4 Configure (only if needed)

Everything configurable lives in **one cell — Section 9**. To point at a different location, add your path to the front of `CANDIDATE_PATHS`:

```python
CANDIDATE_PATHS = [
    Path("/your/custom/path/dataset_35_dermatology__1_.csv"),   # add this line
    Path(DATASET_FILENAME),
    ...
]
```

Other settings you may wish to change:

| Setting | Default | Effect |
|---|---|---|
| `RANDOM_STATE` | 42 | Changing it will produce slightly different numbers — this is expected, and the magnitude of the change is a useful measure of how stable the results are |
| `TEST_SIZE` | 0.20 | Larger test set → tighter test estimates, smaller training set |
| `CV_FOLDS` | 5 | 10 folds gives lower bias at roughly double the runtime |
| `SCORING` | `"f1_macro"` | Tuning objective |

## 28.5 Run

```bash
jupyter lab
```

Open `PRCP-1027_Skin_Disorder_Prediction.ipynb`, then **Run → Run All Cells**.

Cells are ordered so that every variable is defined before use. Run them in order; running out of order will raise `NameError`.

## 28.6 What it produces

An `outputs/` folder containing:

| File | Contents |
|---|---|
| `best_model_Random_Forest.joblib` | The complete fitted pipeline — imputer, scaler and model |
| `model_comparison_results.csv` | Full metrics table for all six models |
| `per_class_f1_scores.csv` | Per-disease F1 for every model |
| `model_metadata.json` | Dataset facts, class mapping, chosen hyperparameters, headline scores |

## 28.7 Using the saved model

```python
import joblib
import pandas as pd

model = joblib.load("outputs/best_model_Random_Forest.joblib")

# new_patients must be a DataFrame with the same 34 columns, in the same order,
# containing RAW values — imputation and scaling are inside the pipeline.
preds = model.predict(new_patients)
probs = model.predict_proba(new_patients)

CLASS_NAMES = {1: "Psoriasis", 2: "Seborrheic dermatitis", 3: "Lichen planus",
               4: "Pityriasis rosea", 5: "Chronic dermatitis", 6: "Pityriasis rubra pilaris"}
print([CLASS_NAMES[p] for p in preds])
```

Because the saved object is the **complete pipeline**, raw unprocessed input can be passed directly. This eliminates train/serve skew — one of the most common sources of silent production failure.

## 28.8 Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `FileNotFoundError` on load | CSV not in a searched path | Place it beside the notebook, or add its path to `CANDIDATE_PATHS` |
| `Age` column has dtype `object` | `'?'` markers not converted | Section 10 handles this via `na_values=['?']` plus `pd.to_numeric`. If you load the file yourself, do the same |
| `ValueError: could not convert string to float: '?'` | Loaded without `na_values` | Use `pd.read_csv(path, na_values=['?'])` |
| `ModuleNotFoundError: xgboost` | Optional package absent | Expected. Section 19.9 detects this and skips gracefully. Install only if you want the supplementary comparison |
| `ModuleNotFoundError: imblearn` | Optional package absent | Expected. `class_weight` is used instead |
| `NameError` on any variable | Cells run out of order | Kernel → Restart Kernel and Run All Cells |
| Slightly different numbers | Different scikit-learn version, different OS, or changed `RANDOM_STATE` | Expected. Differences should be small; large differences indicate a version or configuration issue |
| Tuning feels slow | Grid search across six models | 2–4 minutes is normal. `n_jobs=-1` already parallelises across all cores |

## 28.9 If you are working from the original UCI file instead

This notebook uses the supplied CSV, which has a header row and named columns. The raw UCI `dermatology.data` file has **no header** and uses `'?'` for missing values. To load that format instead:

```python
COLUMN_NAMES = [
    "erythema", "scaling", "definite_borders", "itching", "koebner_phenomenon",
    "polygonal_papules", "follicular_papules", "oral_mucosal_involvement",
    "knee_and_elbow_involvement", "scalp_involvement", "family_history",
    "melanin_incontinence", "eosinophils_in_the_infiltrate", "PNL_infiltrate",
    "fibrosis_of_the_papillary_dermis", "exocytosis", "acanthosis", "hyperkeratosis",
    "parakeratosis", "clubbing_of_the_rete_ridges", "elongation_of_the_rete_ridges",
    "thinning_of_the_suprapapillary_epidermis", "spongiform_pustule",
    "munro_microabcess", "focal_hypergranulosis", "disappearance_of_the_granular_layer",
    "vacuolisation_and_damage_of_basal_layer", "spongiosis",
    "saw-tooth_appearance_of_retes", "follicular_horn_plug",
    "perifollicular_parakeratosis", "inflammatory_monoluclear_inflitrate",
    "band-like_infiltrate", "Age", "class",
]

df = pd.read_csv("dermatology.data", header=None, names=COLUMN_NAMES, na_values=["?"])
```

Everything downstream is unchanged.

<a id="s29"></a>
# 29. Final Verification Report

The cell below regenerates this report from the live notebook state, so the numbers cannot drift out of sync with the analysis above.

In [ ]:
print("=" * 78)
print(" " * 18 + "FINAL VERIFICATION REPORT")
print("=" * 78)

print("\n[1] PROJECT")
print(f"    Project ID          : PRCP-1027")
print(f"    Title               : Skin Disorder Prediction / Skin Disorder")
print(f"    Domain              : Healthcare")
print(f"    Task type           : Multi-class classification (6 classes)")

print("\n[2] VERIFIED DATASET STRUCTURE")
print(f"    File                : {DATASET_PATH.name}")
print(f"    Rows x Columns      : {df.shape[0]} x {df.shape[1]}")
print(f"    Predictors          : {len(FEATURE_COLS)}  ({len(CLINICAL)} clinical + {len(HISTOPATHOLOGICAL)} histopathological)")
print(f"    Target column       : '{TARGET_COL}'")
print(f"    Ordinal features    : {len(ORDINAL_COLS)} (range 0-3, verified)")
print(f"    Continuous features : 1 ({AGE_COL}, range {df[AGE_COL].min():.0f}-{df[AGE_COL].max():.0f})")
print(f"    Structure matches requirements document : {all(ok for _, ok in checks)}")

print("\n[3] CLASS MAPPING AND COUNTS")
_c = df[TARGET_COL].value_counts().sort_index()
for k in LABELS_ORDER:
    print(f"    {k} -> {CLASS_NAMES[k]:<28s} : {_c[k]:3d} patients ({_c[k]/len(df)*100:5.2f}%)")
print(f"    {'':<33s}   ---")
print(f"    {'TOTAL':<33s} : {_c.sum():3d}")
print(f"    Imbalance ratio     : {_c.max()/_c.min():.2f} : 1")

print("\n[4] DATA QUALITY")
print(f"    Missing values      : {int(df.isna().sum().sum())} (all in '{AGE_COL}': {int(df[AGE_COL].isna().sum())}, "
      f"{df[AGE_COL].isna().mean()*100:.2f}% of rows)")
print(f"    Duplicate rows      : {int(df.duplicated().sum())}")
print(f"    Out-of-range values : {sum(int((~df[c].dropna().between(0,3)).sum()) for c in ORDINAL_COLS)}")
print(f"    Constant features   : {len([c for c in FEATURE_COLS if df[c].nunique(dropna=True) <= 1])}")

print("\n[5] SPLIT AND VALIDATION")
print(f"    Training partition  : {len(X_train)} patients ({(1-TEST_SIZE)*100:.0f}%)")
print(f"    Held-out test set   : {len(X_test)} patients ({TEST_SIZE*100:.0f}%)")
print(f"    Stratified          : yes (max class-proportion deviation {max_dev:.2f} pp)")
print(f"    Cross-validation    : StratifiedKFold(n_splits={CV_FOLDS}, shuffle=True, random_state={RANDOM_STATE})")
print(f"    Tuning objective    : {SCORING}")
print(f"    Test set used       : ONCE, in Section 19, after all decisions were frozen")

print("\n[6] PREPROCESSING & FEATURE ENGINEERING")
print(f"    Adopted   : median imputation of {AGE_COL} (in-pipeline, leak-free)")
print(f"                StandardScaler on all 34 features")
print(f"    Tested and rejected : SelectKBest/mutual information (gain within noise)")
print(f"                          PCA (no gain; destroys interpretability)")
print(f"                          SMOTE (non-integer grades on an ordinal scale)")
print(f"    Tuned per model     : class_weight in {{None, 'balanced'}}")

print("\n[7] MODELS EVALUATED")
for i, r in results.iterrows():
    print(f"    {i+1}. {r['Model']:<22s} [{r['Family']}]")

print("\n[8] MEASURED PERFORMANCE (held-out test set, n = {})".format(len(y_test)))
print(f"    {'Model':<22s} {'CV Acc':>9s} {'Test Acc':>9s} {'F1 macro':>9s} {'Kappa':>8s} {'Errors':>7s}")
print("    " + "-" * 68)
for _, r in results.iterrows():
    print(f"    {r['Model']:<22s} {r['CV Acc']:>9.4f} {r['Test Acc']:>9.4f} "
          f"{r['F1 (mac)']:>9.4f} {r['Cohen kappa']:>8.4f} {r['Errors']:>7d}")

print("\n[9] SELECTED MODEL")
_b = results.iloc[0]
print(f"    Model               : {_b['Model']}")
_bp = {k.replace('model__', ''): v for k, v in search_objects[BEST_MODEL_NAME].best_params_.items()}
print(f"    Hyperparameters     : {_bp}")
print(f"    CV accuracy         : {_b['CV Acc']:.4f} +/- {_b['CV Acc std']:.4f}")
print(f"    Test accuracy       : {_b['Test Acc']:.4f}")
print(f"    Test macro-F1       : {_b['F1 (mac)']:.4f}")
print(f"    Test macro-precision: {_b['Precision (mac)']:.4f}")
print(f"    Test macro-recall   : {_b['Recall (mac)']:.4f}")
print(f"    Cohen's kappa       : {_b['Cohen kappa']:.4f}")
print(f"    Errors              : {_b['Errors']} of {len(y_test)} test patients")

print("\n[10] EVALUATION METRICS USED")
print("    Accuracy | Balanced accuracy | Precision (macro, weighted)")
print("    Recall (macro, weighted) | F1 (macro, weighted) | Cohen's kappa")
print("    Confusion matrices | Per-class classification report | CV mean +/- std")

print("\n[11] FILES GENERATED")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"    {p.name:<50s} {p.stat().st_size/1024:8.1f} KB")

print("\n[12] TASK COMPLETION")
print("    Task 1 — Data analysis report          : Sections 11, 12, 13")
print("    Task 2 — Predictive model               : Sections 17, 18, 19")
print("    Task 3 — Suggestions to doctors         : Section 22")
print("    Model Comparison Report                 : Section 20")
print("    Report on Challenges Faced              : Section 21")

print("\n[13] REMAINING LIMITATIONS (see Section 24 in full)")
print("    - 74-patient test set: 95% CI on headline accuracy is approximately [0.906, 0.993]")
print("    - Only 4 test patients in the rarest class (PRP): per-class figures indicative only")
print("    - Single-source cohort: no evidence of transfer to other populations")
print("    - No adult PRP cases: the model will under-predict adult-onset PRP")
print("    - No skin-tone data: performance across skin tones is unmeasured")
print("    - Probabilities are not calibrated")
print("    - NOT clinically validated. Academic prototype only.")

print("\n" + "=" * 78)
print("STATUS: notebook executed end to end.")
print("All figures above were computed in this run — none are hard-coded.")
print("=" * 78)

---

<div align="center">

### End of Notebook

**PRCP-1027 — Skin Disorder Prediction**
Differential Diagnosis of Erythemato-Squamous Diseases using Machine Learning

B.Tech Computer Science (Artificial Intelligence & Machine Learning) — Capstone Project
Domain: Healthcare

---

*All three project tasks, the model comparison report, and the challenges report are contained in this single notebook.*

*Every performance figure reported here was measured by executing the code in this notebook. No results are fabricated or estimated.*

**This work is an academic prototype. It is not a clinically validated diagnostic tool and must not be used to direct patient care.**

</div>